In [1]:
import io
import numpy as np
import scipy as sci
import scipy.constants as scic
import scipy.integrate as scii
import scipy.signal as scs
import scipy.constants as scc
from functools import cmp_to_key
from scipy.interpolate import CubicSpline as cubichers
from scipy.interpolate import interp1d
from typing import Union, Any, List
from dataclasses import dataclass, replace
from dataclasses import field as dcfield
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, Label
from ipywidgets import VBox, interactive_output, IntSlider,fixed, IntText, Dropdown, RadioButtons,interact_manual, Layout, Button, FloatText,Output, HBox
from IPython.display import display, Image
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, FigureCanvasAgg
from matplotlib.backends.backend_tkagg import NavigationToolbar2Tk
from matplotlib.ticker import FormatStrFormatter,EngFormatter
from pandas import DataFrame, concat, read_csv
import warnings
from copy import deepcopy
import jax as jx
import jax.numpy as jxn
from chex import dataclass as jaxdatclass
import optax
import jax.nn as jnn
import jax.scipy.signal as jsig
import jax.scipy.special as jsp
from numba import njit
from functools import partial
from scipy.optimize import least_squares as leasts
from scipy.stats import linregress
import concurrent.futures
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
import re
from importlib import resources
from itertools import product as iterproduct
from functools import lru_cache
from epraya.base_powd import *
from epraya.base_ham import *
from epraya.base_cris import Plotsim
from epraya.base_fit import Fitresult,result,Selectvarian
import matplotlib.cm as cm
from scipy.spatial.transform import Rotation
from jax.scipy.linalg import expm as jexpm
from epraya.base_fit import Packtoscipy, UnpackHam
jx.config.update("jax_enable_x64", True)


import time

class TargetReached(Exception): 
    pass

class BudgetReached(Exception):
    pass
    
class Tracker:
    '''
    Best cost in function of the number of evaluations of the spectrum. Calculates n saves, cost, time, paramters in every iterations.
    '''
    budget=None
    def __init__(self): 
        self.reset()
    def reset(self):
        self.n,self.best,self.hist,self.t0=0,np.inf,[],time.perf_counter()
    target=None      
    def record(self,cost,xfun=None,w=1):
        self.n+=w
        self.tlast=time.perf_counter()-self.t0
        #DOesn't improve
        if cost<self.best:                                 
            self.best=cost
            if xfun is None:
                x=[] 
            else:
                x=list(np.asarray(xfun(),float))
            self.hist.append([self.n,cost,time.perf_counter()-self.t0]+x)
        if self.target is not None and cost<=self.target:
            raise TargetReached
        if self.budget is not None and self.n >= self.budget:
            raise BudgetReached
    def array(self):
        h=np.array(self.hist,float)
        if not len(h):
            return h
        #Last row is the total
        return np.vstack([h,np.r_[self.n,self.best,time.perf_counter()-self.t0,h[-1,3:]]])
        
TRACK=Tracker()

@jaxdatclass
class JHval:
    '''
    Container jax-class for hamiltonial's parameters. Must be initialized with a variable like ``Ham``. To change one parameter use the sintaxis ``Ham.S=1/2``.
    
    Parameters
    ----------
    
    S : float 
        Spin value ex. (1/2,0,3/2).
    g : array_like or float

        g value of system, can be float, for isotropic case or array for anisotropic.
    I : float
        Nuclear spin value

    L : float
        Angular momentum
    A : array_like or float 
        Hyperfine constant, float for isotropic and array for anisotropic

    Q : array_like or float 
        Quadrupole nuclear interaction constant, float for isotropic and array for anisotropic
    D : array_like
        Zero field interaction constants D and E, two value array [0,0]

    Bk2 : array_like
        Stevens k=-/+2 constants
    Bk4 : array_like

        Stevens k=-/+4 constants    
    Bk6 : array_like
        Stevens k=-/+6 constants      
    lc : float

        Spin-orbit interaction constant
    Hpp : array_like
        Peak to peak distance for the voigtian function using [Hg,Hl], for gaussian and lorentzian distance

    eta : float
        weight of the gaussian contribution to the voigtian function, from 0 to 1. If eta is 0, the function is lorentzian and if eta is 1, the function is gaussian.
    weight: float

        Dummy variable by the moment
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
        

    Example
    --------
    
    >>> import epraya as epr 
    >>> Ham=epr.JHval()
    >>> Ham.S=1
    >>> Ham.I=1/2
    >>> Ham.g=[2.003,1.8,1.5]
    >>> print(Ham)
    JHval(S=1, g=[2.003, 1.8, 1.5], I=0.5, L=0.0, A=0.0,
    Q=Array([0, 0, 0], dtype=int32), 
    D=Array([0, 0], dtype=int32), 
    Bk2=[0, 0, 0, 0, 0], Bk4=[0, 0, 0, 0, 0, 0, 0, 0, 0],
    Bk6=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
    lc=0.0, Hpp=Array([0, 1], dtype=int32), eta=0.5, weight=0.0)

    '''
    S: Union[float,int]=1/2   # Spin
    g: Union[List[float],float,int]=dcfield(default_factory=lambda: 2.003)  # g value
    I: float=0.0   # Nuclear spin
    L: float=0.0   # Angular momentum
    A: Union[List[float], float]=dcfield(default_factory=lambda:0.0)     # Hyperfine constant
    Q: Union[list[float], float]=dcfield(default_factory=lambda:jxn.array([0,0,0]))    # Quadrupole interaction constant
    D: Union[list[float], float]=dcfield(default_factory=lambda:jxn.array([0,0]))     # Zero field interaction D and E constants
    Bk2: Union[list[float], float]=dcfield(default_factory=lambda:[0,0,0,0,0])
    Bk4: Union[list[float], float]=dcfield(default_factory=lambda:[0,0,0,0,0,0,0,0,0])
    Bk6: Union[list[float], float]=dcfield(default_factory=lambda:[0,0,0,0,0,0,0,0,0,0,0,0,0])
    lc: float=0.0                          # Spin-orbit interaction constant
    Hpp: List=dcfield(default_factory=lambda:jxn.array([0,1]))
    eta: float=0.5
    weight: float=0.0

@jaxdatclass
class JEco:
    '''
    Container jax-class for the experimental parameters. Must be initialized with a variable like ``Exp``. To change one parameter use the sintaxis ``Exp.Freq=9.40``.
    
    Parameters
    ----------
    
    Freq : float
        Frequency of the microwave in the EPR spectrometer in GHz.
    Points  : int
        Number of points used to take the EPR spectrum.
    Temperature : float
        Temperature of the sample during the measurement in Kelvin.

    Fdirection : list=[0,0,1]
        Direction of incidence of the magnetic field in relation with the lab frame. By deafult it's [0,0,1], the Z direction.
    Mwdirection : list=[1,0,0]
        Direction of incidence of the microwave radiation in relation with the lab frame. By deafult it's [1,0,0], the X direction.
    Frange : list=[0,1]
        Field range used in the EPR spectrum in mT.
    Sampleframe : list==[0,0,0]
        Three Euler angles of the orientation of the sample in relation to the lab frame.
    Molframe : list=[0,0,0]
        Three Euler angles of the orientation of the  paramagnetic molecule in relation to the sample frame.
    gframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor g in the molecular frame.
    Aframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor A in the molecular frame.    
    Dframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor Q in the molecular frame.    
    Qframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor D in the molecular frame.   
        
    Example
    --------
    
    >>> import epraya as epr 
    >>> Exp=epr.JEco()
    >>> Exp.Freq=9.43
    >>> Exp.Points=2046
    >>> Exp.Temperature=306
    >>> Exp.Frange=[0,400]
    >>> Exp.Mwdirection=[0,1,0]
    >>> print(Exp)
    JEco(Freq=9.43, Points=2046, Temperature=306, 
    Fdirection=[0, 0, 1], Mwdirection=[0, 1, 0], 
    Frange=[0, 400], Sampleframe=[0, 0, 0], 
    Molframe=[0, 0, 0], gframe=[0, 0, 0], 
    Aframe=[0, 0, 0], Dframe=[0, 0, 0], Qframe=[0, 0, 0])
    '''
    Freq: float=9.433
    Points: int=4096
    Temperature: float=295.15
    Fdirection: list[float]=dcfield(default_factory=lambda:[0,0,1])
    Mwdirection: list[float]=dcfield(default_factory=lambda:[1,0,0])
    Frange: list[float]=dcfield(default_factory=lambda:[0,1])
    Sampleframe: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Molframe: list[float]=dcfield(default_factory=lambda:[0,0,0])
    gframe: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Aframe: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Dframe: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Qframe: list[float]=dcfield(default_factory=lambda:[0,0,0])


@jaxdatclass
class JEva:
    '''
    Container jax-class for the range of variation of the hamiltonian parameters by defining the minimum and then the maximum value of each variable. Must be initialized with a variable like ``Vary``. To change one parameter use the sintaxis ``Vary.g=[1.5,2.5,1.0,2.0,0.0,1.0]``.
    
    Parameters
    ----------
    
    g : list[float]
        Range to vary the three values of the g tensor.
    A : list[float]  
        Range to vary the three values of the A tensor.
    Q : list[float]  
        Range to vary the three values of the Q tensor.
    D : list[float]
        Range to vary the two values of the D tensor.
    Hpp : list[float]
        Range to vary the two values of the peak to peak distance.
    weight : float
        Dummy variable by the moment.
    
    Example
    --------
    
    >>> import epraya as epr
    >>> Vary=epr.JEva()
    >>> Vary.g=[1.5,2.5,1.0,2.0,0.0,1.0]
    >>> Vary.A=[100,300,200,400,200,250]
    >>> Vary.Hpp=[0,20,10,15]
    >>> print(Vary)
    JEva(g=[1.5, 2.5, 1.0, 2.0, 0.0, 1.0],
    A=[100, 300, 200, 400, 200, 250], Q=0.0, 
    D=0.0, Hpp=[0, 20, 10, 15], weight=0.0)
    '''
    g: Union[list[float],float]=0.0
    A: Union[list[float],float]=0.0     # Hyperfine constant
    Q: Union[list[float],float]=0.0     # Quadrupole interaction constant
    D: Union[list[float],float]=0.0
    Hpp: List=dcfield(default_factory=lambda:jxn.array([0,0]))
    weight: float=0.0

def Jstart():
    '''
    Creates the three JAX containers ``Ham``, ``Exp`` and ``Vary`` for one system.
    
    Returns
    -------
    
    Ham : Class
        Jax container for the hamiltonian parameters.
    Exp : Class
        Jax container for the experimental conditions.
    Vary : Class
        Jax container for the range and parameters to vary.
    
    Example
    -------
    
    >>> import epraya as epr
    >>> Ham, Exp, Vary=epr.Jstart()
    >>> print(Ham,Exp,Vary)
    JHval(S=0.5, g=2.003, I=0.0, L=0.0, A=0.0, 
    Q=Array([0, 0, 0], dtype=int32), D=Array([0, 0], dtype=int32),
    Bk2=[0, 0, 0, 0, 0], Bk4=[0, 0, 0, 0, 0, 0, 0, 0, 0],
    Bk6=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], lc=0.0,
    Hpp=Array([0, 1], dtype=int32), eta=0.5, weight=0.0) 
    JEco(Freq=9.433, Points=4096, Temperature=295.15, 
    Fdirection=[0, 0, 1], Mwdirection=[1, 0, 0], Frange=[0, 1],
    Sampleframe=[0, 0, 0], Molframe=[0, 0, 0], gframe=[0, 0, 0],
    Aframe=[0, 0, 0], Dframe=[0, 0, 0], Qframe=[0, 0, 0]) 
    JEva(g=0.0, A=0.0, Q=0.0, D=0.0, 
    Hpp=Array([0, 0], dtype=int32), weight=0.0)
    '''
    global Exp,Vary,Ham
    Ham,Exp,Vary=JHval(),JEco(),JEva()
    return Ham,Exp,Vary

def JKronecker(a0,b0):
    '''
    Implementation of the Kronecker delta.
    
    Parameters
    ----------

    
    a0 : int
        First value to compare.
    b0 : int
        Second value to compare
    Returns
    -------
    
    1 : int
        If the two values are equal
    0 : int
        If the two values differ
        
    Example
    -------
    
    >>> import epraya as epr
    >>> a, b = 2, 3
    >>> c, d = 2, 4
    >>> print(epr.JKronecker(a,b))
    >>> print(epr.JKronecker(a,c))
    >>> print(epr.JKronecker(b,d))
    0.0
    1.0
    0.0
    '''
    return jxn.where(a0==b0,1.0,0.0)

def JPauli(s):
    '''
    Defines the Pauli matrices for the spin s.

    Parameters
    ----------

    s : float
        Spin operator value of the system.

    Returns
    -------

    sx : jax.numpy.array
        Pauli matrix of the spin x component.

    sy : jax.numpy.array
        Pauli matrix of the spin y component.
    sz : jax.numpy.array
        Pauli matrix of the spin z component.


    Example
    -------

    >>> import epraya as epr
    >>> s = 1
    >>> sx, sy, sz = epr.JPauli(s)
    >>> print(sx, sy, sz)
    [[0.        +0.j 0.70710677+0.j 0.        +0.j]
    [0.70710677+0.j 0.        +0.j 0.70710677+0.j]
    [0.        +0.j 0.70710677+0.j 0.        +0.j]] 
    [ 0.+0.j         -0.-0.70710677j  0.+0.j        ]
    [ 0.+0.70710677j  0.+0.j         -0.-0.70710677j]
    [ 0.+0.j          0.+0.70710677j  0.+0.j        ]] 
    [ 1.  0.  0.]
    [ 0.  0.  0.]
    [ 0.  0. -1.]]
    '''
    #Defines the pauli matrix for all s values:
    ms=jxn.linspace(s,-s,int(2*s+1))
    z1=0.5j
    r,g=0,0
    sz=jxn.diag(ms)
    sx,sy=jxn.zeros([int(2*s+1),int(2*s+1)],dtype=complex),jxn.zeros([int(2*s+1),int(2*s+1)],dtype=complex)
    for r in range (0,int(2*s+1)):
        for g in range (0,int(2*s+1)):
            yn=JKronecker(r+1,g+1)
            ym=JKronecker(r+1,g+2)
            yl=JKronecker(r+2,g+1)
            sx=sx.at[r,g].set(0.5*(ym+yl)*(jxn.sqrt((s+1)*(r+g+1)-((r+1)*(g+1)))))
            sy=sy.at[r,g].set(z1*((ym-yl)*(jxn.sqrt((s+1)*(r+g+1)-((r+1)*(g+1))))))
    return sx,sy,sz

#Spin Orbit term
def JLorbit(sx,sy,sz,lamda,dim,l=0):
    '''
    Function for the spin orbit interaction, with an isotropic coupling constant.
    *It's recommended to use the Stevens Operators instead of this function*

    Parameters
    ----------

    sx : jax.np.array
        Pauli matrix of the spin x component.
    sy : jax.np.array
        Pauli matrix of the spin y component.
    sz : jax.np.array
        Pauli matrix of the spin z component.
    lamda : float
        Isotropic coupling constant.
    dim : int
        Dimension of the total hamiltonian.
    l : float
        Angular momentum operator

    Returns
    -------
    
    orbe : jax.np.array
        Matrix of the spin orbit interaction with dimension dim.

    Example
    -------
    
    >>> import epraya as epr 
    >>> Ham, Exp, Vary=epr.Jstart()
    >>> Ham.S=1/2
    >>> Ham.L=1/2
    >>> Ham.lc=20
    >>> sx,sy,sz=epr.JPauli(Ham.S)
    >>> dim=int((2*Ham.S+1)*(2*Ham.L+1))
    >>> print(epr.JLorbit(sx,sy,sz,Ham.lc,dim,Ham.L))
    [[ 5.+0.j  0.+0.j  0.+0.j  0.+0.j]
    [ 0.+0.j -5.+0.j 10.+0.j  0.+0.j]
    [ 0.+0.j 10.+0.j -5.+0.j  0.+0.j]
    [ 0.+0.j  0.+0.j  0.+0.j  5.+0.j]]
    '''
    if l!=0:
        lx,ly,lz=JPauli(l)
        orbe=lamda*(jxn.kron(lx,sx)+jxn.kron(ly,sy)+jxn.kron(lz,sz))
        orbe=jxn.kron(orbe,jxn.eye(int(dim/(orbe).shape[1])))
        return orbe
    else:
        return np.zeros(dim)

def JHfi(ssx,ssy,ssz,iix,iiy,iiz,at,dim):
    '''
    Function for the hiperfine interaction.
    
    Parameters
    ----------
    ssx : jax.np.array
        Pauli matrix of the spin x component.
    ssy : jax.np.array
        Pauli matrix of the spin y component.
    ssz : jax.np.array
        Pauli matrix of the spin z component.
    iix : jax.np.array
        Pauli matrix of the nuclear spin x component.
    iiy : jax.np.array
        Pauli matrix of the nuclear spin y component.
    iiz : jax.np.array
        Pauli matrix of the nuclear spin z component.
    at : jax.np.array
        Array for the hiperfine interaction constant.
    dim : int
        Dimension of the total hamiltonian.
        
    Returns
    -------
    
    ta : jax.np.array
        Matrix of the hiperfine interaction with dimension dim.


    Example
    -------

    >>> import epraya as epr
    >>> Ham,Exp,_=epr.Jstart()
    >>> Ham.S=1/2
    >>> Ham.I=1/2
    >>> Ham.A=[200,300,200]
    >>> Ham=epr.Jchaframe(Ham,Exp)
    >>> sx,sy,sz=epr.JPauli(Ham.S)
    >>> ix,iy,iz=epr.JPauli(Ham.S)
    >>> dim=int((2*Ham.S+1)*(2*Ham.I+1))
    >>> print(epr.JHfi(sx,sy,sz,ix,iy,iz,Ham.A,dim))
    [[ 50.+0.j   0.+0.j   0.+0.j -25.+0.j]
    [  0.+0.j -50.+0.j 125.+0.j   0.+0.j]
    [  0.+0.j 125.+0.j -50.+0.j   0.+0.j]
    [-25.+0.j   0.+0.j   0.+0.j  50.+0.j]]
    '''
    ta=(at[0,0]*jxn.kron(ssx,iix))+(at[0,1]*jxn.kron(ssx,iiy))+(at[0,2]*jxn.kron(ssx,iiz))+(at[1,0]*jxn.kron(ssy,iix))+(at[1,1]*jxn.kron(ssy,iiy))+(at[1,2]*jxn.kron(ssy,iiz))+(at[2,0]*jxn.kron(ssz,iix))+(at[2,1]*jxn.kron(ssz,iiy))+(at[2,2]*jxn.kron(ssz,iiz))
    ta=jxn.kron(ta,jxn.eye(int(dim/(ta).shape[1])))
    return ta

def JHze(ssx,ssy,ssz,g,biel,dim):
    '''

    Function for the Zeeman interaction.
    
    Parameters
    ----------
    
    ssx : jax.np.array
        Pauli matrix of the spin x component.
    ssy : np.array
        Pauli matrix of the spin y component.
    ssz : jax.np.array
        Pauli matrix of the spin z component.
    g : jax.np.array
        Array for the g factor.
    biel : jax.np.array
        Direction of incidence of the magnetic field.

    dim : int
        Dimension of the total hamiltonian.
        
    Returns
    -------
    
    thz : jax.np.array
        Matrix of the Zeeman interaction with dimension dim.

    Example
    -------

    >>> import epraya as epr
    >>> Ham,Exp,_=epr.Jstart()
    >>> Ham.S=1/2
    >>> Ham.g=[2.003,3.0,2.003]
    >>> Ham=epr.Jchaframe(Ham,Exp)
    >>> sx,sy,sz=epr.JPauli(Ham.S)
    >>> dim=int((2*Ham.S+1))
    >>> print(epr.JHze(sx,sy,sz,Ham.g,[0,0,1],dim))
    [[ 1.0015+0.j  0.    +0.j]
    [ 0.    +0.j -1.0015+0.j]]

    '''
    biel=jxn.asarray(biel,dtype=float)
    biel=biel/jxn.where(jxn.linalg.norm(biel)>0,jxn.linalg.norm(biel),1.0)
    hze=biel[0]*(g[0,0]*ssx+g[0,1]*ssy+g[0,2]*ssz)+biel[1]*(g[1,0]*ssx+g[1,1]*ssy+g[1,2]*ssz)+biel[2]*(g[2,0]*ssx+g[2,1]*ssy+g[2,2]*ssz)
    thz=jxn.kron(hze,jxn.eye(int(dim/(hze).shape[1])))
    return thz

def JIee(ssx1,ssy1,ssz1,ssx2,ssy2,ssz2,X,dim):
    '''
    Function for the electron-electron interaction.

    Parameters
    ----------
    
    ssx1 : jax.np.array
        Pauli matrix of the first spin x component.

    ssy1 : jax.np.array
        Pauli matrix of the first spin y component.
        
    ssz1 : jax.np.array
        Pauli matrix of the first spin z component.
        
    ssx2 : np.array
        Pauli matrix of the second spin x component.
    ssy2 : np.array
        Pauli matrix of the second spin y component.
    ssz2 : np.array
        Pauli matrix of the second spin z component.
    
    X : jax.np.array
        Electron-electron interaction constant.

    dim : int
        Dimension of the total hamiltonian.
    
    Returns
    -------

    eet : jax.np.array
        Matrix of the Electron-Electron interaction with dimension dim.
    
    Example
    -------
    >>> import epraya as epr
    >>> import numpy as np
    >>> Ham,_,_=epr.Jstart()
    >>> Ham.S1=1
    >>> Ham.S2=1/2
    >>> sx1,sy1,sz1=epr.JPauli(Ham.S1)
    >>> sx2,sy2,sz2=epr.JPauli(Ham.S2)
    >>> Ham.X1_2=[200,500,200]
    >>> Ham.X1_2=(Ham.X1_2)*np.eye(3)
    >>> dim=int((2*Ham.S1+1)*(2*Ham.S2+1))
    >>> print(epr.JIee(sx1,sy1,sz1,sx2,sy2,sz2,Ham.X1_2,dim))
    [[ 100.     +0.j    0.     +0.j    0.     +0.j -106.06602+0.j
    0.     +0.j    0.     +0.j]
    [   0.     +0.j -100.     +0.j  247.48737+0.j    0.     +0.j
    0.     +0.j    0.     +0.j]
    [   0.     +0.j  247.48737+0.j    0.     +0.j    0.     +0.j
    0.     +0.j -106.06602+0.j]
    [-106.06602+0.j    0.     +0.j    0.     +0.j    0.     +0.j
    247.48737+0.j    0.     +0.j]
    [   0.     +0.j    0.     +0.j    0.     +0.j  247.48737+0.j
    -100.     +0.j    0.     +0.j]
    [   0.     +0.j    0.     +0.j -106.06602+0.j    0.     +0.j
    0.     +0.j  100.     +0.j]]
    '''
    eet=(X[0,0]*np.kron(ssx1,ssx2))+(X[0,1]*np.kron(ssx1,ssy2))+(X[0,2]*np.kron(ssx1,ssz2))+(X[1,0]*np.kron(ssy1,ssx2))+(X[1,1]*np.kron(ssy1,ssy2))+(X[1,2]*np.kron(ssy1,ssz2))+(X[2,0]*np.kron(ssz1,ssx2))+(X[2,1]*np.kron(ssz1,ssy2))+(X[2,2]*np.kron(ssz1,ssz2))
    eet=jxn.kron(eet,jxn.eye(int(dim/(eet).shape[1])))
    return eet

def JQii(iix,iiy,iiz,q,dim):
    '''

    Function for the nuclear quadrupolar interaction.
    
    Parameters
    ----------
    iix : jax.np.array
        Pauli matrix of the nuclear spin x component.
    iiy : jax.np.array
        Pauli matrix of the nuclear spin y component.
    iiz : jax.np.array
        Pauli matrix of the nuclear spin z component.
    q : jax.np.array
        Array for the quadrupolar interaction constant.
    dim : int
        Dimension of the total hamiltonian.
        
    Returns
    -------
    tql : jax.np.array
        Matrix of the nuclear quadrupolar interaction with dimension dim.

    Example
    -------

    >>> import epraya as epr 
    >>> Ham,Exp,_=epr.Jstart()
    >>> Ham.I=1
    >>> Ham.Q=[0.5,10.0,0]
    >>> Ham=epr.Jchaframe(Ham,Exp)
    >>> ix,iy,iz=epr.JPauli(Ham.I)
    >>> dim=int((2*Ham.I+1))
    >>> print(epr.JQii(ix,iy,iz,Ham.Q,dim))
    [[ 0.  +0.j         -4.75+0.49999997j  0.  +0.j        ]
    [-4.75-0.49999997j  0.  +0.j         -4.75+0.49999997j]
    [ 0.  +0.j         -4.75-0.49999997j  0.  +0.j        ]]
    '''
    hql=(q[0,0]*(iix@iix))+(q[1,1]*(iiy@iiy))+(q[2,2]*(iiz@iiz))+(q[0,1]*((iix@iiy)+(iiy@iix)))+(q[1,2]*((iiy@iiz)+(iiz@iiy)))+(q[2,0]*((iiz@iix)+(iix@iiz)))
    tql=jxn.kron(jxn.eye(int(dim/(hql).shape[1])),hql)
    return tql


def JNhze(I,iix,iiy,iiz,dim,gn,direction=[0,0,1]):
    '''
    Function for the nuclear Zeeman interaction.
    
    Parameters
    ----------
    
    I : float
        Nuclear spin value.
    iix : jax.np.array
        Pauli matrix of the nuclear spin x component.
    iiy : jax.np.array
        Pauli matrix of the nuclear spin y component.
    iiz : jax.np.array
        Pauli matrix of the nuclear spin z component.
    dim : int
        Dimension of the total hamiltonian.
    gn : float
        Nuclear g factor of the element of the paramagnetic center.
    direction : jax.np.array
        Direction of incidence of the magnetic field.

    Returns
    -------
    
    nhz : jax.np.array
        Matrix of the Zeeman nuclear interaction with dimension dim.

    Example
    -------

    >>> import epraya as epr 
    >>> Ham,Exp,_=epr.Jstart()
    >>> Ham.I=1
    >>> gn=epr.gnfactor('55Mn')
    >>> ix,iy,iz=epr.JPauli(Ham.I)
    >>> dim=int((2*Ham.I+1))
    >>> print(epr.JNhze(Ham.I,ix,iy,iz,dim,gn,[0,0,1]))
    [[ 1.3813+0.j  0.    +0.j  0.    +0.j]
    [ 0.    +0.j  0.    +0.j  0.    +0.j]
    [ 0.    +0.j  0.    +0.j -1.3813+0.j]]
    '''
    nd=jxn.asarray(direction,dtype=float)
    nd=nd/jxn.where(jxn.linalg.norm(nd)>0,jxn.linalg.norm(nd),1.0)
    direct=nd[0]*iix+nd[1]*iiy+nd[2]*iiz
    nhz=gn*direct
    nhz=jxn.kron(jxn.eye(int(dim/(nhz).shape[1])),nhz)
    return nhz

def gnfactor(Nucl='None'):
    '''
    Function to read the nuclear g factor from the added table.
    
    Parameter
    ---------
    
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    
    Returns
    -------
    
    gn : float
        Nuclear g factor for the Nucl element.
    '''
    if Nucl!='None':
        route=resources.files(__package__).joinpath("nucleardaat.txt")
        krle=read_csv(route,header=0,sep='\t')
        deq=krle[krle['Symbol']==Nucl]
        return float(deq['gN_factor'].values[0])
    else:
        return 0.0

def Jchaframe(Ham,Exp):
    '''
    Adds the rotations for the constants g, A, Q and D, described by the conditions gframe, Aframe, Qframe and D.frame.
    
    Parameters
    ----------
    
    Ham : Class
        Container of the Hamiltonian parameters.

    Exp : Class
        Container of the Experimental conditions.

    Returns
    -------
    
    Ham : Class
        Container of the Hamiltonian parameters, with the corrected values.
    
    Example
    -------

    >>> import epraya as epr 
    >>> Ham,Exp,_=epr.Jstart()
    >>> Ham.g=[2.0003,2.5,2.5]
    >>> Ham.Q=[0.5,10.0,0]
    >>> print(Ham)
    >>> Ham=epr.Jchaframe(Ham,Exp)
    >>> print(Ham)
    JHval(S=0.5, g=[2.0003, 2.5, 2.5], I=0.0, L=0.0, A=0.0,
    Q=[0.5, 10.0, 0], D=Array([0, 0], dtype=int32), 
    Bk2=[0, 0, 0, 0, 0], Bk4=[0, 0, 0, 0, 0, 0, 0, 0, 0], 
    Bk6=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], lc=0.0,
    Hpp=Array([0, 1], dtype=int32), eta=0.5, weight=0.0)
    JHval(S=0.5, g=Array([[ 2.1690652 ,  0.16222095,  0.20332736],
    [ 0.16222097,  2.7222438 , -0.09966913],
    [ 0.20332736, -0.09966913,  2.3750749 ]], dtype=float32),
    I=0.0, L=0.0, A=Array([[0., 0., 0.],[0., 0., 0.],[0., 0., 0.]],
    dtype=float32), Q=Array([[ 0.5,  0. ,  0. ],[ 0. , 10. ,  0. ],
    [ 0. ,  0. ,  0. ]], dtype=float32), D=Array([[0., 0., 0.],
    [0., 0., 0.],[0., 0., 0.]], dtype=float32), Bk2=[0, 0, 0, 0, 0], 
    Bk4=[0, 0, 0, 0, 0, 0, 0, 0, 0], 
    Bk6=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], lc=0.0, 
    Hpp=Array([0, 1], dtype=int32), eta=0.5, weight=0.0)
    '''
    Ham=JConvtarray(Ham)
    tA=jxn.eye(3)*Ham.A
    tQ=jxn.eye(3)*Ham.Q
    tg=jxn.eye(3)*Ham.g
    D2s=jxn.asarray([-Ham.D[0]/3+Ham.D[1],-Ham.D[0]/3-Ham.D[1],2*Ham.D[0]/3])
    tD=jxn.eye(3)*D2s
    rot=JRotationmat(Exp)
    A1=(JRotmatrix(Exp.Aframe[0],Exp.Aframe[1],Exp.Aframe[2])@rot).T@tA@(JRotmatrix(Exp.Aframe[0],Exp.Aframe[1],Exp.Aframe[2])@rot)
    g1=(JRotmatrix(Exp.gframe[0],Exp.gframe[1],Exp.gframe[2])@rot).T@tg@(JRotmatrix(Exp.gframe[0],Exp.gframe[1],Exp.gframe[2])@rot)
    D2=(JRotmatrix(Exp.Dframe[0],Exp.Dframe[1],Exp.Dframe[2])@rot).T@tD@(JRotmatrix(Exp.Dframe[0],Exp.Dframe[1],Exp.Dframe[2])@rot)
    Q1=(JRotmatrix(Exp.Qframe[0],Exp.Qframe[1],Exp.Qframe[2])@rot).T@tQ@(JRotmatrix(Exp.Qframe[0],Exp.Qframe[1],Exp.Qframe[2])@rot)
    return Ham.replace(A=A1,g=g1,D=D2,Q=Q1)

def JRotationmat(Exp):
    '''
    Creates the rotation matrix to pass from the Lab frame to the molecular frame.
    
    Parameters
    ---------
    
    Exp : Class
        Container for the experimental conditions.

    Returns
    -------
    
    RRmatrix: jax.np.array
        Matrix for the rotation.

    
    Example
    -------
    >>> import epraya as epr 
    >>> _, Exp, _=epr.Jstart()
    >>> Exp.Sampleframe=[30,20,0]
    >>> Exp.Molframe=[60,0,0]
    >>> print(epr.JRotationmat(Exp))
    [[ 0.40355888  0.83092371 -0.38302222]
    [-0.7851017   0.52945382  0.3213938 ]
    [ 0.46984631  0.17101007  0.8660254 ]]
    '''
    RRmatrix=JRotmatrix(Exp.Sampleframe[0],Exp.Sampleframe[1],Exp.Sampleframe[2])@JRotmatrix(Exp.Molframe[0],Exp.Molframe[1],Exp.Molframe[2])
    return RRmatrix.T

def JRotmatrix(alfa,beta,gamma):
    '''
    Creates the rotation matrix based in the Euler angles in degrees.


    Parameters
    ----------

    alfa : float
        Euler angle for rotations around the z axis
    beta : float

        Euler angle for rotations around the y' axis
    gamma : float
        Euler angle for rotations around the z'' axis


    Returns
    -------

    Reuler : np.array
        Rotation matrix

    Example
    -------

    >>> import epraya as epr
    >>> alfa,beta,gamma=20,30,40
    >>> print(epr.JRotmatrix(alfa,beta,gamma))
    [[ 0.40355885  0.96394587 -0.38302222]
    [-0.78510165  0.5294539   0.3213938 ]
    [ 0.4698463   0.17101006  0.8660254 ]]
    '''
    alfa,beta,gamma=jxn.radians(alfa),jxn.radians(beta),jxn.radians(gamma)
    cosg,sing=jxn.cos(gamma),jxn.sin(gamma)
    cosa,sina=jxn.cos(alfa),jxn.sin(alfa)
    cosb,sinb=jxn.cos(beta),jxn.sin(beta)
    eps=10**-9
    cosg=jxn.where(jxn.abs(cosg)<eps,0.0,cosg)
    sing=jxn.where(jxn.abs(sing)<eps,0.0,sing)
    cosa=jxn.where(jxn.abs(cosa)<eps,0.0,cosa)
    sina=jxn.where(jxn.abs(sina)<eps,0.0,sina)
    cosb=jxn.where(jxn.abs(cosb)<eps,0.0,cosb)
    sinb=jxn.where(jxn.abs(sinb)<eps,0.0,sinb)
    Reuler=jxn.array([[(cosg*cosa*cosb)-(sing*sina),(cosg*cosb*sina)+(sing*cosa),-cosg*sinb],
    [-(sing*cosb*cosa)-(cosg*sina),-(sing*cosb*sina)+(cosg*cosa),sing*sinb],
    [sinb*cosa,sina*sinb,cosb]])
    return Reuler

def JConvtarray(Ham):
    '''
    Conditional function to assure that the hamiltonian parameters have the right dimension.

    Parameters
    ----------

    Ham : Class

        Container of the Hamiltonian parameters.

    Returns
    -------

    Ham : Class
        Container of the Hamiltonian parameters with the corrected parameters.
    '''
    def formatj(val,variable):
        iti=jxn.asarray(val,dtype=float)
        if iti.ndim==0:
            return iti*jxn.array([1.0,1.0,1.0])
        elif iti.ndim==1:
            if iti.shape[0]==3:
                return iti
            elif iti.shape[0]==2:
                return jxn.array([iti[0],iti[0],iti[1]])
            else:
                raise RuntimeError(f"Not enough/too many values in {variable}, expected 2 or 3, got {iti.shape[0]}.")

        elif iti.ndim==2:
            if iti.shape!=(3,3):
                raise RuntimeError(f"Matrix {variable} doesn't have the correct dimensions, must be (3,3).")
            return iti

        else:
            raise RuntimeError(f"Wrong data dimensions in {variable}.")
    hg=formatj(Ham.g,"Hval.g")
    hA=formatj(Ham.A,"Hval.A")
    hD=jxn.asarray(Ham.D,dtype=float)
    if hD.ndim!=1:
        raise RuntimeError("Wrong data type/dimensions in Hval.D, must be 1D list or array.")
    if hD.shape[0]==2:
        hD=hD
    elif hD.shape[0]==3:
        hD=jxn.array([3.0*hD[2]/2.0,(hD[0]-hD[1])/2.0])
    else:
        raise RuntimeError(f"Wrong number of values in Hval.D, expected 2 or 3, got {hD.shape[0]}.")

    hQ=jxn.asarray(Ham.Q,dtype=float)
    if hQ.ndim!=1 or hQ.shape[0]!=3:
        raise RuntimeError(f"Wrong values in Hval.Q, expected exactly 3, got {hQ.shape[0] if hQ.ndim==1 else 'matrix'}.")
    hQ=hQ
    return Ham.replace(g=hg,A=hA,D=hD,Q=hQ)

#Stevens Operators
#Rule: k<=2s
def JStevensO(ssx,ssy,ssz,s,Ham,dim):
    '''
    Expanded Stevens operators following the rule k<=2s and the definition by Rudowicz and Chung. Variable Ham.Bk2 takes precedence over Ham.D, so if both are defined, only the value of Ham.Bk2 is used.
	
    Parameters
    ----------
    
    ssx : jax.np.array
        Pauli matrix of the spin x component.
    ssy : jax.np.array
        Pauli matrix of the spin y component.
    ssz : jax.np.array
        Pauli matrix of the spin z component.
    s : float
        Spin operator value.
    Ham : Class
        Hamiltonian parameters container.
    dim : int
        Dimension of the total hamiltonian.
        
    Returns
    -------
    
    totales : jax.np.array
        Matrix with the contribution of the relevant Stevens operators.
        
    Example
    -------
    >>> import epraya as epr
    >>> Ham, _, _ = epr.Jstart()
    >>> Ham.S=1
    >>> Ham.D=[700,200]
    >>> Ham=epr.Jchaframe(Ham,Exp)
    >>> dim=int(2*Ham.S+1)
    >>> sx,sy,sz=epr.JPauli(Ham.S)
    >>> print(epr.JStevensO(sx,sy,sz,Ham.S,Ham,dim))
    [[ 233.33333+0.j    0.     +0.j  199.99998+0.j]
    [   0.     +0.j -466.66666+0.j    0.     +0.j]
    [ 199.99998+0.j    0.     +0.j  233.33333+0.j]]
    '''
    k=int(2*s)
    bk2=jxn.asarray(Ham.Bk2,dtype=float)  
    usebk2=jxn.any(bk2!=0)
    B22=jxn.where(usebk2,bk2[0],(Ham.D[0,0]-Ham.D[1,1])/2)
    B21=jxn.where(usebk2,bk2[1],Ham.D[0,2])
    B20=jxn.where(usebk2,bk2[2],3*Ham.D[2,2]/2)
    Bq21=jxn.where(usebk2,bk2[3],Ham.D[1,2])
    Bq22=jxn.where(usebk2,bk2[4],Ham.D[0,1])
    B44,B43,B42,B41,B40,Bq41,Bq42,Bq43,Bq44=Ham.Bk4
    B66,B65,B64,B63,B62,B61,B60,Bq61,Bq62,Bq63,Bq64,Bq65,Bq66=Ham.Bk6
    sour=(int(k-k%2))
    if sour==0:
        return jxn.kron(jxn.zeros((int(2*s+1),int(2*s+1))),jxn.eye(int(dim/(jxn.zeros((int(2*s+1),int(2*s+1)))).shape[1])))
    else:
        k=np.arange(0,sour+1,2)[1:]
        sxminus=ssx-(ssy*1.0j)
        sxsum=ssx+(ssy*1.0j)
        sxminus2=jxn.linalg.matrix_power(sxminus,2)
        sxsum2=jxn.linalg.matrix_power(sxsum,2)
        ssz2=jxn.linalg.matrix_power(ssz,2)
        xs=s*(s+1)
        eye=jxn.eye(int(2*s+1),dtype=complex)
        if k[-1]>=4:
            ssz3=jxn.linalg.matrix_power(ssz,3)
            ssz4=jxn.linalg.matrix_power(ssz,4)
            sxminus3=jxn.linalg.matrix_power(sxminus,3)
            sxsum3=jxn.linalg.matrix_power(sxsum,3)
            sxminus4=jxn.linalg.matrix_power(sxminus,4)
            sxsum4=jxn.linalg.matrix_power(sxsum,4)
        if k[-1]>=6:
            ssz6=jxn.linalg.matrix_power(ssz,6)
            ssz5=jxn.linalg.matrix_power(ssz,5)
            sxminus5=jxn.linalg.matrix_power(sxminus,5)
            sxsum5=jxn.linalg.matrix_power(sxsum,5)
            sxminus6=jxn.linalg.matrix_power(sxminus,6)
            sxsum6=jxn.linalg.matrix_power(sxsum,6)
        total=0
        for ris in range(0,len(k)):
            if (k[ris])==2:
                #q=0
                temp=B20*(3*ssz2-xs*eye)/3
                #q=1
                temp+=B21*(ssx@ssz+ssz@ssx)
                #q=-1
                temp+=Bq21*(ssy@ssz+ssz@ssy)
                #q=2
                temp+=B22*0.5*(sxminus2+sxsum2)
                #q=-2
                temp+=(1j)*(Bq22*0.5*(sxminus2-sxsum2))
                total+=temp
            if (k[ris])==4:
                #q=0
                temp=B40*((35*ssz4)-(((30*xs)-25)*ssz2)+(eye*((-6*xs)+(3*xs**2))))
                #q=1
                temp+=B41*(0.25)*((35*ssz3-(30*xs*ssz)+25*ssz)@(sxsum+sxminus)+(sxsum+sxminus)@(35*ssz3-(30*xs*ssz)+25*ssz))
                #q=-1
                temp+=(-1j)*Bq41*(0.25)*((35*ssz3-(30*xs*ssz)+25*ssz)@(sxsum-sxminus)+(sxsum-sxminus)@(35*ssz3-(30*xs*ssz)+25*ssz))
                #q=2
                temp+=B42*0.25*(((7*ssz2-xs*eye-5*eye)@(sxsum2+sxminus2))+((sxsum2+sxminus2)@(7*ssz2-xs*eye-5*eye)))
                #q=-2
                temp+=Bq42*0.25*(-1j)*(((7*ssz2-xs*eye-5*eye)@(sxsum2-sxminus2))+((sxsum2-sxminus2)@(7*ssz2-xs*eye-5*eye)))
                #q=3
                temp+=B43*0.25*((ssz@(sxminus3+sxsum3))+((sxminus3+sxsum3)@ssz))
                #q=-3
                temp+=(-1j)*Bq43*0.25*((ssz@(sxsum3-sxminus3))+((sxsum3-sxminus3)@ssz))
                #q=4
                temp+=B44*0.5*(sxminus4+sxsum4)
                #q=-4
                temp+=(-1j)*Bq44*0.5*(sxsum4-sxminus4)
                total+=temp
            if (k[ris])==6:
                #q=0
                temp=B60*((231*ssz6)-(315*xs*ssz4)+735*ssz4+(105*xs*xs*ssz2)-(525*xs*ssz2)+(294*ssz2)-(5*xs**3*eye)+(40*xs**2*eye)-60*xs*eye)
                #q=1
                temp+=B61*0.25*(((231*ssz5)-(315*xs-735)*ssz3+(105*xs**2-525*xs+294)*ssz)@(sxsum+sxminus)+(sxsum+sxminus)@((231*ssz5)-(315*xs-735)*ssz3+(105*xs**2-525*xs+294)*ssz))
                #q=-1
                temp+=(-1j)*Bq61*0.25*(((231*ssz5)-(315*xs-735)*ssz3+(105*xs**2-525*xs+294)*ssz)@(sxsum-sxminus)+(sxsum-sxminus)@((231*ssz5)-(315*xs-735)*ssz3+(105*xs**2-525*xs+294)*ssz))
                #q=2
                temp+=0.25*B62*((33*ssz4-18*xs*ssz2-123*ssz2+(xs**2*eye)+10*xs*eye+102*eye)@(sxminus2+sxsum2)+(sxminus2+sxsum2)@(33*ssz4-18*xs*ssz2-123*ssz2+(xs**2*eye)+10*xs*eye+102*eye))
                #q=-2
                temp+=(-1j)*0.25*Bq62*((33*ssz4-18*xs*ssz2-123*ssz2+(xs**2*eye)+10*xs*eye+102*eye)@(sxminus2-sxsum2)+(sxminus2-sxsum2)@(33*ssz4-18*xs*ssz2-123*ssz2+(xs**2*eye)+10*xs*eye+102*eye))
                #q=3
                temp+=B63*0.25*(((11*ssz3-(3*xs*ssz)-(59*ssz))@(sxminus3+sxsum3))+((sxminus3+sxsum3)@(11*ssz3-(3*xs*ssz)-(59*ssz))))
                #q=-3
                temp+=(-1j)*Bq63*0.25*(((11*ssz3-(3*xs*ssz)-(59*ssz))@(sxsum3-sxminus3))+((sxsum3-sxminus3)@(11*ssz3-(3*xs*ssz)-(59*ssz))))
                #q=4
                temp+=B64*0.25*(((11*ssz2-xs*eye-38*eye)@(sxminus4+sxsum4))+((sxminus4+sxsum4)@(11*ssz2-xs*eye-38*eye)))
                #q=-4
                temp+=(-1j)*Bq64*0.25*(((11*ssz2-xs*eye-38*eye)@(sxsum4-sxminus4))+((sxsum4-sxminus4)@(11*ssz2-xs*eye-38*eye)))
                #q=5
                temp+=B65*0.25*(ssz@(sxsum5+sxminus5)+(sxsum5+sxminus5)@ssz)
                #q=-5
                temp+=(-1j)*Bq65*0.25*(ssz@(sxsum5-sxminus5)+(sxsum5-sxminus5)@ssz)
                #q=6
                temp+=B66*0.5*(sxminus6+sxsum6)
                #q=-6
                temp+=(-1j)*Bq66*0.5*(sxsum6-sxminus6)
                total+=temp
        totales=jxn.kron(total,jxn.eye(int(dim/(total).shape[1])))
        return totales

def JStevensrot(sx,sy,sz,s,Ham,dim,Exp):
    '''
    Stevens operators with all the terms in the same frame: k=2 from the D tensor rotated in Jchaframe,
    k=4 and k=6 built in the crystal frame and rotated in spin space with the same matrix.
    '''
    H2=Ham.replace(Bk2=[0]*5,Bk4=[0]*9,Bk6=[0]*13)
    H46=Ham.replace(D=jxn.zeros((3,3)))
    h2=JStevensO(sx,sy,sz,s,H2,dim)
    h46=JStevensO(sx,sy,sz,s,H46,dim)
    M=np.asarray(Rotationmat(Exp),float)           
    if s>=2 and not np.allclose(M,np.eye(3)):
        rv=Rotation.from_matrix(M).as_rotvec()
        Us=jexpm(-1j*(rv[0]*sx+rv[1]*sy+rv[2]*sz))
        U=jxn.kron(Us,jxn.eye(dim//Us.shape[0]))      # acts only on the electron spin
        h46=U.conj().T@h46@U
    return h2+h46


def JMsmi(I,S,L=0):
    '''
    Determinates the quantum numbers for the spin, nuclear spin and angular momentum.

    Parameters
    ----------
    I : float
        Nuclear spin operator value.
    S : float
        Spin operator value.
    L : float
        Angular momentum operator value.

    Returns
    -------

    sl : jax.np.array
        Quantum numbers of the spin operator.
    nl : jax.np.array
        Quantum numbers of the nuclear spin operator.
    ll : jax.np.array
        Quantum numbers of the angular momentum operator

    Example
    -------

    >>> import epraya as epr
    >>> Ham, _, _ = epr.Jstart()
    >>> Ham.S=1
    >>> Ham.I=1
    >>> Ham.L=0
    >>> print(epr.JMsmi(Ham.I,Ham.S,Ham.L))
    (Array([ 1.,  1.,  1.,  0.,  0.,  0., -1., -1., -1.], dtype=float32),
    Array([ 1.,  0., -1.,  1.,  0., -1.,  1.,  0., -1.], dtype=float32), 
    Array([0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32))
    '''
    if I<0 or S<0 or L<0:
        raise ValueError('Spin values cannot be negative')
    dim=int(2*S+1)*int(2*I+1)*int(2*L+1)
    poss=jxn.linspace(S,-S,int(2*S+1))
    posi=jxn.linspace(I,-I,int(2*I+1))
    posl=jxn.linspace(L,-L,int(2*L+1))
    ll,sl,il=jxn.meshgrid(posl,poss,posi,indexing='ij')
    return sl.flatten(),il.flatten(),ll.flatten()



#For powder
@jx.jit
def JLorentzp(field,Int,rfield,Hpp):
    '''
    Defines the lorentzian profile of the spectrum, using the resonant fields and calculated intensities.
    
    Parameters
    ----------
    
    field : jax.np.array
        Array of the field values of the spectrum.

    Int : jax.np.array
        Array of the intensity calculated in the *field* values 
    rfield : jax.np.array
        List of the resonant fields of the system.

    Hpp : float
        Peak to peak distance in mT.

    Returns
    -------
    
    espec : jax.np.array

        EPR spectrum
    
    Example
    -------
    .. code-block:: python
    
       import matplotlib.pyplot as plt
       import epraya as epr
       import numpy as np
       Ham,Exp,_=epr.Jstart()
       Ham.Hpp=[0,2]
       Exp.Frange=[200,500]
       Exp.Points=2000
       fieldr=np.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
       mu=350
       sigma=50
       rsonant=np.array([300,350,320])
       inten=np.random.normal(mu,sigma,len(rsonant))
       spc=epr.JLorentzp(fieldr,inten,rsonant,Ham.Hpp[1])

       plt.plot(fieldr,spc)
       plt.grid()
       plt.show()    
       
    .. image:: /_static/jlorentz.png
       :alt: Plot of the JLorentzp function
       :align: center
    '''
    espec=jxn.zeros(len(field),dtype=float)
    gamma=jxn.sqrt(3.0)*Hpp
    gamma2=gamma/2.0
    dif=field[:,None]-rfield[None,:]
    ert=(-dif*gamma/jxn.pi)/((dif**2+gamma2**2)**2)
    espec=jxn.sum(Int*ert,axis=1)
    return espec
@jx.jit
def JGaussp(field,Int,rfield,Hpp):
    '''
    Defines the gaussian profile of the spectrum, using the resonant fields and calculated intensities.
    
    Parameters
    ----------
    
    field : jax.np.array
        Array of the field values of the spectrum.
    Int : jax.np.array
        Array of the intensity calculated in the *field* values 
    rfield : jax.np.array
        List of the resonant fields of the system.
    Hpp : float
        Peak to peak distance in mT.
    
    Returns
    -------
    
    espec : jax.np.array
        EPR spectrum
    
    Example
    -------
    .. code-block:: python
       
       import matplotlib.pyplot as plt
       import epraya as epr
       import numpy as np
       Ham,Exp,_=epr.Jstart()
       Ham.Hpp=[2,0]
       Exp.Frange=[200,500]
       Exp.Points=2000
       fieldr=np.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
       mu=350
       sigma=50
       rsonant=np.array([300,350,320])
       inten=np.random.normal(mu,sigma,len(rsonant))
       spc=epr.JGaussp(fieldr,inten,rsonant,Ham.Hpp[0])

       plt.plot(fieldr,spc)
       plt.grid()
       plt.show()  
       
    .. image:: /_static/Jgauss.png
       :alt: Plot of the JGaussp function
       :align: center
    '''

    espec=jxn.zeros(len(field),dtype=float)
    gamma=jxn.sqrt(jxn.log(2.0)/2.0)*Hpp
    ymax=jxn.sqrt(jxn.log(2.0)/jxn.pi)*(1/gamma)
    dif=field[:,None]-rfield[None,:]
    ert=((-ymax*2.0*jxn.log(2.0)*dif)/gamma**2)*jxn.exp(-jxn.log(2.0)*dif**2/gamma**2)
    espec=jxn.sum(Int*ert,axis=1)
    return espec

@jx.jit
def JVoigtp(sketch,dB,Hpp):
    '''
    Determinates the voigtian profile of the spectrum by convolution using dB as step of field.
    
    Parameters
    ----------

    sketch : jax.np.array
        Initial sketch of the spectrum.
    dB : float
		Differential of field for convolution.
    Hpp : list
        Peak to peak distance in mT, for gaussian and lorentzian profiles.
    
    Returns
    -------

    espec : jax.np.array
        EPR spectrum
    
    '''
    n=sketch.shape[0]
    npad=2*n
    k=2*jxn.pi*jxn.arange(npad//2+1)/(npad*dB)
    H=(1j*k)*jxn.exp(-0.5*((Hpp[0]/2.0)*k)**2-(jxn.sqrt(3.0)/2.0*Hpp[1])*jxn.abs(k))
    return jxn.fft.irfft(jxn.fft.rfft(sketch,n=npad)*H,n=npad)[:n]/dB

#Find energy values in function of field
@partial(jx.jit, static_argnames=['hifi'])
def JPadaptarray(espac,h1,hx,hy,hz,nx,ny,nz,hifi=False):
    '''
    Constructs the Zeeman hamiltonian, adds it to the complete one and finds the energy values and eigenvectors.
    
    Parameters
    ----------
    
    espac : np.array
        Array with the values of the magnetic field.
    h1 : np.array
        Hamiltonian matrix that contains all no Zeeman interactions.
    hx : np.array
        Hamiltonian matrix of the Zeeman interaction in the x direction.
    hy : np.array
        Hamiltonian matrix of the Zeeman interaction in the y direction.
    hz : np.array
        Hamiltonian matrix of the Zeeman interaction in the z direction.
    nx : float
        Weight coefficient for the interaction in the x direction.
    ny : float
        Weight coefficient for the interaction in the y direction.
    nz : float
        Weight coefficient for the interaction in the z direction.
    hifi : bool
        Bool to calculate the Jacobian matrix.
    Returns
    -------

    Elist : np.array
        Array of the energy values of the hamiltonian.
    Vlist : np.array
        Array of the eigenvectors of the hamiltonian.
    h2 : np.array
        Total Zeeman interaction matrix.
    '''
    h2=nx*hx+ny*hy+nz*hz
    h3=h1[None,:,:]+h2[None,:,:]*espac[:,None,None]

    Elist,Vlist=containeigh(h3,hifi=hifi)
    return Elist,Vlist,h2
    
# Makes the approximation by the assigment problem solution
def Hungarian(cost):
    '''
    Solves the assigment problem with the J-V method implemented in scipy, to organize the eigenvectors and energies of the hamiltonian and relate them with the quantum numbers, depending of the change in the cost.
    
    Parameters
    -----------
    
    cost : list
        Cost matrix of the eigenvectors and energies configuration
        
    Returns 
    -------
    
    novo : array
        Sorted eigenvectors and energies of the hamiltonian
    '''
    rowidx,colidx=sci.optimize.linear_sum_assignment(np.array(cost))
    novo=colidx[np.argsort(rowidx)]
    return novo.astype(np.int32)

def Jungarian(cost):
    '''
    Wrap function to call the Scipy J-V method outside of JAX using ShapeDtypeStruct and pure_callback, in order to not interfere in the ADAM implementation.
    
    Parameters
    -----------
    
    cost : list
        Cost matrix of the eigenvectors and energies configuration
    
    Returns
    
    val : function
        Pure call back of the hungarian function.
        
    '''
    shake=jx.ShapeDtypeStruct((cost.shape[0],),jxn.int32)
    return jx.pure_callback(Hungarian,shake,cost,vmap_method='sequential')

@jx.jit
def JPretrack(Enegria, Vector):
    '''
    Organize the eigenvectors and energies to relate them with the quantum numbers of the system, taking as references the values at high field. It use the laxscan function from JAX to calculate the best configuration.
    
    Parameters
    ----------
    
    Enegria : jax.np.array
        Array of the energy values of the hamiltonian.
    Vector : jax.np.array
        Array of the eigenvectors of the hamiltonian.
    

    Returns
    -------
    tE : jax.numpy.vstack
        Concatenate array of the energy values of the Hamiltonian.
        
    tV : jax.numpy.vstack
        Concatenate array of the eigenvectors values of the Hamiltonian.
    '''
    def tmback(cara,vals):
        oldE,oldV=cara
        actE,actV=vals
        supermatrix=jxn.abs(jxn.dot(oldV.conj().T,actV))
        cost=1.0-supermatrix
        Edif=jxn.abs(oldE[:,None]-actE[None,:])
        maxEdif=jxn.max(Edif)
        maxEdif=jxn.where(maxEdif==0.0,1e-10,maxEdif)
        coste=jxn.where(maxEdif>0,Edif/maxEdif,0.0)
        tot=cost+0.1*coste
        idx=Jungarian(jx.lax.stop_gradient(tot))
        sorE=actE[idx]
        sorV=actV[:,idx]
        return (sorE,sorV),(sorE,sorV)

    eincar=(Enegria[-1],Vector[-1])
    restof=(Enegria[:-1],Vector[:-1])
    _,(tre,trv)=jx.lax.scan(tmback,eincar,restof,reverse=True)
    tE=jxn.vstack([tre,Enegria[-1:]])
    tV=jxn.vstack([trv,Vector[-1:]])
    return tE,tV

@jx.jit
def JBoltfactor(Eghz,di,dj,Temp):
    '''

    Calculates the Boltzmann distribution for the intensity, using states *di* and *dj* and their related energies.
    
    Parameters
    ----------

    Eghz : float
        Approximated energy value of hamiltonian.
    di : float
        State di of the system.
    dj : float
        State dj of the system.
    Temp : float
        Temperature of the system.
    
    Returns
    -------
    
    popui-popuj : jax.np.array
        Temperature dependent Boltzmann distribution.

    '''
    h=scc.h
    kb=scc.k
    conver=1e9*h 
    Ej=Eghz*conver
    Temp=jxn.where(Temp<=0.0,1.0,Temp)
    beta=1.0/(kb*Temp)
    Emin=jxn.min(Ej)
    boltz=jxn.exp(-beta*(Ej-Emin))
    Z=jxn.sum(boltz)
    popui=boltz[di]/Z
    popuj=boltz[dj]/Z
    return jxn.where(Temp<=0.0,1.0,jxn.abs(popui-popuj))

@partial(jx.jit,static_argnames=['dim','hifi','ktra'])
def JNresina(Blist,Elist,Vlist,dim,Freq,isx,isy,isz,nx,ny,nz,Tem,Hpp,h2,hifi=False,ktra=None,mw=None):
    '''
    Determinates the resonant fields and intensities of the spectrum using the expression for the first order perturbation limit. The intensity is calculated as the product of the transition rate (probability of transition), the Boltzmann factor (Boltzmann distribution) and a frecuency to field conversion factor. 
    
    Initially, the function calculates the intensities for all the block of orientations, then finds the resonant fields searching for the values that comply with the resonant equation and have a crossing. Finally it relates the resonant fields and intensity values depending in the probability of transitions, producing the arrays of intensity, fields and the total number of transitions.
    
    Parameters
    ----------
    
    Blist : jax.np.array
        Magnetic field values that are evaluated to find the resonant fields.
    Elist : jax.np.array
        List of the energies from the hamiltonian.
    Vlist : jax.np.array
        List of the eigenvectors of the hamiltonian.
    dim : float
        Dimension of the total hamiltonian matrix.
    Freq : float
        Frequency of operation of the microwave radiation in GHz.
    isx : jax.np.array
        Pauli matrix of the total spin (electronic, nuclear spin and angular momentum) contribution in the x direction.
    isy : jax.np.array
        Pauli matrix of the total spin (electronic, nuclear spin and angular momentum) contribution in the y direction.
    isz : jax.np.array
        Pauli matrix of the total spin (electronic, nuclear spin and angular momentum) contribution in the z direction.
    nx : jax.np.array 
        Vectors (points in the grid) to consider in the calculation of the spectrum in the x direction.
    ny : jax.np.array 
        Vectors (points in the grid) to consider in the calculation of the spectrum in the y direction.
    nz : jax.np.array 
        Vectors (points in the grid) to consider in the calculation of the spectrum in the z direction.
    Tem : float
        Temperature of the system.
    h2 : jax.np.array
        Zeeman interaction matrix of the system.
    hifi : bool
        Bool condition for the Jacobian calculation.
    ktra : float
        Reference value of number of transitions.
    mw : jax.np.array
        Unit vector of the microwave field B1 (single crystal). If None, B1 is averaged over the plane perpendicular to B0 (powder).
    
    Returns
    -------
	
    ffres : jax.np.array 
        List of the resonant fields of the system.
    ffint : jax.np.array 
        List of the intensity of the spectrum, evaluated in the resonant fields.
    ntrans : jax.np.array
        Zero dimensional array with the number of possible transitions related to the resonant fields.
    '''
    iidx,jidx=jxn.triu_indices(dim,k=1)
    npairs=iidx.shape[0]
    #Find energy crossings first
    diffv=jxn.abs(Elist[:,jidx]-Elist[:,iidx])-Freq
    dEl,dEr=diffv[:-1],diffv[1:]
    cross=(dEl*dEr<=0.0)&(dEl!=dEr)
    denom=dEr-dEl
    denom=jxn.where(denom==0.0,1e-8,denom)
    t=jxn.where(cross,-dEl/denom,0.0)
    Bl=Blist[:-1,None]
    Br=Blist[1:,None]
    res=Bl+t*(Br-Bl)
    cross=cross&(res>(Blist[0]+2.0))
    ntrans=jxn.sum(cross).astype(jxn.float64)
    #Scores for transition possibility
    Ktra=min(ktra or (2500 if hifi else 1000),cross.size)
    scores=jxn.where(cross.reshape(-1),1.0,-1.0)
    topv,idx=jx.lax.top_k(scores,Ktra)
    idx=jx.lax.stop_gradient(idx)
    valid=topv>0.0
    #Calculates intensity in just the crossings
    fieldrange=idx//npairs     
    pair=idx%npairs 
    ii,jj=iidx[pair],jidx[pair]
    tk=t[fieldrange,pair]
    rk=res[fieldrange,pair]
    #Interpolates the eigenvectors in the crossings
    def Intervec(v1,v2,tt):
        #Noramlizes and makes sure they have the same phase
        rov=jxn.sum(jxn.conj(v1)*v2,axis=-1,keepdims=True)
        mod=jxn.abs(rov)
        ok=mod>1e-12
        ph=jxn.where(ok,jxn.conj(rov)/jxn.where(ok,mod,1.0),1.0)
        v=(1.0-tt)[:,None]*v1+tt[:,None]*(v2*ph)
        n=jxn.linalg.norm(v,axis=-1,keepdims=True)
        return v/jxn.where(n<1e-30,1e-30,n)
    vik=Intervec(Vlist[fieldrange,:,ii],Vlist[fieldrange+ 1,:,ii],tk)
    vjk=Intervec(Vlist[fieldrange,:,jj],Vlist[fieldrange+ 1,:,jj],tk)
    #Calculates in the resonant field its intensity
    rops=jxn.stack([isx,isy,isz])
    rovi=jxn.einsum('cde,ke->kcd',rops,vik)
    T=jxn.einsum('kcd,kd->kc',rovi,jxn.conj(vjk))
    M2=jxn.sum(jxn.abs(T)**2,axis=1)
    if mw is None:
        #B1 averaged in the plane perpendicular to B0 (powder)
        Mn=nx*T[:,0]+ny*T[:,1]+nz*T[:,2]
        prob=M2-jxn.abs(Mn)**2
    else:
        #Fixed B1 direction (single crystal), as |<j|mw·g·S|i>|^2 in Eresonant
        Mm=mw[0]*T[:,0]+mw[1]*T[:,1]+mw[2]*T[:,2]
        prob=jxn.abs(Mm)**2
    #Frequency to field
    h2vik=jxn.einsum('de,ke->kd',h2,vik)
    h2vjk=jxn.einsum('de,ke->kd',h2,vjk)
    dert=jxn.real(jxn.sum(jxn.conj(vik)*h2vik,axis=1))
    izrt=jxn.real(jxn.sum(jxn.conj(vjk)*h2vjk,axis=1))
    gma=jxn.abs(izrt-dert)
    gma=jxn.where(gma<1e-6,1e-6,gma)
    gema=1.0/gma
    #Boltzman distribution
    Eres=(1.0-tk)[:,None]*Elist[fieldrange]+tk[:,None]*Elist[fieldrange+1]
    Ej=Eres*(1e9*scc.h)
    Temp=jxn.where(Tem<=0.0,1.0,Tem)
    beta=1.0/(scc.k*Temp)
    Emin=jxn.min(Ej,axis=-1,keepdims=True)
    boltz=jxn.exp(-beta*(Ej-Emin))
    boltz=boltz/jxn.sum(boltz,axis=-1,keepdims=True)
    ar=jxn.arange(Ktra)
    boltzm=jxn.abs(boltz[ar,ii]-boltz[ar,jj])
    boltzm=jxn.where(Tem<=0.0,1.0,boltzm)
    eintensy=prob*gema*boltzm
    ffres=jxn.where(valid,rk,0.0)
    ffint=jxn.where(valid,eintensy,0.0)
    return ffres,ffint,ntrans

def Meshtriangle():
    '''
    Creates the baricentral mesh of triangles for the JCaltriangle, that calculates the contribution of the resonant fields and their intensities in the final spectrum. It is defined as an independent function because it can interfere with the ADAM algorithm monitoring of the variables.
    
    Returns
    -------
    
    w1 : jax.np.array
        Vector of the points in the x direction of the grid.
    w2 : jax.np.array
        Vector of the points in the y direction of the grid.
    w3 : jax.np.array
        Vector of the points in the z direction of the grid.
    '''
    numd=20
    tpoints=(numd*(numd+1))/2.0
    w1,w2,w3=[],[],[]
    for i in range(numd):
        for j in range(numd-i):
            k=numd-1-i-j
            w1.append(i/(numd-1))
            w2.append(j/(numd-1))
            w3.append(k/(numd-1))
    w1=jxn.array(w1)[:,None]
    w2=jxn.array(w2)[:,None]
    w3=jxn.array(w3)[:,None]
    return w1,w2,w3,tpoints


@partial(jx.jit, static_argnames=['points'])
#@jx.checkpoint
def JCaltriangle(Bmin,dB,allres,allint,transi,hulk,weight,points):
    '''
    Creates a sketch spectrum using a barycentral mesh of triangles to calculate the contribution of the resonant fields and their intensities in the final spectrum. The result of this process is later convolute to generate the corrected EPR spectrum. Using the JAX framework, calculates multiple points at the same time with the function lax_scan and the Takeonetriangle wrap function.
    
    Parameters
    ----------
    
    Bmin : float
        Minimun value of the magnetic field.
    dB : float
        Delta difference between the magnetic field values.
    allres: jax.np.array
        List of the resonant fields of the system.
    allint : jax.np.array
        List of the intensity of the spectrum, evaluated in the resonant fields.
    transi : jax.np.array
        List of the possible transitions between the energy levels, related to the resonant fields.
    hulk : jax.np.arraynp.array
        Smaller convex poligon tha contains all the points require for the simulation.
    weight : jax.np.arraynp.array
        Normalized value of the weight of the points to evaluate it's contribution to the spectrum.
        
    Returns
    -------
    
    estotal : jax.np.array
        Array that contains the resulting spectrum sketch.
    
    '''
    def Takeonetriangle(trindex,valid):
        i1,i2,i3=trindex[0],trindex[1],trindex[2]
        B1,B2,B3=allres[i1],allres[i2],allres[i3]
        I1,I2,I3=allint[i1],allint[i2],allint[i3]
        weig=(weight[i1]+weight[i2]+weight[i3])/3.0
        pointweg=weig/tpoints
        Bin=ww1*B1+ww2*B2+ww3*B3
        Iint=(ww1*I1+ww2*I2+ww3*I3)*pointweg
        diferb=(Bin-Bmin)/dB
        fdiferb=diferb.flatten()
        fIint=Iint.flatten()
        vecl=jxn.floor(fdiferb).astype(jxn.int32)
        vecr=vecl+1
        fracr=fdiferb-vecl
        fracl=1.0-fracr
        Ilef=fIint*fracl
        Irig=fIint*fracr
        rleft=(vecl>=0)&(vecl<points)
        rright=(vecr>=0)&(vecr<points)
        vecl=jxn.where(rleft,vecl,0)
        vecr=jxn.where(rright,vecr,0)
        Ilef=jxn.where(rleft,Ilef,0.0)
        Irig=jxn.where(rright,Irig,0.0)
        sketch1=jxn.zeros(points)
        sketch1=sketch1.at[vecl].add(Ilef)
        sketch1=sketch1.at[vecr].add(Irig)
        n1,n2,n3=transi[i1],transi[i2],transi[i3]
        taketrian=(n1==n2)&(n2==n3)&(n1>0)&valid
        return jxn.where(taketrian,sketch1,jxn.zeros_like(sketch1))
    csize=200
    nreal=hulk.shape[0]
    pad=(csize-(nreal%csize))%csize
    hulkpad=jxn.pad(hulk,((0,pad),(0,0)),constant_values=0)
    validpad=jxn.pad(jxn.ones(nreal,dtype=bool),(0,pad),constant_values=False)
    batch=hulkpad.shape[0]//csize
    hulkbatch=hulkpad.reshape((batch,csize,3))
    validbatch=validpad.reshape((batch,csize)) 
    @jx.checkpoint
    def EWsize(carry,xs):
        batchidx,validb=xs
        bspc=jx.vmap(Takeonetriangle)(batchidx,validb)
        return carry+jxn.sum(bspc,axis=0),None
    estotal,_=jx.lax.scan(EWsize,jxn.zeros(points),(hulkbatch,validbatch))
    return estotal
    
ww1,ww2,ww3,tpoints=Meshtriangle()

def JPowder(Hamer,Expe,Nucl='None',M=70,graph=True,ktra=None,ndev=None):
    '''
    Wrap function for the simulation of the EPR spectrum for powder samples. It also use the SOPHE modified method to create the triangular grid and its weights.
    
    Parameters
    ----------
    
    Hamer : Class
        Container for the hamiltonian parameters of the system.
    
    Expe : Class
        Container for the experimental conditions.
        
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    
    M : int
        Number of divisions for the Delaunay grid.
        
    graph : Bool
        Plots the resulting spectrum.
    ktra : float
        Reference value of number of transitions.
    ndev : int
	Number of devices to use in the simulation.    

    Returns
    -------
    
    Blist : jax.np.array
        Array of the magnetic field.
    epc : jax.np.array
        Array of the counts of the spectrum.
    
    Example
    -------
    
    .. code-block:: python
    
       import matplotlib.pyplot as plt
       import epraya as epr
       import numpy as np
       Ham,Exp,_=epr.Jstart()
       Ham.S=3/2
       Ham.I=1
       Ham.g=np.array([2.003, 2, 2])
       Ham.A=np.array([200, 200, 200])  #Hyperfine constant
       Ham.D=np.array([800,200])      #Zero field D and E
       Ham.Hpp=[0,10]
       Exp.Freq=9.4
       Exp.Points=4096
       Exp.Temperature=300
       Exp.Frange=[0,800]
       B,spc=epr.JPowder(Ham,Exp)
       
    .. image:: /_static/jaxp.png
       :alt: Plot of the spectrum of the Jpowder function
       :align: center
    
    '''
    iwas,jwas,kwas,weight,hulk=Delaunay(Expe,M)
    iwas,jwas,kwas,weight,hulk=jxn.array(iwas),jxn.array(jwas),jxn.array(kwas),jxn.array(weight),jxn.array(hulk)
    Blist,epc=JCalpowder(Hamer,Expe,iwas,jwas,kwas,weight,hulk,Nucl,ktra=ktra,ndev=ndev)
    if graph:
        plt.figure(figsize=(10,6))
        plt.plot(Blist,epc,color='navy',label='Spectrum')
        plt.xlabel('Magnetic Field [mT]')
        formatter=EngFormatter(sep='') 
        plt.gca().yaxis.set_major_formatter(formatter)
        plt.ylabel('Counts [A. U.]')
        plt.xlim(Expe.Frange[0],Expe.Frange[1])
        plt.grid()
        plt.legend()
        plt.show(block=False)
    return Blist,epc

def JCalpowder(Hamer,Expe,iwas,jwas,kwas,weight,hulk,Nucl='None',hifi=False,ktra=None,ndev=None,info=False):
    '''
    Function for the simulation of the EPR spectrum for powder samples using the JAX functions. Instead of making the calculations with fors cycles, uses the vmap function for the calculations, dividing the points of calculations in blocks, that pass using the Oneori (One orientation) and Processvmap functions.
    
    Parameters
    ----------
    
    Hamer : Class
        Container for the hamiltonian parameters of the system.
    
    Expe : Class
        Container for the experimental conditions.
        
    iwas : jax.np.array
        Vectors (points in the grid) to consider in the calculation of the spectrum in the x direction.
    
    jwas : jax.np.array
        Vectors (points in the grid) to consider in the calculation of the spectrum in the y direction.
        
    kwas : jax.np.array
        Vectors (points in the grid) to consider in the calculation of the spectrum in the z direction.

    weight : jax.np.array
        Normalized value of the weight of the points to evaluate it's contribution to the spectrum.
    
    hulk : jax.np.array
        Smaller convex poligon tha contains all the points require for the simulation.
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    hifi : bool
        Bool value for the calculation of the Jacobian matrix.
    ktra : float
        Reference value of number of transitions.
    ndev : int
	Number of devices to use in the simulation.
    info : Bool
        Variable to pass the number of transitions.

    Returns
    -------
    Blist2 : jax.np.array
        Array of the magnetic field.


    espectotal : jax.np.array
        Array of the counts of the spectrum.
    
    Example
    -------
    >>> import matplotlib.pyplot as plt
    >>> import epraya as epr
    >>> import numpy as np
    >>> import jax.numpy as jxn
    >>> Ham,Exp,_=epr.Jstart()
    >>> iwas,jwas,kwas,weight,hulk=epr.Delaunay(Exp)
    >>> iwas,jwas,kwas,weight,hulk=jxn.array(iwas),jxn.array(jwas),jxn.array(kwas),jxn.array(weight),jxn.array(hulk)
    >>> Ham.S=3/2
    >>> Ham.I=1
    >>> Ham.g=np.array([2.003, 2, 2])
    >>> Ham.A=np.array([200, 200, 200])  #Hyperfine constant
    >>> Ham.D=np.array([800,200])      #Zero field D and E
    >>> Ham.Hpp=[0,10]
    >>> Exp.Freq=9.4
    >>> Exp.Points=4096
    >>> Exp.Temperature=300
    >>> Exp.Frange=[0,800]
    >>> print(epr.JCalpowder(Ham,Exp,iwas,jwas,kwas,weight,hulk))
    (Array([0.00000000e+00, 1.95360195e-01, 3.90720391e-01, ...,
       7.99609280e+02, 7.99804640e+02, 8.00000000e+02], dtype=float64),
    Array([ 1.48600800e-22,  1.06329540e-21,  6.49499460e-22, ...,
        2.06205909e-21,  5.04622598e-22, -1.50266650e-21], dtype=float64))   
    '''
    frange0=jxn.where(Expe.Frange[0]<0.0,1e-4,Expe.Frange[0])
    Ham=Hamer.replace(A=jxn.asarray(Hamer.A)/1000.0,D=jxn.asarray(Hamer.D)/1000.0,Hpp=jxn.asarray(Hamer.Hpp)/1.0,Q=jxn.asarray(Hamer.Q)/1000.0,
                     Bk2=jxn.asarray(Hamer.Bk2)/1000.0,Bk4=jxn.asarray(Hamer.Bk4)/1000.0,Bk6=jxn.asarray(Hamer.Bk6)/1000.0)
    etas=Ham.eta
    etas=jxn.where(Ham.Hpp[1]==0.0,0.0,etas)
    etas=jxn.where(Ham.Hpp[0]==0.0,1.0,etas)
    Ham=Ham.replace(eta=etas)
    dim=int(2*Ham.S+1)*int(2*Ham.I+1)*int(2*Ham.L+1)
    Exp=Expe
    Ham=Jchaframe(Ham,Exp)
    sx,sy,sz=JPauli(Ham.S)
    ix,iy,iz=JPauli(Ham.I)
    isx=jxn.kron(sx,jxn.eye(int(2*Ham.I+1)))
    isx=jxn.kron(jxn.eye(int(2*Ham.L+1)),isx)
    isx=jxn.asarray(isx,dtype=jxn.complex64)
    isy=jxn.kron(sy,jxn.eye(int(2*Ham.I+1)))
    isy=jxn.kron(jxn.eye(int(2*Ham.L+1)),isy)
    isy=jxn.asarray(isy,dtype=jxn.complex64)
    isz=jxn.kron(sz,jxn.eye(int(2*Ham.I+1)))
    isz=jxn.kron(jxn.eye(int(2*Ham.L+1)),isz)
    isz=jxn.asarray(isz,dtype=jxn.complex64)
    #Transition rate with the Zeeman part
    issx=Ham.g[0,0]*isx+Ham.g[0,1]*isy+Ham.g[0,2]*isz
    issy=Ham.g[1,0]*isx+Ham.g[1,1]*isy+Ham.g[1,2]*isz
    issz=Ham.g[2,0]*isx+Ham.g[2,1]*isy+Ham.g[2,2]*isz
    isx,isy,isz=issx,issy,issz
    E=Exp.Freq
    beta=(scic.physical_constants["Bohr magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
    betan=(scic.physical_constants["nuclear magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
    hzex=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,[1,0,0],dim),dtype=jxn.complex64)
    hzey=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,[0,1,0],dim),dtype=jxn.complex64)
    hzez=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,[0,0,1],dim),dtype=jxn.complex64)
    h1=jxn.zeros((dim,dim),dtype='complex64')
    if Ham.S>=1:
        h1=h1+JStevensrot(sx,sy,sz,Ham.S,Ham,dim,Exp)
    if Ham.L!=0:
        h1=h1+JLorbit(sx,sy,sz,Ham.lc,dim,Ham.L)
    if Ham.I!=0:
        h1=h1+JHfi(sx,sy,sz,ix,iy,iz,Ham.A,dim)
        h1=h1+JQii(ix,iy,iz,Ham.Q,dim)
        gnk=gnfactor(Nucl)
        nhzex=jxn.asarray(betan*JNhze(Ham.I,ix,iy,iz,dim,gnk,[1,0,0]),dtype=jxn.complex64)
        nhzey=jxn.asarray(betan*JNhze(Ham.I,ix,iy,iz,dim,gnk,[0,1,0]),dtype=jxn.complex64)
        nhzez=jxn.asarray(betan*JNhze(Ham.I,ix,iy,iz,dim,gnk,[0,0,1]),dtype=jxn.complex64)
        hzex-=nhzex
        hzey-=nhzey
        hzez-=nhzez
    h1=jxn.asarray(h1,dtype=jxn.complex64)
    Blist1=jxn.linspace(frange0,Exp.Frange[1],500)
    dB=(Exp.Frange[1]-Exp.Frange[0])/(Exp.Points-1)
    Bmin=Exp.Frange[0]
    ndev=jx.local_device_count() if ndev is None else int(ndev)
    P=dict(B=Blist1,h1=h1,hx=hzex,hy=hzey,hz=hzez,isx=isx,isy=isy,isz=isz,Freq=jxn.asarray(Exp.Freq,float),Temp=jxn.asarray(Exp.Temperature,float),Hpp=Ham.Hpp)
    allres,allint,ntrans,K=Jorientblocks(P,iwas,jwas,kwas,dim,hifi,ktra,ndev)
    sketch=JCaltriangle(Bmin,dB,allres,allint,ntrans,hulk,weight,Exp.Points)
    espectotal=JVoigtp(sketch,dB,Ham.Hpp)
    Blist2=jxn.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
    if info:
        return Blist2,espectotal,K,jx.lax.stop_gradient(jxn.max(ntrans))
    return Blist2,espectotal

@partial(jx.jit,static_argnames=('dim','K','hifi'))
def makeoneori(P,nx,ny,nz,dim,K,hifi):
    def oneori(nnx,nny,nnz):
        Elist,Vlist,h2=JPadaptarray(P['B'],P['h1'],P['hx'],P['hy'],P['hz'],nnx,nny,nnz,hifi)
        return JNresina(P['B'],Elist,Vlist,dim,P['Freq'],P['isx'],P['isy'],P['isz'],nnx,nny,nnz,P['Temp'],P['Hpp'],h2,hifi=hifi,ktra=K)
    return jx.vmap(oneori)(nx,ny,nz)

@partial(jx.jit,static_argnames=('dim','K','hifi'))
def Jlaxscan(P,bi,bj,bk,dim,K,hifi):
    @jx.checkpoint
    def step(c,param):
        return c,makeoneori(P,*param,dim=dim,K=K,hifi=hifi)
    return jx.lax.scan(step,None,(bi,bj,bk))[1]

@lru_cache(maxsize=None)
def Jlaxscanp(dim,K,hifi):
    return jx.pmap(partial(Jlaxscan,dim=dim,K=K,hifi=hifi),in_axes=(None,0,0,0))

def Jorientblocks(P,iwas,jwas,kwas,dim,hifi=False,ktra=None,ndev=None):
    '''
    Resonant fields and intensities for all the orientations, by blocks of csize orientations.
    Common to JCalpowder and Jcalmulta. Returns allres, allint, ntrans and the K used.
    '''
    ndev=jx.local_device_count() if ndev is None else int(ndev)
    #Block size: max 32, min 8, for approx 3 GB
	#On GPU, batched eigh fails above 10000 matrices for dim>32
    csize=int(np.clip(3e9/ndev//(500*(2.2*dim**2*16+20*dim*8)),8,32))
    if jx.default_backend()=='gpu' and dim>32:
        csize=min(csize,12)
    K=int(ktra) if ktra is not None else estimatek(P,iwas,jwas,kwas,dim,hifi,csize=min(16,csize))
    tlen=len(iwas)
    blk=csize*ndev
    plen=(blk-(tlen%blk))%blk
    bi,bj,bk=[jxn.pad(v,(0,plen)).reshape(ndev,-1,csize) for v in (iwas,jwas,kwas)]
    #Condition the process to have a K higher than the number of transitions
    while True:
        if ndev==1:
            r,i,n=Jlaxscan(P,bi[0],bj[0],bk[0],dim=dim,K=K,hifi=hifi)
        else:
            r,i,n=Jlaxscanp(dim,K,hifi)(P,bi,bj,bk)
        r,i,n =r.reshape(-1,r.shape[-1])[:tlen],i.reshape(-1,i.shape[-1])[:tlen],n.reshape(-1)[:tlen]
        if isinstance(n,jx.core.Tracer) or int(jxn.max(n))<=K:
            return r,i,n,K
        K*=2


def estimatek(P,nx,ny,nz,dim,hifi,csize=16,nsample=256,margin=1.5):
    idx=np.unique(np.linspace(0,len(nx)-1,nsample).astype(int))
    idx=np.concatenate([idx,np.full((-len(idx))%csize,idx[-1])])
    nmax=0.0
    for s in range(0,len(idx),csize):
        sl=idx[s:s+csize]
        _,_,nt=makeoneori(P,nx[sl],ny[sl],nz[sl],dim=dim,K=1,hifi=hifi)
        nmax=max(nmax,float(jxn.max(nt)))
    return int(np.ceil(nmax*margin))+4

def Jresonant(Hamer,Expe,graph=True,table=True,Nucl='None',Mcolor=True,relevance=1e-4):
    '''
    Wrap function for the simulation of the EPR spectrum for monocrystal samples. Also creates the table of transitions and energy diagrams of the system.

    
    Parameters
    ----------
    

    Hamer : Class
        Container for the hamiltonian parameters of the system.
    Expe : Class
        Container for the experimental conditions.
    graph : Bool
        Plots the resulting spectrum.

    table : Bool
        Creates the table of transitions.
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 

    Mcolor : Bool
        Bool value to follow with colors the change in the expectation value for |mS| near anticrossings.
    relevance : float
        Relative threshold for the transition strength (Intensity/Boltzmann factor), if lower it is discarded. Default is 1e-4.
    Returns
    -------
    

    Blist : jax.np.array

        Array of the magnetic field.
    epc : jax.np.array
        Array of the counts of the spectrum.
    

    Example
    -------
    
    .. code-block:: python
    
       import matplotlib.pyplot as plt
       import epraya as epr

       import numpy as np
       Ham,Exp,_=epr.Jstart()
       Ham.S=3/2
       Ham.I=1
       Ham.g=np.array([2.003, 2, 2])
       Ham.A=np.array([200, 200, 200])  #Hyperfine constant
       Ham.D=np.array([800,200])      #Zero field D and E
       Ham.Hpp=[0,10]
       Exp.Freq=9.4
       Exp.Points=4096

       Exp.Temperature=300
       Exp.Frange=[0,800]
       B,spc=epr.Jresonant(Ham,Exp)

       
     .. image:: /_static/tabaj.png
       :alt: Table of transitions of the Jresonant function
       :align: center
       
    .. image:: /_static/jreso.png
       :alt: Plot of the spectrum of the Jresonant function
       :align: center
       
    .. image:: /_static/diaj.png
       :alt: Energy diagram of the Jresonant function
       :align: center    
    '''
    slit,nlit,llit,_=Msmi(Hamer.I,Hamer.S,Hamer.L)
    Blist,epc,h1,hzex,hzey,hzez,hmw=Calresonant(Hamer,Expe,Nucl,diagram=True)
    Blist=np.array(Blist)
    ndir=jxn.array(Expe.Fdirection,dtype=jxn.float64)
    ndir=ndir/jxn.linalg.norm(ndir)
    nx,ny,nz=ndir[0],ndir[1],ndir[2]
    hze=nx*hzex+ny*hzey+nz*hzez
    Elist,Vlist,h2=JPadaptarray(Blist,h1,hzex,hzey,hzez,nx,ny,nz,False)
    #For the energy diagrams
    #Elist,Vlist=JPretrack(Elist,Vlist)
    splines=cubichers(Blist,Elist,axis=0)
    maxvector=Vlist[-1]
    maxvector=Fieldframe(maxvector,Expe.Fdirection,Hamer.S,Hamer.I)
    curvebasis=Assingstatestobasis(maxvector)
    dim=Elist.shape[1]
    resfield=[]
    resonants=[]
    relev=[]
    for i in range(dim):
        for j in range(i+1, dim):
            diffv=jxn.abs(Elist[:,j]-Elist[:,i])-Expe.Freq
            signch=np.where(np.diff(np.signbit(diffv)))[0]
            for k in signch:
                bstart,bend=Blist[k],Blist[k+1]
                def deltaE(b):
                    return np.real(jxn.abs(splines(b)[j]-splines(b)[i]))-Expe.Freq
                try:
                    res=sci.optimize.root_scalar(deltaE,bracket=[bstart,bend],method='brentq')
                    if res.converged:
                        Hres=h1+hze*res.root
                        Eres,Vres=jxn.linalg.eigh(Hres)
                        localbasis=Assingstatestobasis(Fieldframe(Vres,Exp.Fdirection,Ham.S,Ham.I))
                        basis1=localbasis[i]
                        basis2=localbasis[j]
                        vecci=Vres[:,i]
                        veccj=Vres[:,j]
                        #Calculation for intensities
                        vecci/=np.linalg.norm(vecci)
                        veccj/=np.linalg.norm(veccj)
                        trament=veccj.conj().T@hmw@vecci
                        prob=np.abs(trament)**2
                        #Frecuency to field
                        dert=vecci.conj().T@hze@vecci
                        izrt=veccj.conj().T@hze@veccj
                        gma=np.abs(izrt-dert)
                        if gma<1e-6:
                            gma=1e-6
                        gema=1/gma
                        ms1,ms2=slit[basis1],slit[basis2]
                        mi1,mi2=nlit[basis1],nlit[basis2]
                        dms=np.abs(ms1-ms2)
                        dmi=np.abs(mi1-mi2)
                        if np.isclose(dms,1) and np.isclose(dmi,0):
                            ttyp="Allowed"
                        elif np.isclose(dms,2):
                            ttyp="Forbidden (2)"
                        elif np.isclose(dms,3):
                            ttyp="Forbidden (3)"
                        elif not np.isclose(dmi,0):
                            ttyp="Forbidden (N)"
                        else:
                            ttyp="Forbidden"
                        state1=Getlabel(basis1,slit,nlit,llit,Hamer.L,Hamer.I)
                        state2=Getlabel(basis2,slit,nlit,llit,Hamer.L,Hamer.I)
                        resonants.append({'field': res.root,'inx': (i, j),'bainx': (basis1,basis2),'type': ttyp,'transition': f"{state1} <-> {state2}",'relevance': prob*gema})
                        resfield.append(res.root)
                        relev.append(prob*gema)
                except ValueError:
                    pass
    #Discard the lower relevance resonances
    if len(relev)>0:
        smax=max(relev)
        keep=[]
        if smax>0:
            for k in range(len(relev)):
                if relev[k]>=relevance*smax:
                    keep.append(k)
        resonants=[resonants[k] for k in keep]
        resfield=[resfield[k] for k in keep]
        for r in resonants:
            r['relevance']=r['relevance']/smax
    if len(resfield)>0:
        if table:
            df=DataFrame(data=resonants)
            dfdis=df[['field','transition','type']].copy()
            dfl=dfdis.iloc[::2].reset_index(drop=True)
            dfr=dfdis.iloc[1::2].reset_index(drop=True)
            dfdis=concat([dfl, dfr],axis=1)
            dfdis.columns=['Field (mT)','Transition','Type','Field (mT)','Transition','Type']
            dfdis['Field (mT)']=dfdis['Field (mT)'].round(3)
            if is_notebook():
                from IPython.display import display
                display(dfdis)
            else:
                print(dfdis)
        else:
            print("No resonant fields detected in selected range")
        if graph:
            Plotsim(espac1=Blist,inten1=epc,resfield=resfield,espac2=Blist,enegria=Elist,vectors=Vlist,curvebasis=curvebasis,splines=splines,resonants=resonants,Ham=Hamer,Exp=Expe,Mcolor=Mcolor)
    return Blist,epc

def Calresonant(Hamer,Expe,Nucl='None',diagram=False,hifi=False,ktra=None,info=False):
    '''
    Function for the calculation of the EPR cw spectrum of monocristal systems. Uses a formulation similar to the *Eresonant* function, to calculate the resonant fields and intensities, but calculates de absorption curve (first integral) of the spectrum that is numerically  derived, producing the final spectrum.
    
    Parameters
    ----------
    
    Hamer : Class
        Container for the hamiltonian parameters of the system.
    
    Expe : Class
        Container for the experimental conditions.
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    diagram : Bool
        To pass the Energy data.
    hifi : Bool
        Bool to calculate the Jacobian matrix. 
    info : Bool
        Variable to pass the number of transitions.
        
    Returns
    -------
    
    Blist : jax.np.array
        Array of the magnetic field.
    epc : jax.np.array
        Array of the counts of the spectrum.
    h1 : jax.np.array
        All not Zeeman interaction matrix.
    hzex : jax.np.array
        Zeeman matrix in the x direction.
    hzey : jax.np.array
        Zeeman matrix in the x direction.
    hzez : jax.np.array
        Zeeman matrix in the x direction.
    hmw : jax.np.array
        Microwave field Zeeman interaction.
    Example
    -------

    >>> import matplotlib.pyplot as plt
    >>> import epraya as epr
    >>> import numpy as np
    >>> Ham,Exp,_=epr.Jstart()
    >>> Ham.S=3/2
    >>> Ham.I=1
    >>> Ham.g=np.array([2.003, 2, 2])
    >>> Ham.A=np.array([200, 200, 200])  #Hyperfine constant
    >>> Ham.D=np.array([800,200])      #Zero field D and E
    >>> Ham.Hpp=[0,10]
    >>> Exp.Freq=9.4
    >>> Exp.Points=4096
    >>> Exp.Temperature=300
    >>> Exp.Frange=[0,800]
    >>> print(epr.Calresonant(Ham,Exp)[:2])
    (Array([0.00000000e+00, 1.95360195e-01, 3.90720391e-01, ...,
       7.99609280e+02, 7.99804640e+02, 8.00000000e+02], dtype=float64),
    Array([-1.64840330e-07, -1.10427100e-07, -8.03249933e-08, ...,
    -8.85167972e-08, -8.84171157e-08, -8.83673152e-08], dtype=float64))
    '''
    frange0=jxn.where(Expe.Frange[0]<0.0,1e-4,Expe.Frange[0])
    ndir1=np.asarray(Expe.Fdirection,dtype=float)
    mdir1=np.asarray(Expe.Mwdirection,dtype=float)
    norm2=np.linalg.norm(ndir1)
    norm3=np.linalg.norm(mdir1)
    if norm2>0 and norm3>0 and abs(ndir1@mdir1)/(norm2*norm3)>0.99:
        print("WARNING: Mwdirection is (almost) parallel to Fdirection. Use a Mwdirection perpendicular to Fdirection for a standard cavity.")
        
    Ham=Hamer.replace(A=jxn.asarray(Hamer.A)/1000.0,D=jxn.asarray(Hamer.D)/1000.0,Hpp=jxn.asarray(Hamer.Hpp)/1.0,Q=jxn.asarray(Hamer.Q)/1000.0,
                     Bk2=jxn.asarray(Hamer.Bk2)/1000.0,Bk4=jxn.asarray(Hamer.Bk4)/1000.0,Bk6=jxn.asarray(Hamer.Bk6)/1000.0)
    etas=Ham.eta
    etas=jxn.where(Ham.Hpp[1]==0.0,0.0,etas)
    etas=jxn.where(Ham.Hpp[0]==0.0,1.0,etas)
    Ham=Ham.replace(eta=etas)
    slit,nlit,llit=JMsmi(Ham.I,Ham.S,Ham.L)
    dim=int(2*Ham.S+1)*int(2*Ham.I+1)*int(2*Ham.L+1)
    Exp=Expe
    ndir=jxn.array(Expe.Fdirection,dtype=jxn.float64)
    ndir=ndir/jxn.linalg.norm(ndir)
    nx,ny,nz=ndir[0],ndir[1],ndir[2]
    mdir=jxn.array(Expe.Mwdirection,dtype=jxn.float64)
    mdir=mdir/jxn.linalg.norm(mdir)
    mx,my,mz=mdir[0],mdir[1],mdir[2]
    Ham=Jchaframe(Ham,Exp)
    sx,sy,sz=JPauli(Ham.S)
    ix,iy,iz=JPauli(Ham.I)
    isx=jxn.kron(sx,jxn.eye(int(2*Ham.I+1)))
    isx=jxn.kron(jxn.eye(int(2*Ham.L+1)),isx)
    isx=jxn.asarray(isx,dtype=jxn.complex64)
    isy=jxn.kron(sy,jxn.eye(int(2*Ham.I+1)))
    isy=jxn.kron(jxn.eye(int(2*Ham.L+1)),isy)
    isy=jxn.asarray(isy,dtype=jxn.complex64)
    isz=jxn.kron(sz,jxn.eye(int(2*Ham.I+1)))
    isz=jxn.kron(jxn.eye(int(2*Ham.L+1)),isz)
    isz=jxn.asarray(isz,dtype=jxn.complex64)
    #Transition rate with the Zeeman part
    issx=Ham.g[0,0]*isx+Ham.g[0,1]*isy+Ham.g[0,2]*isz
    issy=Ham.g[1,0]*isx+Ham.g[1,1]*isy+Ham.g[1,2]*isz
    issz=Ham.g[2,0]*isx+Ham.g[2,1]*isy+Ham.g[2,2]*isz
    isx,isy,isz=issx,issy,issz
    E=Exp.Freq
    beta=(scic.physical_constants["Bohr magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
    betan=(scic.physical_constants["nuclear magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
    hzex=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,[1,0,0],dim),dtype=complex)
    hzey=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,[0,1,0],dim),dtype=complex)
    hzez=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,[0,0,1],dim),dtype=complex)
    hmw=jxn.asarray(beta*JHze(sx,sy,sz,Ham.g,Exp.Mwdirection,dim),dtype=complex)
    h1=jxn.zeros((dim,dim),dtype='complex64')
    if Ham.S>=1:
        h1=h1+JStevensrot(sx,sy,sz,Ham.S,Ham,dim,Exp)
    if Ham.L!=0:
        h1=h1+JLorbit(sx,sy,sz,Ham.lc,dim,Ham.L)
    if Ham.I!=0:
        h1=h1+JHfi(sx,sy,sz,ix,iy,iz,Ham.A,dim)
        h1=h1+JQii(ix,iy,iz,Ham.Q,dim)
        gnk=gnfactor(Nucl)
        nhzex=jxn.asarray(betan*JNhze(Ham.I,ix,iy,iz,dim,gnk,[1,0,0]),dtype=complex)
        nhzey=jxn.asarray(betan*JNhze(Ham.I,ix,iy,iz,dim,gnk,[0,1,0]),dtype=complex)
        nhzez=jxn.asarray(betan*JNhze(Ham.I,ix,iy,iz,dim,gnk,[0,0,1]),dtype=complex)
        hzex-=nhzex
        hzey-=nhzey
        hzez-=nhzez
    h1=jxn.asarray(h1,dtype=complex)
    Blist1=jxn.linspace(frange0,Exp.Frange[1],500)
    Blist2=jxn.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
    dB=(Exp.Frange[1]-Exp.Frange[0])/(Exp.Points-1)
    Bmin=Exp.Frange[0]
    Elist,Vlist,h2=JPadaptarray(Blist1,h1,hzex,hzey,hzez,nx,ny,nz,hifi)
    resfield,intensy,ntrans=JNresina(Blist1,Elist,Vlist,dim,Exp.Freq,isx,isy,isz,nx,ny,nz,Exp.Temperature,Ham.Hpp,h2,hifi=hifi,ktra=ktra,mw=mdir)
    #FFT convolution as in JPowder
    n=Exp.Points
    npad=2*n
    k=2*jxn.pi*jxn.arange(npad//2+1)/(npad*dB)
    S=jxn.exp(-1j*k[:,None]*(resfield-Bmin)[None,:])@intensy
    H=(1j*k)*jxn.exp(-0.5*((Ham.Hpp[0]/2.0)*k)**2-(jxn.sqrt(3.0)/2.0*Ham.Hpp[1])*jxn.abs(k))
    espectotal=jxn.fft.irfft(S*H,n=npad)[:n]/dB
    if info:
        return Blist2,espectotal,h1,hzex,hzey,hzez,hmw,jx.lax.stop_gradient(ntrans)
    return Blist2,espectotal,h1,hzex,hzey,hzez,hmw
   
@jaxdatclass
class Mjhval:
    '''
    Jax container class for the hamiltonial parameters of  two systems. Must be initialized with a variable like ``Ham``. For the nth-system, it can be change using ``Ham.SN=1/2``.
    
    
    *Note: The Hpp is the same for the two systems.*
    
    **Warning: The Stevens operators constants are only defined for the first system.**
    
    Parameters
    ----------
    
    S : float 
        Spin value ex. (1/2,0,3/2).
    g : array_like or float
        g value of system, can be float, for isotropic case or array for anisotropic.
    I : float
        Nuclear spin value
    L : float
        Angular momentum
    A : array_like or float 
        Hyperfine constant, float for isotropic and array for anisotropic
    Q : array_like or float 
        Quadrupole nuclear interaction constant, float for isotropic and array for anisotropic
    D : array_like
        Zero field interaction constants D and E, two value array [0,0]
    Bk2 : array_like
        Stevens k=-/+2 constants
    Bk4 : array_like
        Stevens k=-/+4 constants    
    Bk6 : array_like
        Stevens k=-/+6 constants      
    lc : float
        Spin-orbit interaction constant
    Hpp : array_like
        Peak to peak distance for the voigtian function using [Hg,Hl], for gaussian and lorentzian distance
    eta : float
        weight of the gaussian contribution to the voigtian function, from 0 to 1. If eta is 0, the function is lorentzian and if eta is 1, the function is gaussian.
    weight: float
        Dummy variable by the moment
    Nucl : str
        Isotope of the sample. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    A1_2 : array_like
        Hipefine interaction between the spin 1 and nuclear spin 2.
    X1_2 : array_like
        Electron-Electron interaction between spins 1 and 2.
        
    Example
    -------
    >>> import epraya as epr
    >>> Ham=Mjhval()
    >>> Ham.S1=1
    Ham.I1=1/2
    >>> Ham.g1=[2.003,1.8,1.5]
    >>> Ham.S2=2
    >>> Ham.I2=1
    >>> Ham.g2=[2.003,1.8,1.5]
    >>> Ham.A1_2=[200,300,200]
    >>> Ham.X1_2=[500,1000,500]
    >>> print(Ham)
    Mjhval(S1=1, S2=2, g1=[2.003, 1.8, 1.5], g2=[2.003, 1.8, 1.5],
    I1=0.5, I2=1, L1=0.0, L2=0.0, A1=0.0, A2=0.0, 
    Q1=Array([0, 0, 0], dtype=int32), Q2=Array([0, 0, 0], dtype=int32),
    D1=Array([0, 0], dtype=int32), D2=Array([0, 0], dtype=int32), 
    Bk2=[0, 0, 0, 0, 0], Bk4=[0, 0, 0, 0, 0, 0, 0, 0, 0],
    Bk6=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], lc1=0.0, lc2=0.0,
    A1_2=[200, 300, 200], A2_1=0.0, X1_2=[500, 1000, 500], 
    Hpp=Array([0, 1], dtype=int32), eta=0.5, weight=0.0)
    '''
    S1: Union[float,int]=1/2   # Spin
    S2: Union[float,int]=1/2   # Spin
    g1: Union[List[float],float,int]=dcfield(default_factory=lambda: 2.003)  # g value
    g2: Union[List[float],float,int]=dcfield(default_factory=lambda: 2.003)  # g value
    I1: float=0.0   # Nuclear spin
    I2: float=0.0   # Nuclear spin
    L1: float=0.0   # Angular momentum
    L2: float=0.0   # Angular momentum
    A1: Union[List[float], float]=dcfield(default_factory=lambda:0.0)     # Hyperfine constant
    A2: Union[List[float], float]=dcfield(default_factory=lambda:0.0)     # Hyperfine constant
    Q1: Union[list[float], float]=dcfield(default_factory=lambda:jxn.array([0,0,0]))    # Quadrupole interaction constant
    Q2: Union[list[float], float]=dcfield(default_factory=lambda:jxn.array([0,0,0]))    # Quadrupole interaction constant
    D1: Union[list[float], float]=dcfield(default_factory=lambda:jxn.array([0,0]))     # Zero field interaction D and E constants
    D2: Union[list[float], float]=dcfield(default_factory=lambda:jxn.array([0,0]))     # Zero field interaction D and E constants
    Bk2: Union[list[float], float]=dcfield(default_factory=lambda:[0,0,0,0,0])
    Bk4: Union[list[float], float]=dcfield(default_factory=lambda:[0,0,0,0,0,0,0,0,0])
    Bk6: Union[list[float], float]=dcfield(default_factory=lambda:[0,0,0,0,0,0,0,0,0,0,0,0,0])
    lc1: float=0.0                          # Spin-orbit interaction constant
    lc2: float=0.0                          # Spin-orbit interaction constant
    A1_2: Union[List[float], float]=dcfield(default_factory=lambda:0.0)     # Hyperfine constant
    A2_1: Union[List[float], float]=dcfield(default_factory=lambda:0.0)     # Hyperfine constant
    X1_2: Union[List[float], float]=dcfield(default_factory=lambda:0.0)
    Hpp: List=dcfield(default_factory=lambda:jxn.array([0,1]))
    eta: float=0.5
    weight: float=0.0

@jaxdatclass
class JEmco:
    '''
    Jax container class for the experimental parameters of two systems. Must be initialized with a variable like ``Exp``. To change one parameter use the sintaxis ``Exp.Freq=9.40``.
    
    Parameters
    ----------
    Freq : float
        Frequency of the microwave in the EPR spectrometer in GHz.
    Points  : int
        Number of points used to take the EPR spectrum.
    Temperature : float
        Temperature of the sample during the measurement in Kelvin.

    Fdirection : list=[0,0,1]
        Direction of incidence of the magnetic field in relation with the lab frame. By deafult it's [0,0,1], the Z direction.
    Mwdirection : list=[1,0,0]
        Direction of incidence of the microwave radiation in relation with the lab frame. By deafult it's [1,0,0], the X direction.
    Frange : list=[0,1]
        Field range used in the EPR spectrum in mT.
    Sampleframe : list==[0,0,0]
        Three Euler angles of the orientation of the sample in relation to the lab frame.
    Molframe : list=[0,0,0]
        Three Euler angles of the orientation of the  paramagnetic molecule in relation to the sample frame.
    gframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor g in the molecular frame.
    Aframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor A in the molecular frame.    
    Dframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor Q in the molecular frame.    
    Qframe : list=[0,0,0]
        Three Euler angles of the orientation of the tensor D in the molecular frame.   
        
    Example
    --------
    
    >>> import epraya as epr 
    >>> Exp=epr.JEmco()
    >>> Exp.Freq=9.43
    >>> Exp.Points=2046
    >>> Exp.Temperature=306
    >>> Exp.Frange=[0,400]
    >>> Exp.Mwdirection=[0,1,0]
    >>> Exp.gframe1=[20,30,40]
    >>> Exp.gframe2=[20,30,40]
    >>> print(Exp)
    JEmco(Freq=9.43, Points=2046, Temperature=306, Fdirection=[0, 0, 1],
    Mwdirection=[0, 1, 0], Frange=[0, 400], Sampleframe1=[0, 0, 0],
    Molframe1=[0, 0, 0], gframe1=[20, 30, 40], Aframe1=[0, 0, 0],
    Dframe1=[0, 0, 0], Qframe1=[0, 0, 0], Sampleframe2=[0, 0, 0],
    Molframe2=[0, 0, 0], gframe2=[20, 30, 40], Aframe2=[0, 0, 0], 
    Dframe2=[0, 0, 0], Qframe2=[0, 0, 0])
    '''
    Freq: float=9.433
    Points: int=4096
    Temperature: float=295.15
    Fdirection: list[float]=dcfield(default_factory=lambda:[0,0,1])
    Mwdirection: list[float]=dcfield(default_factory=lambda:[1,0,0])
    Frange: list[float]=dcfield(default_factory=lambda:[0,1])
    Sampleframe1: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Molframe1: list[float]=dcfield(default_factory=lambda:[0,0,0])
    gframe1: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Aframe1: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Dframe1: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Qframe1: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Sampleframe2: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Molframe2: list[float]=dcfield(default_factory=lambda:[0,0,0])
    gframe2: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Aframe2: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Dframe2: list[float]=dcfield(default_factory=lambda:[0,0,0])
    Qframe2: list[float]=dcfield(default_factory=lambda:[0,0,0])

@jaxdatclass
class JEmva:
    '''
    Jax-container class for the range of variation of the hamiltonian parameters by defining the minimum and then the maximum value of each variable. Works for 2 systems and must be initialized with a variable like ``Vary``. To change one parameter use the sintaxis ``Vary.g=[1.5,2.5,1.0,2.0,0.0,1.0]``.
    
    *Note: The Hpp is the same for the two systems.*
    
    Parameters
    ----------
    g : list[float]
        Range to vary the three values of the g tensor.
    A : list[float]  
        Range to vary the three values of the A tensor.
    Q : list[float]  
        Range to vary the three values of the Q tensor.
    D : list[float]
        Range to vary the two values of the D tensor.
    Hpp : list[float]
        Range to vary the two values of the peak to peak distance.
    weight : float
        Dummy variable by the moment.
    
    Example
    --------
    
    >>> import epraya as epr
    >>> Vary=epr.JEmva()
    >>> Vary.g1=[1.5,2.5,1.0,2.0,0.0,1.0]
    >>> Vary.A1=[100,300,200,400,200,250]
    >>> Vary.g2=[1.2,1.5,2.003,2.005,2.5,2.6]
    >>> Vary.A2=[200,250,500,600,200,250]
    >>> Vary.Hpp=[0,20,10,15]
    >>> print(Vary)
    JEmva(g1=[1.5, 2.5, 1.0, 2.0, 0.0, 1.0], 
    A1=[100, 300, 200, 400, 200, 250], Q1=0.0, 
    D1=0.0, g2=[1.2, 1.5, 2.003, 2.005, 2.5, 2.6],
    A2=[200, 250, 500, 600, 200, 250], Q2=0.0, 
    D2=0.0, Hpp=[0, 20, 10, 15], weight=0.0)
    '''
    g1: Union[list[float],float]=0.0
    A1: Union[list[float],float]=0.0     # Hyperfine constant
    Q1: Union[list[float],float]=0.0     # Quadrupole interaction constant
    D1: Union[list[float],float]=0.0
    g2: Union[list[float],float]=0.0
    A2: Union[list[float],float]=0.0     # Hyperfine constant
    Q2: Union[list[float],float]=0.0     # Quadrupole interaction constant
    D2: Union[list[float],float]=0.0
    Hpp: List=dcfield(default_factory=lambda:jxn.array([0,0]))
    weight: float=0.0

def Jmstart():
    '''
    Creates the three JAX containers ``Ham``, ``Exp`` and ``Vary`` for the two systems.
    
    Returns
    -------
    
    Ham : Class
        Jax container for the hamiltonian parameters of the two systems.
    Exp : Class
        Jax container for the experimental conditions of the two systems.
    Vary : Class
        Jax container for the range and parameters to vary of the two systems. 
    
    Example
    -------
    
    >>> import epraya as epr
    >>> Ham, Exp, Vary=epr.Jmstart()
    >>> print(Ham,Exp,Vary)
    Mjhval(S1=0.5, S2=0.5, g1=2.003, g2=2.003, I1=0.0, I2=0.0,
    L1=0.0, L2=0.0, A1=0.0, A2=0.0, Q1=Array([0, 0, 0], dtype=int32),
    Q2=Array([0, 0, 0], dtype=int32), D1=Array([0, 0], dtype=int32),
    D2=Array([0, 0], dtype=int32), Bk2=[0, 0, 0, 0, 0], 
    Bk4=[0, 0, 0, 0, 0, 0, 0, 0, 0], 
    Bk6=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], lc1=0.0, lc2=0.0,
    A1_2=0.0, A2_1=0.0, X1_2=0.0, Hpp=Array([0, 1], dtype=int32),
    eta=0.5, weight=0.0) 
    JEmco(Freq=9.433, Points=4096, Temperature=295.15, 
    Fdirection=[0, 0, 1], Mwdirection=[1, 0, 0], Frange=[0, 1],
    Sampleframe1=[0, 0, 0], Molframe1=[0, 0, 0], gframe1=[0, 0, 0],
    Aframe1=[0, 0, 0], Dframe1=[0, 0, 0], Qframe1=[0, 0, 0], 
    Sampleframe2=[0, 0, 0], Molframe2=[0, 0, 0], gframe2=[0, 0, 0],
    Aframe2=[0, 0, 0], Dframe2=[0, 0, 0], Qframe2=[0, 0, 0]) 
    JEmva(g1=0.0, A1=0.0, Q1=0.0, D1=0.0, g2=0.0, A2=0.0, Q2=0.0,
    D2=0.0, Hpp=Array([0, 0], dtype=int32), weight=0.0)
    '''
    Ham,Exp,Vary=Mjhval(),JEmco(),JEmva()
    return Ham,Exp,Vary

def JMulpol(maham,Expe,Nucl1='None',Nucl2='None',M=70,graph=True,hifi=False,ktra=None,ndev=None):
    '''
    Wrap function for the simulation of the EPR spectrum for powder samples of two systems. If there is an interaction between the systems (electron-eletron or hiperfine), solves the total hamiltonian. Otherwise, sums the contributions to the total spectrum.
    
    Parameters
    ----------
    
    maham : Class
        Container for the hamiltonian parameters of the two systems.
    Expe : Class
        Container for the experimental conditions.
    Nucl1 : str
        Isotope of the first system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    Nucl2 : str
        Isotope of the second system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    graph : Bool
        PLots the spectrum of the multisystem.
    hifi : Bool
        Bool value to calculate the Jacobian matrix.
    ktra : float
        Reference value of number of transitions.
    ndev : int
		Number of devices
		
    Returns
    -------
    
    Blist : jax.np.array
        Array of the magnetic field.
    epc : jax.np.array
        Array of the counts of the spectrum.

    Example
    -------

    .. code-block:: python
    
       import matplotlib.pyplot as plt
       import epraya as epr
       import numpy as np
       Ham,Exp,_=epr.Jmstart()
       Ham.S1=3/2
       Ham.I1=1
       Ham.S2=1
       Ham.I2=1
       Ham.g1=np.array([2.003, 2, 2])
       Ham.g2=np.array([1.5, 1.5, 1.5])
       Ham.A1=np.array([200, 200, 200])  #Hyperfine constant
       Ham.D1=np.array([800,200])      #Zero field D and E
       Ham.Hpp=[0,10]
       Exp.Freq=9.4
       Exp.Points=4096
       Exp.Temperature=300
       Exp.Frange=[0,800]
       B,spc=epr.JMulpol(Ham,Exp)
       
    .. image:: /_static/jpol.png
       :alt: Plot of the JMulpol function
       :align: center
    '''
    Blist,epc=Jcalmulta(maham,Expe,Nucl1,Nucl2,M,hifi,ktra,ndev)
    if graph:
        plt.figure(figsize=(10,6))
        plt.plot(Blist,epc,color='navy',label='Spectrum')
        plt.xlabel('Magnetic field [mT]')
        formatter=EngFormatter(sep='') 
        plt.gca().yaxis.set_major_formatter(formatter)
        plt.ylabel('Counts [A. U.]')
        plt.xlim(Expe.Frange[0],Expe.Frange[1])
        plt.grid()
        plt.legend()
        plt.show(block=False)
    return Blist,epc
    

def Jcalmulta(maham,Expe,Nucl1='None',Nucl2='None',M=70,hifi=False,ktra=None,ndev=None):
    '''
    Determinates the spectrum for a two paramagnetic centers system. Follows the same logic from the function Jpowder, but adapted to the two centers system.
    
    Parameters
    ----------
    
    maham : Class
        Container for the hamiltonian parameters of the two systems.
    Expe : Class
        Container for the experimental conditions.
    Nucl1 : str
        Isotope of the first system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    Nucl2 : str
        Isotope of the second system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    graph : Bool
        PLots the spectrum of the multisystem.
    hifi : Bool
        Bool value to calculate the Jacobian matrix.
    ktra : float
        Reference value of number of transitions.
    ndev : int
		Number of devices
	
    Returns
    -------
    
    fielde : jax.np.array
        Array of the magnetic field.
    specs : jax.np.array
        Array of the counts of the spectrum.
    
    Example
    -------
    
    >>> import matplotlib.pyplot as plt
    >>> import epraya as epr
    >>> import numpy as np
    >>> Ham,Exp,_=epr.Jmstart()
    >>> Ham.S1=3/2
    >>> Ham.I1=1
    >>> Ham.S2=1
    >>> Ham.I2=1
    >>> Ham.g1=np.array([2.003, 2, 2])
    >>> Ham.g2=np.array([1.5, 1.5, 1.5])
    >>> Ham.A1=np.array([200, 200, 200])  #Hyperfine constant
    >>> Ham.D1=np.array([800,200])      #Zero field D and E
    >>> Ham.Hpp=[0,10]
    >>> Exp.Freq=9.4
    >>> Exp.Points=4096
    >>> Exp.Temperature=300
    >>> Exp.Frange=[0,800]
    >>> print(epr.Jcalmulta(Ham,Exp))
    (Array([0.00000000e+00, 1.95360195e-01, 3.90720391e-01, ...,
    7.99609280e+02, 7.99804640e+02, 8.00000000e+02], dtype=float64),
    Array([-1.51007828e-21,  1.87293010e-21,  1.91587492e-21, ...,
    3.33966042e-21,  1.25252035e-21, -1.98829450e-21], dtype=float64))
    '''
    maham.X1_2=np.asarray(maham.X1_2)
    maham.A1_2=np.asarray(maham.A1_2)
    maham.A2_1=np.asarray(maham.A2_1)
    Ham1=JHval()
    Ham2=JHval()
    Exp1=JEco()
    Exp2=JEco()
    Ham1.S,Ham1.g,Ham1.I,Ham1.L,Ham1.A,Ham1.Q,Ham1.D,Ham1.Bk2,Ham1.Bk4,Ham1.Bk6,Ham1.lc,Ham1.Hpp,Ham1.eta=maham.S1,maham.g1,maham.I1,maham.L1,maham.A1,maham.Q1,maham.D1,maham.Bk2,maham.Bk4,maham.Bk6,maham.lc1,maham.Hpp,maham.eta
    Ham2.S,Ham2.g,Ham2.I,Ham2.L,Ham2.A,Ham2.Q,Ham2.D,Ham2.lc,Ham2.Hpp,Ham2.eta=maham.S2,maham.g2,maham.I2,maham.L2,maham.A2,maham.Q2,maham.D2,maham.lc2,maham.Hpp,maham.eta
    Exp1.Freq,Exp1.Points,Exp1.Temperature,Exp1.Fdirection,Exp1.Frange,Exp1.Sampleframe,Exp1.Molframe,Exp1.gframe,Exp1.Aframe,Exp1.Dframe,Exp1.Qframe=Expe.Freq,Expe.Points,Expe.Temperature,Expe.Fdirection,Expe.Frange,Expe.Sampleframe1,Expe.Molframe1,Expe.gframe1,Expe.Aframe1,Expe.Dframe1,Expe.Qframe1
    Exp2.Freq,Exp2.Points,Exp2.Temperature,Exp2.Fdirection,Exp2.Frange,Exp2.Sampleframe,Exp2.Molframe,Exp2.gframe,Exp2.Aframe,Exp2.Dframe,Exp2.Qframe=Expe.Freq,Expe.Points,Expe.Temperature,Expe.Fdirection,Expe.Frange,Expe.Sampleframe2,Expe.Molframe2,Expe.gframe2,Expe.Aframe2,Expe.Dframe2,Expe.Qframe2
    iwas,jwas,kwas,weight,hulk=Delaunay(Exp1,M)
    iwas,jwas,kwas,weight,hulk=jxn.array(iwas),jxn.array(jwas),jxn.array(kwas),jxn.array(weight),jxn.array(hulk)
    if np.allclose(maham.X1_2,0.0) and np.allclose(maham.A1_2,0.0) and np.allclose(maham.A2_1,0.0):
        fielde,specs1=JCalpowder(Ham1,Exp1,iwas,jwas,kwas,weight,hulk,Nucl1,hifi=hifi,ktra=ktra,ndev=ndev)
        _,specs2=JCalpowder(Ham2,Exp2,iwas,jwas,kwas,weight,hulk,Nucl2,hifi=hifi,ktra=ktra,ndev=ndev)
        specs=specs1+specs2
    else:
        frange0=jxn.where(Exp1.Frange[0]<0.0,1e-4,Exp1.Frange[0])
        Ham1=Ham1.replace(A=jxn.asarray(Ham1.A)/1000.0,D=jxn.asarray(Ham1.D)/1000.0,Hpp=jxn.asarray(Ham1.Hpp)/1.0,Q=jxn.asarray(Ham1.Q)/1000.0,
                     Bk2=jxn.asarray(Ham1.Bk2)/1000.0,Bk4=jxn.asarray(Ham1.Bk4)/1000.0,Bk6=jxn.asarray(Ham1.Bk6)/1000.0)
        Ham2=Ham2.replace(A=jxn.asarray(Ham2.A)/1000.0,D=jxn.asarray(Ham2.D)/1000.0,Hpp=jxn.asarray(Ham2.Hpp)/1.0,Q=jxn.asarray(Ham2.Q)/1000.0,
                     Bk2=jxn.asarray(Ham2.Bk2)/1000.0,Bk4=jxn.asarray(Ham2.Bk4)/1000.0,Bk6=jxn.asarray(Ham2.Bk6)/1000.0)
        etas=Ham1.eta
        etas=jxn.where(Ham1.Hpp[1]==0.0,0.0,etas)
        etas=jxn.where(Ham1.Hpp[0]==0.0,1.0,etas)
        Ham1=Ham1.replace(eta=etas)
        Ham2=Ham2.replace(eta=etas)
        dim=int(2*Ham1.S+1)*int(2*Ham1.I+1)*int(2*Ham1.L+1)*int(2*Ham2.S+1)*int(2*Ham2.I+1)*int(2*Ham2.L+1)
        dim1=int(2*Ham1.S+1)*int(2*Ham1.I+1)*int(2*Ham1.L+1)
        dim2=int(2*Ham2.S+1)*int(2*Ham2.I+1)*int(2*Ham2.L+1)
        Ham1=Jchaframe(Ham1,Exp1)
        Ham2=Jchaframe(Ham2,Exp2)
        sx1,sy1,sz1=JPauli(Ham1.S)
        ix1,iy1,iz1=JPauli(Ham1.I)
        sx2,sy2,sz2=JPauli(Ham2.S)
        ix2,iy2,iz2=JPauli(Ham2.I)
        beta=(scic.physical_constants["Bohr magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
        betan=(scic.physical_constants["nuclear magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
        h1=np.zeros((dim,dim),dtype='complex')
        hzex=0
        hzey=0
        hzez=0
        hzex1=jxn.asarray(beta*JHze(sx1,sy1,sz1,Ham1.g,[1,0,0],dim1),dtype=complex)
        hzey1=jxn.asarray(beta*JHze(sx1,sy1,sz1,Ham1.g,[0,1,0],dim1),dtype=complex)
        hzez1=jxn.asarray(beta*JHze(sx1,sy1,sz1,Ham1.g,[0,0,1],dim1),dtype=complex)
        hzex2=jxn.asarray(beta*JHze(sx2,sy2,sz2,Ham2.g,[1,0,0],dim2),dtype=complex)
        hzey2=jxn.asarray(beta*JHze(sx2,sy2,sz2,Ham2.g,[0,1,0],dim2),dtype=complex)
        hzez2=jxn.asarray(beta*JHze(sx2,sy2,sz2,Ham2.g,[0,0,1],dim2),dtype=complex)

        hzex+=jxn.kron(hzex1,jxn.eye(dim2,dtype=complex))+jxn.kron(jxn.eye(dim1,dtype=complex),hzex2)
        hzey+=jxn.kron(hzey1,jxn.eye(dim2,dtype=complex))+jxn.kron(jxn.eye(dim1,dtype=complex),hzey2)
        hzez+=jxn.kron(hzez1,jxn.eye(dim2,dtype=complex))+jxn.kron(jxn.eye(dim1,dtype=complex),hzez2)
        if Ham1.S>=1:
            h1=h1+JStevensrot(sx1,sy1,sz1,Ham1.S,Ham1,dim,Exp1)
        if Ham2.S>=1:
            h1=h1+JStevensrot(sx2,sy2,sz2,Ham2.S,Ham2,dim,Exp2)
        if Ham1.L!=0:
            h1=h1+JLorbit(sx1,sy1,sz1,Ham1.lc,dim,Ham1.L)
        if Ham2.L!=0:
            h1=h1+JLorbit(sx2,sy2,sz2,Ham2.lc,dim,Ham2.L)
        if Ham1.I!=0:
            h1=h1+JHfi(sx1,sy1,sz1,ix1,iy1,iz1,Ham1.A,dim)
            h1=h1+JQii(ix1,iy1,iz1,Ham1.Q,dim)
            gnk=gnfactor(Nucl1)
            nhzex=jxn.asarray(betan*JNhze(Ham1.I,ix1,iy1,iz1,dim1,gnk,[1,0,0]),dtype=complex)
            nhzey=jxn.asarray(betan*JNhze(Ham1.I,ix1,iy1,iz1,dim1,gnk,[0,1,0]),dtype=complex)
            nhzez=jxn.asarray(betan*JNhze(Ham1.I,ix1,iy1,iz1,dim1,gnk,[0,0,1]),dtype=complex)
            hzex-=jxn.kron(nhzex,jxn.eye(dim2,dtype=complex))
            hzey-=jxn.kron(nhzey,jxn.eye(dim2,dtype=complex))
            hzez-=jxn.kron(nhzez,jxn.eye(dim2,dtype=complex))
        if Ham2.I!=0:
            h1=h1+JHfi(sx2,sy2,sz2,ix2,iy2,iz2,Ham2.A,dim)
            h1=h1+JQii(ix2,iy2,iz2,Ham2.Q,dim)
            gnk=gnfactor(Nucl2)
            nhzex=jxn.asarray(betan*JNhze(Ham2.I,ix2,iy2,iz2,dim2,gnk,[1,0,0]),dtype=complex)
            nhzey=jxn.asarray(betan*JNhze(Ham2.I,ix2,iy2,iz2,dim2,gnk,[0,1,0]),dtype=complex)
            nhzez=jxn.asarray(betan*JNhze(Ham2.I,ix2,iy2,iz2,dim2,gnk,[0,0,1]),dtype=complex)
            hzex-=jxn.kron(jxn.eye(dim1,dtype=complex),nhzex)
            hzey-=jxn.kron(jxn.eye(dim1,dtype=complex),nhzey)
            hzez-=jxn.kron(jxn.eye(dim1,dtype=complex),nhzez)
        if jxn.any(maham.A1_2):
            Aref=jxn.asarray(maham.A1_2)/1000.0
            Aref=Aref*jxn.eye(3)
            h1+=JHfi(sx1,sy1,sz1,ix2,iy2,iz2,Aref,dim)
        if jxn.any(maham.A2_1):
            Aref=jxn.asarray(maham.A2_1)/1000.0
            Aref=Aref*jxn.eye(3)
            h1+=JHfi(sx2,sy2,sz2,ix1,iy1,iz1,Aref,dim)
        if jxn.any(maham.X1_2):
            Xref=jxn.asarray(maham.X1_2)/1000.0
            Xref=Xref*jxn.eye(3)
            h1+=JIee(sx1,sy1,sz1,sx2,sy2,sz2,Xref,dim)
        h1=jxn.asarray(h1,dtype=complex)
        #For the total magnetic moment of the system
        stodx=np.zeros((dim,dim),dtype='complex')
        stody=np.zeros((dim,dim),dtype='complex')
        stodz=np.zeros((dim,dim),dtype='complex')
        dimwns1=int((2*Ham1.I+1)*(2*Ham1.L+1))
        dimwns2=int((2*Ham2.I+1)*(2*Ham2.L+1))
        if Ham1.S>0:
            gx1,gy1,gz1=[Ham1.g[a,0]*sx1+Ham1.g[a,1]*sy1+Ham1.g[a,2]*sz1 for a in range(3)]
            sx1r=jxn.kron(gx1,jxn.eye(dimwns1,dtype=complex))
            sy1r=jxn.kron(gy1,jxn.eye(dimwns1,dtype=complex))
            sz1r=jxn.kron(gz1,jxn.eye(dimwns1,dtype=complex))
            stodx+=jxn.kron(sx1r,jxn.eye(dim2,dtype=complex))
            stody+=jxn.kron(sy1r,jxn.eye(dim2,dtype=complex))
            stodz+=jxn.kron(sz1r,jxn.eye(dim2,dtype=complex))
        if Ham2.S>0:
            gx2,gy2,gz2=[Ham2.g[a,0]*sx2+Ham2.g[a,1]*sy2+Ham2.g[a,2]*sz2 for a in range(3)]
            sx2r=jxn.kron(gx2,jxn.eye(dimwns2,dtype=complex))
            sy2r=jxn.kron(gy2,jxn.eye(dimwns2,dtype=complex))
            sz2r=jxn.kron(gz2,jxn.eye(dimwns2,dtype=complex))
            stodx+=jxn.kron(jxn.eye(dim1,dtype=complex),sx2r)
            stody+=jxn.kron(jxn.eye(dim1,dtype=complex),sy2r)
            stodz+=jxn.kron(jxn.eye(dim1,dtype=complex),sz2r)
        Blist1=jxn.linspace(frange0,Exp1.Frange[1],500)
        dB=(Exp1.Frange[1]-Exp1.Frange[0])/(Exp1.Points-1)
        Bmin=Exp1.Frange[0]
        P=dict(B=Blist1,h1=h1,hx=hzex,hy=hzey,hz=hzez,isx=stodx,isy=stody,isz=stodz,Freq=jxn.asarray(Exp1.Freq,float),Temp=jxn.asarray(Exp1.Temperature,float),Hpp=Ham1.Hpp)
        allres,allint,ntrans,K=Jorientblocks(P,iwas,jwas,kwas,dim,hifi,ktra,ndev)
        sketch=JCaltriangle(Bmin,dB,allres,allint,ntrans,hulk,weight,Exp1.Points)
        specs=JVoigtp(sketch,dB,Ham1.Hpp)
        fielde=jxn.linspace(Exp1.Frange[0],Exp1.Frange[1],Exp1.Points)
    return fielde,specs

def JMusic(maham,Expe,Nucl1='None',Nucl2='None',graph=True,hifi=False):
    '''
    Wrap function for the simulation of the EPR spectrum for monocristal samples of two systems. If there is an interaction between the systems (electron-eletron or hiperfine), solves the total hamiltonian. Otherwise, sums the contributions to the total spectrum.
    
    Parameters
    ----------
    maham : Class
        Container for the hamiltonian parameters of the two systems.
    Expe : Class
        Container for the experimental conditions.
    Nucl1 : str
        Isotope of the first system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    Nucl2 : str
        Isotope of the second system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    graph : Bool
        PLots the spectrum of the multisystem.
    hifi : Bool
        Bool value to calculate the Jacobian matrix.
    Returns
    -------
    
    Blist : jax.np.array
        Array of the magnetic field.
    epc : jax.np.array
        Array of the counts of the spectrum.

    Example
    -------

    .. code-block:: python
    
       import matplotlib.pyplot as plt
       import epraya as epr
       import numpy as np
       Ham,Exp,_=epr.Jmstart()
       Ham.S1=3/2
       Ham.I1=1
       Ham.S2=1
       Ham.I2=1
       Ham.g1=np.array([2.003, 2, 2])
       Ham.g2=np.array([1.5, 1.5, 1.5])
       Ham.A1=np.array([200, 200, 200])  #Hyperfine constant
       Ham.D1=np.array([800,200])      #Zero field D and E
       Ham.Hpp=[0,10]
       Exp.Freq=9.4
       Exp.Points=4096
       Exp.Temperature=300
       Exp.Frange=[0,800]
       B,spc=epr.JMusic(Ham,Exp)
       
    .. image:: /_static/jmusic.png
       :alt: Plot of the Jmusic function
       :align: center
    '''
    Blist,epc=Jcalmusic(maham,Expe,Nucl1,Nucl2,hifi=hifi)
    if graph:
        plt.figure(figsize=(10,6))
        plt.plot(Blist,epc,color='navy',label='Spectrum')
        plt.xlabel('Magnetic field [mT]')
        plt.ylabel('Counts [A. U.]')
        plt.xlim(Expe.Frange[0],Expe.Frange[1])
        plt.grid()
        plt.legend()
        plt.show(block=False)
    return Blist,epc

def Jevoigt(rfields,inten,Blist,Hpp):
    '''
    Function for the calculation of the voigtian profile of singlecrystal samples. Follows the same structure that Voigtconv, making the convolution of the spectrum, using the resonant fields as reference.

    Parameters
    ----------

    rfields : jax.np.array
        Array of the resonant fields of the system.
    inten : jax.np.array
        Array of the intensities related to the resonant fields.
    Blist : jax.np.array
        Array of the magnetic field of the spectrum.
    Hpp : list
        List of the gaussian and lorentzian contributions to the spectrum.

    Returns 
    -------
    
    spc : jax.np.array
        EPR spectrum.
    
    '''
    n=len(Blist)
    dB=(Blist[-1]-Blist[0])/(n-1)
    npad=2*n
    k=2*jxn.pi*jxn.arange(npad//2+1)/(npad*dB)
    #Voigt function definition
    fvoigt=(1j*k)*jxn.exp(-0.5*((Hpp[0]/2.0)*k)**2-(jxn.sqrt(3.0)/2.0*Hpp[1])*jxn.abs(k))
    S=jxn.exp(-1j*jxn.outer(k,jxn.asarray(rfields,float)-Blist[0]))@jxn.asarray(inten,float)
    return jxn.fft.irfft(S*fvoigt,n=npad)[:n]/dB


def Jcalmusic(maham,Expe,Nucl1='None',Nucl2='None',hifi=False):
    '''
    Determinates the spectrum for a two paramagnetic centers monocristal system. Follows the same logic from the function Jresonant, but adapted to the two centers system.
    
    
    Parameters
    ----------
    
    maham : Class
        Container for the hamiltonian parameters of the two systems.
    Expe : Class
        Container for the experimental conditions.
    Nucl1 : str
        Isotope of the first system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    Nucl2 : str
        Isotope of the second system. Can be the quantum number and the element or only the element ('55Mn' or 'Mn') 
    graph : Bool
        PLots the spectrum of the multisystem.
    hifi : Bool
        Bool value to calculate the Jacobian matrix.
    Returns
    -------
    
    fielde : jax.np.array
        Array of the magnetic field.
        
    specs : jax.np.array
        Array of the counts of the spectrum.
    
    Example
    -------
    
    >>> import matplotlib.pyplot as plt
    >>> import epraya as epr
    >>> import numpy as np
    >>> Ham,Exp,_=epr.Jmstart()
    >>> Ham.S1=3/2
    >>> Ham.I1=1
    >>> Ham.S2=1
    >>> Ham.I2=1
    >>> Ham.g1=np.array([2.003, 2, 2])
    >>> Ham.g2=np.array([1.5, 1.5, 1.5])
    >>> Ham.A1=np.array([200, 200, 200])  #Hyperfine constant
    >>> Ham.D1=np.array([800,200])      #Zero field D and E
    >>> Ham.Hpp=[0,10]
    >>> Exp.Freq=9.4
    >>> Exp.Points=4096
    >>> Exp.Temperature=300
    >>> Exp.Frange=[0,800]
    >>> print(epr.Jcalmusic(Ham,Exp))
    (Array([0.00000000e+00, 1.95360195e-01, 3.90720391e-01, ...,
    7.99609280e+02, 7.99804640e+02, 8.00000000e+02], dtype=float64),
    Array([-1.60669060e-07, -9.66774880e-08, -5.69966581e-08, ...,
    -2.01984373e-07, -2.01714437e-07, -2.01579599e-07], dtype=float64))
    '''
    maham.X1_2=np.asarray(maham.X1_2)
    maham.A1_2=np.asarray(maham.A1_2)
    maham.A2_1=np.asarray(maham.A2_1)
    Ham1=JHval()
    Ham2=JHval()
    Ham1.S,Ham1.g,Ham1.I,Ham1.L,Ham1.A,Ham1.Q,Ham1.D,Ham1.Bk2,Ham1.Bk4,Ham1.Bk6,Ham1.lc,Ham1.Hpp,Ham1.eta=maham.S1,maham.g1,maham.I1,maham.L1,maham.A1,maham.Q1,maham.D1,maham.Bk2,maham.Bk4,maham.Bk6,maham.lc1,maham.Hpp,maham.eta
    Ham2.S,Ham2.g,Ham2.I,Ham2.L,Ham2.A,Ham2.Q,Ham2.D,Ham2.lc,Ham2.Hpp,Ham2.eta=maham.S2,maham.g2,maham.I2,maham.L2,maham.A2,maham.Q2,maham.D2,maham.lc2,maham.Hpp,maham.eta
    Exp1=JEco()
    Exp2=JEco()
    Exp1.Freq,Exp1.Points,Exp1.Temperature,Exp1.Fdirection,Exp1.Mwdirection,Exp1.Frange,Exp1.Sampleframe,Exp1.Molframe,Exp1.gframe,Exp1.Aframe,Exp1.Dframe,Exp1.Qframe=Expe.Freq,Expe.Points,Expe.Temperature,Expe.Fdirection,Expe.Mwdirection,Expe.Frange,Expe.Sampleframe1,Expe.Molframe1,Expe.gframe1,Expe.Aframe1,Expe.Dframe1,Expe.Qframe1
    Exp2.Freq,Exp2.Points,Exp2.Temperature,Exp2.Fdirection,Exp2.Mwdirection,Exp2.Frange,Exp2.Sampleframe,Exp2.Molframe,Exp2.gframe,Exp2.Aframe,Exp2.Dframe,Exp2.Qframe=Expe.Freq,Expe.Points,Expe.Temperature,Expe.Fdirection,Expe.Mwdirection,Expe.Frange,Expe.Sampleframe2,Expe.Molframe2,Expe.gframe2,Expe.Aframe2,Expe.Dframe2,Expe.Qframe2

    if np.allclose(maham.X1_2,0.0) and np.allclose(maham.A1_2,0.0) and np.allclose(maham.A2_1,0.0):
        fielde,specs1,_,_,_,_,_=Calresonant(Ham1,Exp1,Nucl1,hifi=hifi)
        _,specs2,_,_,_,_,_=Calresonant(Ham2,Exp2,Nucl2,hifi=hifi)
        specs=specs1+specs2
    else:
        frange0=jxn.where(Exp1.Frange[0]<0.0,1e-4,Exp1.Frange[0])
        Ham1=Ham1.replace(A=jxn.asarray(Ham1.A)/1000.0,D=jxn.asarray(Ham1.D)/1000.0,Hpp=jxn.asarray(Ham1.Hpp)/1.0,Q=jxn.asarray(Ham1.Q)/1000.0,
                     Bk2=jxn.asarray(Ham1.Bk2)/1000.0,Bk4=jxn.asarray(Ham1.Bk4)/1000.0,Bk6=jxn.asarray(Ham1.Bk6)/1000.0)
        Ham2=Ham2.replace(A=jxn.asarray(Ham2.A)/1000.0,D=jxn.asarray(Ham2.D)/1000.0,Hpp=jxn.asarray(Ham2.Hpp)/1.0,Q=jxn.asarray(Ham2.Q)/1000.0,
                     Bk2=jxn.asarray(Ham2.Bk2)/1000.0,Bk4=jxn.asarray(Ham2.Bk4)/1000.0,Bk6=jxn.asarray(Ham2.Bk6)/1000.0)
        etas=Ham1.eta
        etas=jxn.where(Ham1.Hpp[1]==0.0,0.0,etas)
        etas=jxn.where(Ham1.Hpp[0]==0.0,1.0,etas)
        Ham1=Ham1.replace(eta=etas)
        Ham2=Ham2.replace(eta=etas)
        dim=int(2*Ham1.S+1)*int(2*Ham1.I+1)*int(2*Ham1.L+1)*int(2*Ham2.S+1)*int(2*Ham2.I+1)*int(2*Ham2.L+1)
        dim1=int(2*Ham1.S+1)*int(2*Ham1.I+1)*int(2*Ham1.L+1)
        dim2=int(2*Ham2.S+1)*int(2*Ham2.I+1)*int(2*Ham2.L+1)
        ndir=jxn.array(Exp1.Fdirection,dtype=jxn.float64)
        ndir=ndir/jxn.linalg.norm(ndir)
        nx,ny,nz=ndir[0],ndir[1],ndir[2]
        mdir=jxn.array(Exp1.Mwdirection,dtype=jxn.float64)
        mdir=mdir/jxn.linalg.norm(mdir)
        mx,my,mz=mdir[0],mdir[1],mdir[2]
        Ham1=Jchaframe(Ham1,Exp1)
        Ham2=Jchaframe(Ham2,Exp2)
        sx1,sy1,sz1=JPauli(Ham1.S)
        ix1,iy1,iz1=JPauli(Ham1.I)
        sx2,sy2,sz2=JPauli(Ham2.S)
        ix2,iy2,iz2=JPauli(Ham2.I)
        E=Exp1.Freq
        beta=(scic.physical_constants["Bohr magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
        betan=(scic.physical_constants["nuclear magneton"][0]/scic.physical_constants["Planck constant"][0])/1e12
        h1=np.zeros((dim,dim),dtype='complex')
        hzex=0
        hzey=0
        hzez=0
        hzex1=jxn.asarray(beta*JHze(sx1,sy1,sz1,Ham1.g,[1,0,0],dim1),dtype=complex)
        hzey1=jxn.asarray(beta*JHze(sx1,sy1,sz1,Ham1.g,[0,1,0],dim1),dtype=complex)
        hzez1=jxn.asarray(beta*JHze(sx1,sy1,sz1,Ham1.g,[0,0,1],dim1),dtype=complex)
        hzex2=jxn.asarray(beta*JHze(sx2,sy2,sz2,Ham2.g,[1,0,0],dim2),dtype=complex)
        hzey2=jxn.asarray(beta*JHze(sx2,sy2,sz2,Ham2.g,[0,1,0],dim2),dtype=complex)
        hzez2=jxn.asarray(beta*JHze(sx2,sy2,sz2,Ham2.g,[0,0,1],dim2),dtype=complex)

        hzex+=jxn.kron(hzex1,jxn.eye(dim2,dtype=complex))+jxn.kron(jxn.eye(dim1,dtype=complex),hzex2)
        hzey+=jxn.kron(hzey1,jxn.eye(dim2,dtype=complex))+jxn.kron(jxn.eye(dim1,dtype=complex),hzey2)
        hzez+=jxn.kron(hzez1,jxn.eye(dim2,dtype=complex))+jxn.kron(jxn.eye(dim1,dtype=complex),hzez2)
        if Ham1.S>=1:
            h1=h1+JStevensrot(sx1,sy1,sz1,Ham1.S,Ham1,dim,Exp1)
        if Ham2.S>=1:
            h1=h1+JStevensrot(sx2,sy2,sz2,Ham2.S,Ham2,dim,Exp2)
        if Ham1.L!=0:
            h1=h1+JLorbit(sx1,sy1,sz1,Ham1.lc,dim,Ham1.L)
        if Ham2.L!=0:
            h1=h1+JLorbit(sx2,sy2,sz2,Ham2.lc,dim,Ham2.L)
        if Ham1.I!=0:
            h1=h1+JHfi(sx1,sy1,sz1,ix1,iy1,iz1,Ham1.A,dim)
            h1=h1+JQii(ix1,iy1,iz1,Ham1.Q,dim)
            gnk=gnfactor(Nucl1)
            nhzex=jxn.asarray(betan*JNhze(Ham1.I,ix1,iy1,iz1,dim1,gnk,[1,0,0]),dtype=complex)
            nhzey=jxn.asarray(betan*JNhze(Ham1.I,ix1,iy1,iz1,dim1,gnk,[0,1,0]),dtype=complex)
            nhzez=jxn.asarray(betan*JNhze(Ham1.I,ix1,iy1,iz1,dim1,gnk,[0,0,1]),dtype=complex)
            hzex-=jxn.kron(nhzex,jxn.eye(dim2,dtype=complex))
            hzey-=jxn.kron(nhzey,jxn.eye(dim2,dtype=complex))
            hzez-=jxn.kron(nhzez,jxn.eye(dim2,dtype=complex))
        if Ham2.I!=0:
            h1=h1+JHfi(sx2,sy2,sz2,ix2,iy2,iz2,Ham2.A,dim)
            h1=h1+JQii(ix2,iy2,iz2,Ham2.Q,dim)
            gnk=gnfactor(Nucl2)
            nhzex=jxn.asarray(betan*JNhze(Ham2.I,ix2,iy2,iz2,dim2,gnk,[1,0,0]),dtype=complex)
            nhzey=jxn.asarray(betan*JNhze(Ham2.I,ix2,iy2,iz2,dim2,gnk,[0,1,0]),dtype=complex)
            nhzez=jxn.asarray(betan*JNhze(Ham2.I,ix2,iy2,iz2,dim2,gnk,[0,0,1]),dtype=complex)
            hzex-=jxn.kron(jxn.eye(dim1,dtype=complex),nhzex)
            hzey-=jxn.kron(jxn.eye(dim1,dtype=complex),nhzey)
            hzez-=jxn.kron(jxn.eye(dim1,dtype=complex),nhzez)
        if jxn.any(maham.A1_2):
            Aref=jxn.asarray(maham.A1_2)/1000.0
            Aref=Aref*jxn.eye(3)
            h1+=JHfi(sx1,sy1,sz1,ix2,iy2,iz2,Aref,dim)
        if jxn.any(maham.A2_1):
            Aref=jxn.asarray(maham.A2_1)/1000.0
            Aref=Aref*jxn.eye(3)
            h1+=JHfi(sx2,sy2,sz2,ix1,iy1,iz1,Aref,dim)
        if jxn.any(maham.X1_2):
            Xref=jxn.asarray(maham.X1_2)/1000.0
            Xref=Xref*jxn.eye(3)
            h1+=JIee(sx1,sy1,sz1,sx2,sy2,sz2,Xref,dim)
        h1=jxn.asarray(h1,dtype=complex)
        #For the total magnetic moment of the system
        stodx=np.zeros((dim,dim),dtype='complex')
        stody=np.zeros((dim,dim),dtype='complex')
        stodz=np.zeros((dim,dim),dtype='complex')
        dimwns1=int((2*Ham1.I+1)*(2*Ham1.L+1))
        dimwns2=int((2*Ham2.I+1)*(2*Ham2.L+1))
        if Ham1.S>0:
            gx1,gy1,gz1=[Ham1.g[a,0]*sx1+Ham1.g[a,1]*sy1+Ham1.g[a,2]*sz1 for a in range(3)]
            sx1r=jxn.kron(gx1,jxn.eye(dimwns1,dtype=complex))
            sy1r=jxn.kron(gy1,jxn.eye(dimwns1,dtype=complex))
            sz1r=jxn.kron(gz1,jxn.eye(dimwns1,dtype=complex))
            stodx+=jxn.kron(sx1r,jxn.eye(dim2,dtype=complex))
            stody+=jxn.kron(sy1r,jxn.eye(dim2,dtype=complex))
            stodz+=jxn.kron(sz1r,jxn.eye(dim2,dtype=complex))
        if Ham2.S>0:
            gx2,gy2,gz2=[Ham2.g[a,0]*sx2+Ham2.g[a,1]*sy2+Ham2.g[a,2]*sz2 for a in range(3)]
            sx2r=jxn.kron(gx2,jxn.eye(dimwns2,dtype=complex))
            sy2r=jxn.kron(gy2,jxn.eye(dimwns2,dtype=complex))
            sz2r=jxn.kron(gz2,jxn.eye(dimwns2,dtype=complex))
            stodx+=jxn.kron(jxn.eye(dim1,dtype=complex),sx2r)
            stody+=jxn.kron(jxn.eye(dim1,dtype=complex),sy2r)
            stodz+=jxn.kron(jxn.eye(dim1,dtype=complex),sz2r)
        hze=nx*hzex+ny*hzey+nz*hzez
        Blist1=jxn.linspace(frange0,Exp1.Frange[1],500)
        Elist,Vlist,h2=JPadaptarray(Blist1,h1,hzex,hzey,hzez,nx,ny,nz,hifi)
        resfield,intensy,_=JNresina(Blist1,Elist,Vlist,dim,Exp1.Freq,stodx,stody,stodz,nx,ny,nz,
                                    Exp1.Temperature,Ham1.Hpp,h2,hifi=hifi,mw=mdir)
        fielde=jxn.linspace(Exp1.Frange[0],Exp1.Frange[1],Exp1.Points)
        specs=Jevoigt(resfield,intensy,fielde,Ham1.Hpp)
    return fielde,specs
 
def Showparam(param,Vary):
    if 'gx' in param.keys():
        gxf1=Vary.g[0]+(Vary.g[1]-Vary.g[0])*param['gx']
        gyf1=Vary.g[2]+(Vary.g[3]-Vary.g[2])*param['gy']
        gzf1=Vary.g[4]+(Vary.g[5]-Vary.g[4])*param['gz']
        print(f"| gx: {gxf1:.4f} | gy: {gyf1:.4f} | gz: {gzf1:.4f} |")
    if 'Ax' in param.keys():
        Axf1=Vary.A[0]+(Vary.A[1]-Vary.A[0])*param['Ax']
        Ayf1=Vary.A[2]+(Vary.A[3]-Vary.A[2])*param['Ay']
        Azf1=Vary.A[4]+(Vary.A[5]-Vary.A[4])*param['Az']
        print(f"| Ax: {Axf1:.4f} | Ay: {Ayf1:.4f} | Az: {Azf1:.4f} |")
    if 'D' in param.keys():
        Dx1=Vary.D[0]+(Vary.D[1]-Vary.D[0])*param['D']
        Ey1=Vary.D[2]+(Vary.D[3]-Vary.D[2])*param['E']
        print(f"| D: {Dx1:.1f} | E: {Ey1:.1f} |")
    if 'Qx' in param.keys():
        Qxf1=Vary.Q[0]+(Vary.Q[1]-Vary.Q[0])*param['Qx']
        Qyf1=Vary.Q[2]+(Vary.Q[3]-Vary.Q[2])*param['Qy']
        Qzf1=Vary.Q[4]+(Vary.Q[5]-Vary.Q[4])*param['Qz']
        print(f"| Qx: {Qxf1:.4f} | Qy: {Qyf1:.4f} | Qz: {Qzf1:.4f} |")
    if 'Hpp1' in param.keys():
        Hppx=Vary.Hpp[0]+(Vary.Hpp[1]-Vary.Hpp[0])*param['Hpp1']
        Hppy=Vary.Hpp[2]+(Vary.Hpp[3]-Vary.Hpp[2])*param['Hpp2']
        print(f"| Hppg: {Hppx:.1f} | Hppl: {Hppy:.1f} |") 
        
def Showparam2(param,Vary):
    if 'gx1' in param.keys():
        gxf1=Vary.g1[0]+(Vary.g1[1]-Vary.g1[0])*param['gx1']
        gyf1=Vary.g1[2]+(Vary.g1[3]-Vary.g1[2])*param['gy1']
        gzf1=Vary.g1[4]+(Vary.g1[5]-Vary.g1[4])*param['gz1']
        print(f"| gx1: {gxf1:.4f} | gy1: {gyf1:.4f} | gz1: {gzf1:.4f} |")
    if 'gx2' in param.keys():
        gxf2=Vary.g2[0]+(Vary.g2[1]-Vary.g2[0])*param['gx2']
        gyf2=Vary.g2[2]+(Vary.g2[3]-Vary.g2[2])*param['gy2']
        gzf2=Vary.g2[4]+(Vary.g2[5]-Vary.g2[4])*param['gz2']
        print(f"| gx2: {gxf2:.4f} | gy2: {gyf2:.4f} | gz2: {gzf2:.4f} |")
    if 'Ax1' in param.keys():
        Axf1=Vary.A1[0]+(Vary.A1[1]-Vary.A1[0])*param['Ax1']
        Ayf1=Vary.A1[2]+(Vary.A1[3]-Vary.A1[2])*param['Ay1']
        Azf1=Vary.A1[4]+(Vary.A1[5]-Vary.A1[4])*param['Az1']
        print(f"| Ax1: {Axf1:.4f} | Ay1: {Ayf1:.4f} | Az1: {Azf1:.4f} |")
    if 'Ax2' in param.keys():
        Axf2=Vary.A2[0]+(Vary.A2[1]-Vary.A2[0])*param['Ax2']
        Ayf2=Vary.A2[2]+(Vary.A2[3]-Vary.A2[2])*param['Ay2']
        Azf2=Vary.A2[4]+(Vary.A2[5]-Vary.A2[4])*param['Az2']
        print(f"| Ax2: {Axf2:.4f} | Ay2: {Ayf2:.4f} | Az2: {Azf2:.4f} |")
    if 'D1' in param.keys():
        Dx1=Vary.D1[0]+(Vary.D1[1]-Vary.D1[0])*param['D1']
        Ey1=Vary.D1[2]+(Vary.D1[3]-Vary.D1[2])*param['E1']
        print(f"| D1: {Dx1:.1f} | E1: {Ey1:.1f} |")
    if 'D2' in param.keys():
        Dx2=Vary.D2[0]+(Vary.D2[1]-Vary.D2[0])*param['D2']
        Ey2=Vary.D2[2]+(Vary.D2[3]-Vary.D2[2])*param['E2']
        print(f"| D2: {Dx2:.1f} | E2: {Ey2:.1f} |")
    if 'Qx1' in param.keys():
        Qxf1=Vary.Q1[0]+(Vary.Q1[1]-Vary.Q1[0])*param['Qx1']
        Qyf1=Vary.Q1[2]+(Vary.Q1[3]-Vary.Q1[2])*param['Qy1']
        Qzf1=Vary.Q1[4]+(Vary.Q1[5]-Vary.Q1[4])*param['Qz1']
        print(f"| Qx1: {Qxf1:.4f} | Qy1: {Qyf1:.4f} | Qz1: {Qzf1:.4f} |")
    if 'Qx2' in param.keys():
        Qxf2=Vary.Q2[0]+(Vary.Q2[1]-Vary.Q2[0])*param['Qx2']
        Qyf2=Vary.Q2[2]+(Vary.Q2[3]-Vary.Q2[2])*param['Qy2']
        Qzf2=Vary.Q2[4]+(Vary.Q2[5]-Vary.Q2[4])*param['Qz2']
        print(f"| Qx2: {Qxf2:.4f} | Qy2: {Qyf2:.4f} | Qz2: {Qzf2:.4f} |")
    if 'Hpp1' in param.keys():
        Hppx=Vary.Hpp[0]+(Vary.Hpp[1]-Vary.Hpp[0])*param['Hpp1']
        Hppy=Vary.Hpp[2]+(Vary.Hpp[3]-Vary.Hpp[2])*param['Hpp2']
        print(f"| Hppg: {Hppx:.1f} | Hppl: {Hppy:.1f} |")

def Fromsigtophy(param,Ham,Vary):
    pravals={}
    if 'gx' in param.keys():
        gx=Vary.g[0]+(Vary.g[1]-Vary.g[0])*param['gx']
        gy=Vary.g[2]+(Vary.g[3]-Vary.g[2])*param['gy']
        gz=Vary.g[4]+(Vary.g[5]-Vary.g[4])*param['gz']
        gg=jxn.array([gx,gy,gz])
        pravals['g']=gg
    else:
        gg=Ham.g
    if 'Ax' in param.keys():
        Ax=Vary.A[0]+(Vary.A[1]-Vary.A[0])*param['Ax']
        Ay=Vary.A[2]+(Vary.A[3]-Vary.A[2])*param['Ay']
        Az=Vary.A[4]+(Vary.A[5]-Vary.A[4])*param['Az']
        AA=jxn.array([Ax,Ay,Az])
        pravals['A']=AA
    else:
        AA=Ham.A
    if 'D' in param.keys():
        Dx=Vary.D[0]+(Vary.D[1]-Vary.D[0])*param['D']
        Ey=Vary.D[2]+(Vary.D[3]-Vary.D[2])*param['E']
        DD=jxn.array([Dx,Ey])
        pravals['D']=DD
    else:
        DD=Ham.D
    if 'Qx' in param.keys():
        Qx=Vary.Q[0]+(Vary.Q[1]-Vary.Q[0])*param['Qx']
        Qy=Vary.Q[2]+(Vary.Q[3]-Vary.Q[2])*param['Qy']
        Qz=Vary.Q[4]+(Vary.Q[5]-Vary.Q[4])*param['Qz']
        QQ=jxn.array([Qx,Qy,Qz])
        pravals['Q']=QQ
    else:
        QQ=Ham.Q
    if 'Hpp1' in param.keys():
        Hppx=Vary.Hpp[0]+(Vary.Hpp[1]-Vary.Hpp[0])*param['Hpp1']
        Hppy=Vary.Hpp[2]+(Vary.Hpp[3]-Vary.Hpp[2])*param['Hpp2']
        HHpp=jxn.array([Hppx,Hppy])
        pravals['Hpp']=HHpp
    else:
        HHpp=Ham.Hpp
    Hat=Ham.replace(g=gg,A=AA,D=DD,Q=QQ,Hpp=HHpp)
    return Hat,pravals

def Fromsigtophy2(param,Ham,Vary):
    pravals={}
    if 'gx1' in param.keys():
        gx1=Vary.g1[0]+(Vary.g1[1]-Vary.g1[0])*param['gx1']
        gy1=Vary.g1[2]+(Vary.g1[3]-Vary.g1[2])*param['gy1']
        gz1=Vary.g1[4]+(Vary.g1[5]-Vary.g1[4])*param['gz1']
        gg1=jxn.array([gx1,gy1,gz1])
        pravals['g1']=gg1
    else:
        gg1=Ham.g1
    if 'gx2' in param.keys():
        gx2=Vary.g2[0]+(Vary.g2[1]-Vary.g2[0])*param['gx2']
        gy2=Vary.g2[2]+(Vary.g2[3]-Vary.g2[2])*param['gy2']
        gz2=Vary.g2[4]+(Vary.g2[5]-Vary.g2[4])*param['gz2']
        gg2=jxn.array([gx2,gy2,gz2])
        pravals['g2']=gg2
    else:
        gg2=Ham.g2
    if 'Ax1' in param.keys():
        Ax1=Vary.A1[0]+(Vary.A1[1]-Vary.A1[0])*param['Ax1']
        Ay1=Vary.A1[2]+(Vary.A1[3]-Vary.A1[2])*param['Ay1']
        Az1=Vary.A1[4]+(Vary.A1[5]-Vary.A1[4])*param['Az1']
        AA1=jxn.array([Ax1,Ay1,Az1])
        pravals['A1']=AA1
    else:
        AA1=Ham.A1
    if 'Ax2' in param.keys():
        Ax2=Vary.A2[0]+(Vary.A2[1]-Vary.A2[0])*param['Ax2']
        Ay2=Vary.A2[2]+(Vary.A2[3]-Vary.A2[2])*param['Ay2']
        Az2=Vary.A2[4]+(Vary.A2[5]-Vary.A2[4])*param['Az2']
        AA2=jxn.array([Ax2,Ay2,Az2])
        pravals['A2']=AA2
    else:
        AA2=Ham.A2
    if 'D1' in param.keys():
        Dx1=Vary.D1[0]+(Vary.D1[1]-Vary.D1[0])*param['D1']
        Ey1=Vary.D1[2]+(Vary.D1[3]-Vary.D1[2])*param['E1']
        DD1=jxn.array([Dx1,Ey1])
        pravals['D1']=DD1
    else:
        DD1=Ham.D1
    if 'D2' in param.keys():
        Dx2=Vary.D2[0]+(Vary.D2[1]-Vary.D2[0])*param['D2']
        Ey2=Vary.D2[2]+(Vary.D2[3]-Vary.D2[2])*param['E2']
        DD2=jxn.array([Dx2,Ey2])
        pravals['D2']=DD2
    else:
        DD2=Ham.D2
    if 'Qx1' in param.keys():
        Qx1=Vary.Q1[0]+(Vary.Q1[1]-Vary.Q1[0])*param['Qx1']
        Qy1=Vary.Q1[2]+(Vary.Q1[3]-Vary.Q1[2])*param['Qy1']
        Qz1=Vary.Q1[4]+(Vary.Q1[5]-Vary.Q1[4])*param['Qz1']
        QQ1=jxn.array([Qx1,Qy1,Qz1])
        pravals['Q1']=QQ1
    else:
        QQ1=Ham.Q1
    if 'Qx2' in param.keys():
        Qx2=Vary.Q2[0]+(Vary.Q2[1]-Vary.Q2[0])*param['Qx2']
        Qy2=Vary.Q2[2]+(Vary.Q2[3]-Vary.Q2[2])*param['Qy2']
        Qz2=Vary.Q2[4]+(Vary.Q2[5]-Vary.Q2[4])*param['Qz2']
        QQ2=jxn.array([Qx2,Qy2,Qz2])
        pravals['Q2']=QQ2
    else:
        QQ2=Ham.Q2
    if 'Hpp1' in param.keys():
        Hppx=Vary.Hpp[0]+(Vary.Hpp[1]-Vary.Hpp[0])*param['Hpp1']
        Hppy=Vary.Hpp[2]+(Vary.Hpp[3]-Vary.Hpp[2])*param['Hpp2']
        HHpp=jxn.array([Hppx,Hppy])
        pravals['Hpp']=HHpp
    else:
        HHpp=Ham.Hpp
    Hat=Ham.replace(g1=gg1,g2=gg2,A1=AA1,A2=AA2,D1=DD1,D2=DD2,Q1=QQ1,Q2=QQ2,Hpp=HHpp)
    return Hat,pravals
    
def Plotbriggs(Blis,expr,espc):
    plt.figure(figsize=(8,6))
    plt.plot(Blis,expr,label='Data')
    plt.plot(Blis,espc/np.max(espc)*np.max(expr),label='Fit')
    formatter=EngFormatter(sep='') 
    plt.gca().yaxis.set_major_formatter(formatter)
    plt.xlabel('Field [mT]')
    plt.ylabel('Counts [A. U.]')
    plt.grid()
    plt.legend()
    plt.title('EPR Spectrum')
    plt.show()
    
def Computejacobian(func,pflat,unrav,Ham,Exp,expr,mode,iwas,jwas,kwas,weight,hulk,K=None):
    pflatj=jxn.array(pflat)
    nparam=len(pflatj)
    @jx.jit
    def Jacolumn(pflat_,tangent_):
        _,col=jx.jvp(lambda p: func(p,unrav,Ham,Exp,expr,mode,iwas,jwas,kwas,weight,hulk,K=K),(pflat_,),(tangent_,))
        return col
    cols=[]
    for i in range(nparam):
        tangent=jxn.zeros_like(pflatj).at[i].set(1.0)
        col=Jacolumn(pflatj,tangent)
        cols.append(np.asarray(col))
    return np.column_stack(cols)

def Residualsjax(pflat,unrav,Ham,Exp,expr,mode,iwas,jwas,kwas,weight,hulk,K=None):
    pravals=unrav(pflat)
    Hat=Ham.replace(g=pravals.get('g',Ham.g),A=pravals.get('A',Ham.A),D=pravals.get('D',Ham.D),Q=pravals.get('Q',Ham.Q),Hpp=pravals.get('Hpp',Ham.Hpp))
    if mode=='p':
        _,simul=JCalpowder(Hat,Exp,iwas,jwas,kwas,weight,hulk,hifi=True,ktra=K,ndev=1)
    else:
        _,simul,_,_,_,_,_=Calresonant(Hat,Exp,diagram=False,hifi=True,ktra=K)
    #Change to L2 norm instead of maximum 
    normi=jxn.maximum(jxn.linalg.norm(simul),1e-8)
    norme=jxn.maximum(jxn.linalg.norm(expr),1e-8)
    enorm=expr/norme
    snorm=simul/normi
    return snorm-enorm


def Toconsiderset(pravals,Vary,J,residuals,tol=1e-6):
    from jax.flatten_util import ravel_pytree
    p,_=ravel_pytree(pravals)
    lo,_=ravel_pytree({k:jxn.asarray(getattr(Vary,k),dtype=float)[0::2] for k in pravals})
    hi,_=ravel_pytree({k:jxn.asarray(getattr(Vary,k),dtype=float)[1::2] for k in pravals})
    p,lo,hi=np.asarray(p),np.asarray(lo),np.asarray(hi)
    span=hi-lo
    fixed=span<=0.0
    frac=(p-lo)/np.where(fixed,1.0,span)
    #Get chi2/2 derivative respect to parameters
    grad=J.T@np.asarray(residuals)            # d(chi2/2)/dp
    atlo=(~fixed)&(frac<=tol)&(grad>0.0)
    athi=(~fixed)&(frac>=1.0-tol)&(grad<0.0)
    return fixed|atlo|athi,span


def BuildresJax(pravals,Ham,Exp,expr,Vary,mode,method,iwas=None,jwas=None,kwas=None,weight=None,hulk=None,success=True,message='',iterations=None,K=None):
    from jax.flatten_util import ravel_pytree
    pflat,unrav=ravel_pytree(pravals)
    Hat=Ham.replace(g=pravals.get('g',Ham.g),A=pravals.get('A',Ham.A),D=pravals.get('D',Ham.D),Q=pravals.get('Q',Ham.Q),Hpp=pravals.get('Hpp',Ham.Hpp))
    if mode=='p':
        Blis,espc=JCalpowder(Hat,Exp,iwas,jwas,kwas,weight,hulk,hifi=True,ktra=K,ndev=1)
    else:
        Blis,espc,_,_,_,_,_=Calresonant(Hat,Exp,diagram=False,hifi=True,ktra=K)

    residuals=Residualsjax(pflat,unrav,Ham,Exp,expr,mode,iwas,jwas,kwas,weight,hulk,K=K)
    J=Computejacobian(Residualsjax,pflat,unrav,Ham,Exp,expr,mode,iwas,jwas,kwas,weight,hulk,K=K)
    active,span=Toconsiderset(pravals,Vary,J,residuals)
    free=~active
    n,p=len(residuals),J.shape[1]
    perr=np.full(p,np.nan)
    variance=np.full((p,p),np.nan)
    deno=max(n-int(free.sum()),1)
    chi2=float(jxn.sum(residuals**2))
    redchi2=chi2/deno
    if free.any():
        sf=span[free]
        varu,perru,_=Selectvarian(J[:,free]*sf,redchi2)
        perr[free]=perru*sf
        variance[np.ix_(free,free)]=varu*np.outer(sf,sf)
    errdict=unrav(perr)
    return Fitresult(Ham=Hat,spc=espc,params=np.asarray(pflat),method=method,chi2=chi2,redchi2=redchi2,residuals=np.asarray(residuals),denochi=deno,variance=variance,paramerrors=perr,paramerrorsdict=errdict,success=success,message=message,iterations=iterations),Blis

def Residualsjax2(pflat,unrav,Ham,Exp,expr,mode):
    pravals=unrav(pflat)
    Hat=Ham.replace(g1=pravals.get('g1',Ham.g1),g2=pravals.get('g2',Ham.g2),A1=pravals.get('A1',Ham.A1),A2=pravals.get('A2',Ham.A2),D1=pravals.get('D1',Ham.D1),D2=pravals.get('D2',Ham.D2),Q1=pravals.get('Q1',Ham.Q1),Q2=pravals.get('Q2',Ham.Q2),Hpp=pravals.get('Hpp',Ham.Hpp))
    if mode=='p':
        _,simul=JMulpol(Hat,Exp,graph=False,hifi=True,ktra=1000)
    elif mode=='c':
        _,simul=JMusic(Hat,Exp,graph=False,hifi=True)
    #Change to L2 norm instead of maximum 
    normi=jxn.maximum(jxn.linalg.norm(simul),1e-8)
    norme=jxn.maximum(jxn.linalg.norm(expr),1e-8)
    enorm=expr/norme
    snorm=simul/normi
    return snorm-enorm

def BuildresJax2(pravals,Ham,Exp,expr,Vary,mode,method,success=True,message='',iterations=None):
    from jax.flatten_util import ravel_pytree
    pflat,unrav=ravel_pytree(pravals)
    Hat=Ham.replace(g1=pravals.get('g1',Ham.g1),g2=pravals.get('g2',Ham.g2),A1=pravals.get('A1',Ham.A1),A2=pravals.get('A2',Ham.A2),D1=pravals.get('D1',Ham.D1),D2=pravals.get('D2',Ham.D2),Q1=pravals.get('Q1',Ham.Q1),Q2=pravals.get('Q2',Ham.Q2),Hpp=pravals.get('Hpp',Ham.Hpp))
    if mode=='p':
        Blis,espc=JMulpol(Hat,Exp,graph=False,hifi=True,ktra=1000)
    else:
        Blis,espc=JMusic(Hat,Exp,graph=False,hifi=True)
    residuals=Residualsjax2(pflat,unrav,Ham,Exp,expr,mode)
    pj=jxn.array(pflat)
    jac=jx.jit(lambda p,t: jx.jvp(lambda q: Residualsjax2(q,unrav,Ham,Exp,expr,mode),(p,),(t,))[1])
    J=np.column_stack([np.asarray(jac(pj,jxn.zeros_like(pj).at[i].set(1.0))) for i in range(len(pj))])
    active,span=Toconsiderset(pravals,Vary,J,residuals)
    free=~active
    n,p=len(residuals),J.shape[1]
    perr=np.full(p,np.nan)
    variance=np.full((p,p),np.nan)
    deno=max(n-int(free.sum()),1)
    chi2=float(jxn.sum(residuals**2))
    redchi2=chi2/deno
    if free.any():
        sf=span[free]
        varu,perru,_=Selectvarian(J[:,free]*sf,redchi2)
        perr[free]=perru*sf
        variance[np.ix_(free,free)]=varu*np.outer(sf,sf)
    errdict=unrav(perr)
    
    return Fitresult(Ham=Hat,spc=espc,params=np.asarray(pflat),method=method,chi2=chi2,redchi2=redchi2,residuals=np.asarray(residuals),denochi=deno,variance=variance,paramerrors=perr,paramerrorsdict=errdict,success=success,message=message,iterations=iterations),Blis


def Proyectar(p): 
    return jx.tree.map(lambda x:jxn.clip(x,0.,1.),p)

def Progradnorm(p,g):
    #Gradient norm without the components blocked by an active bound
    def free(x,gx):
        blocked=((x<=0.0)&(gx>0.0))|((x>=1.0)&(gx<0.0))
        return jxn.where(blocked,0.0,gx)
    return optax.global_norm(jx.tree.map(free,p,g))


def Jestimatek(SHam,dExp,Vary,param,M,Nucl='None',mode='p',nsamp=8,margin=1.5,seed=0):
    keys=jx.random.split(jx.random.PRNGKey(seed),nsamp)
    cands=[Fromsigtophy(param,SHam,Vary)[0]]
    cands+=[Fromsigtophy(jx.tree.map(lambda x: jx.random.uniform(k,x.shape),param),SHam,Vary)[0] for k in keys]
    if mode=='p':
        iwas,jwas,kwas,weight,hulk=[jxn.array(a) for a in Delaunay(dExp,M)]
        nm=lambda H: JCalpowder(H,dExp,iwas,jwas,kwas,weight,hulk,Nucl,ktra=1000,ndev=1,info=True)[3]
    else:
        nm=lambda H: Calresonant(H,dExp,Nucl,info=True)[7]
    return int(np.ceil(max(int(nm(H)) for H in cands)*margin))+4

def Briggs(Hamer,Exp,Vary,expr,maximal=2000,eps=1e-11,mode='p',M=70,gtoler=1e-5,Nucl='None',Nucl1='None',Nucl2='None',cgrad=1.0):
    '''
    Fitting function for the experimental data using the ADAM Algorithm. Uses the *Optax* (part of the Deepmind proyect) ADAM algorithm implementation with a learning rate of 0.007. The parameters are normalized to [0,1] within the range given in *Vary* 
    It's recommended to use the function in VS code, Jupyter or Colab, because the process can be stop at any moment using the stop process button of the notebook.
    
    Parameters
    ----------
    Hamer: Class
        Container for the hamiltonian parameters.
    Exp : Class
        Container for the experimental conditions.
    Vary : Class
        Container for the range and parameters to vary.
    
    exper : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of iterations to evalue the function.
    eps : float
        Tolerance value for the error. Default is 1e-11
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    M : int
        Number of divisions for the Delaunay grid.
    gtoler : float
        Relative tolerance of the projected gradient norm with respect to the error. Default is 1e-5.
    Nucl: str
	Isotope for the simulation of system.
    Nucl1: str
	First isotope for the simulation of the multisystem.
    Nucl2: str
	Second isotope for the simulation of the multisystem.

    Returns
    -------
    Ham : Class
        Container for the fitted hamiltonian parameters.
    fitresult : Class 
        Container for the results of the Fitting function.
        
    Example
    -------
    >>> import matplotlib.pyplot as plt
    >>> import epraya as epr
    >>> import numpy as np
    >>> #epr.Sload('ZCr5600-300K.dat',4096,[2,3])
    >>> B,spc=epr.Seek()
    >>> Ham,Exp,Vary=epr.Jstart()
    >>> Ham.S=3/2
    >>> Ham.I=0
    >>> Ham.Hpp=[0, 30.0]
    >>> Exp.Freq=9.43
    >>> Exp.Points=len(B)
    >>> Exp.Frange=[B[0],B[-1]]
    >>> Exp.Temperature = 300
    >>> Ham.g=np.array([2.2,2,2])
    >>> Ham.D=np.array([500,200])  
    >>> Vary.g=[1.5,2.5,1.5,2.5,1.5,2.5]
    >>> Vary.D=[200,1000,0.5,1000]
    >>> Vary.Hpp=[0.0, 100.0, 2.0, 100.0]
    >>> de=epr.Briggs(Ham,Exp,Vary,spc,maximal=700,mode='p')    
    Step   1 | Error: 8.31270e-02 |
    | gx: 2.1947 | gy: 1.9938 | gz: 1.9938 |
    | D: 504.7 | E: 204.0 |
    | Hppg: 0.0 | Hppl: 30.5 |
    Step  11 | Error: 3.10342e-02 |
    | gx: 2.1398 | gy: 1.9395 | gz: 1.9335 |
    | D: 550.5 | E: 240.1 |
    | Hppg: 0.0 | Hppl: 35.8 |
    Step  21 | Error: 8.40639e-03 |
    | gx: 2.0893 | gy: 1.8980 | gz: 1.8871 |
    | D: 589.7 | E: 280.1 |
    | Hppg: 0.0 | Hppl: 41.0 |
    Step  31 | Error: 4.93202e-03 |
    | gx: 2.0695 | gy: 1.8849 | gz: 1.8696 |
    | D: 629.3 | E: 319.8 |
    | Hppg: 0.0 | Hppl: 45.3 |
    Step  41 | Error: 3.02956e-03 |
    | gx: 2.0745 | gy: 1.8955 | gz: 1.8743 |
    | D: 665.0 | E: 339.4 |
    | Hppg: 0.0 | Hppl: 47.9 |
    Step  51 | Error: 2.32793e-03 |
    | gx: 2.0834 | gy: 1.9114 | gz: 1.8861 |
    | D: 684.6 | E: 347.6 |
    | Hppg: 0.0 | Hppl: 49.0 |
    Step  61 | Error: 2.24381e-03 |
    | gx: 2.0767 | gy: 1.9193 | gz: 1.8918 |
    | D: 686.8 | E: 348.8 |
    | Hppg: 0.0 | Hppl: 49.1 |
    Step  71 | Error: 1.99499e-03 |
    | gx: 2.0626 | gy: 1.9208 | gz: 1.8925 |
    | D: 682.1 | E: 347.8 |
    | Hppg: 0.0 | Hppl: 48.8 |
    Step  81 | Error: 2.28818e-03 |
    | gx: 2.0433 | gy: 1.9176 | gz: 1.8939 |
    | D: 692.1 | E: 342.7 |
    | Hppg: 0.0 | Hppl: 48.5 |
    Step  91 | Error: 1.99445e-03 |
    | gx: 2.0455 | gy: 1.9236 | gz: 1.9031 |
    | D: 704.7 | E: 336.8 |
    | Hppg: 0.0 | Hppl: 48.2 |
    Process ended at step 101, with error: 1.93857e-03 |
    | gx: 2.0523 | gy: 1.9298 | gz: 1.9095 |
    | D: 709.5 | E: 333.3 |
    | Hppg: 0.0 | Hppl: 48.2 |

    chi2 = 7.97762e+00 | chi2 residual = 1.95147e-03
    Parameters error (1 sigma): D: 17.32 | E: 14.82 | Hppg: Undeterminated | Hppl: 0.4423 | gx: 0.002274 | gy: 0.01138 | gz: 0.01202

       
    .. image:: /_static/briggs.png
       :alt: Plot of the briggs function
       :align: center
    '''
    global result
    result.clear()
    if isinstance(Hamer,JHval):
      Ham=deepcopy(Hamer)
      class StaticHam:
        def replace(self,**kwargs):
          import copy
          nham=copy.copy(self)
          for k,v in kwargs.items():
            setattr(nham,k,v)
          return nham
      class StaticExp:
          pass
      iwas,jwas,kwas,weight,hulk=None,None,None,None,None
      if mode=='p':
          iwas,jwas,kwas,weight,hulk=Delaunay(Exp,M)
          iwas,jwas,kwas,weight,hulk=jxn.array(iwas),jxn.array(jwas),jxn.array(kwas),jxn.array(weight),jxn.array(hulk)
	  
      SHam=StaticHam()
      SHam.S=float(Ham.S)
      SHam.I=float(Ham.I)
      SHam.L=float(Ham.L)
      SHam.lc=float(Ham.lc)
      SHam.Bk2=np.array(Ham.Bk2).tolist()
      SHam.Bk4=np.array(Ham.Bk4).tolist()
      SHam.Bk6=np.array(Ham.Bk6).tolist()
      SHam.Hpp=np.array(Ham.Hpp)
      SHam.eta=float(Ham.eta)
      SHam.g=np.array(Ham.g)
      SHam.A=np.array(Ham.A)
      SHam.Q=np.array(Ham.Q)
      SHam.D=np.array(Ham.D)
      Vary.Hpp=np.array(Vary.Hpp)
      dExp=StaticExp()
      dExp.Points=int(Exp.Points)
      dExp.Freq=float(Exp.Freq)
      dExp.Temperature=float(Exp.Temperature)
      dExp.Frange=np.array(Exp.Frange)
      dExp.Sampleframe=Exp.Sampleframe
      dExp.Molframe=Exp.Molframe
      dExp.gframe=Exp.gframe
      dExp.Aframe=Exp.Aframe
      dExp.Qframe=Exp.Qframe
      dExp.Dframe=Exp.Dframe
      dExp.Fdirection=Exp.Fdirection
      dExp.Mwdirection=Exp.Mwdirection
      def initpara(Ham,Vara):
          param={}
          def safelog(val,under,over):
              div=jxn.where(over==under,1e-10,over-under)
              frat=(val-under)/div
              safe=jxn.clip(frat,0,1)
              return safe
          if Vara.g!=0.0:
              param['gx']=safelog(Ham.g[0],Vara.g[0],Vara.g[1])
              param['gy']=safelog(Ham.g[1],Vara.g[2],Vara.g[3])
              param['gz']=safelog(Ham.g[2],Vara.g[4],Vara.g[5])
          if Vara.A!=0.0:
              param['Ax']=safelog(Ham.A[0],Vara.A[0],Vara.A[1])
              param['Ay']=safelog(Ham.A[1],Vara.A[2],Vara.A[3])
              param['Az']=safelog(Ham.A[2],Vara.A[4],Vara.A[5])
          if Vara.D!=0.0:
              param['D']=safelog(Ham.D[0],Vara.D[0],Vara.D[1])
              param['E']=safelog(Ham.D[1],Vara.D[2],Vara.D[3])
          if Vara.Q!=0.0:
              param['Qx']=safelog(Ham.Q[0],Vara.Q[0],Vara.Q[1])
              param['Qy']=safelog(Ham.Q[1],Vara.Q[2],Vara.Q[3])
              param['Qz']=safelog(Ham.Q[2],Vara.Q[4],Vara.Q[5])
          if jxn.any(Vara.Hpp):
              param['Hpp1']=safelog(Ham.Hpp[0],Vara.Hpp[0],Vara.Hpp[1])
              param['Hpp2']=safelog(Ham.Hpp[1],Vara.Hpp[2],Vara.Hpp[3])
          return param
      param=initpara(Ham,Vary)
      optimus=optax.adam
      optimus=optax.chain(optax.clip_by_global_norm(1.0),optax.adam(learning_rate=0.007))#,optax.zero_nans(),optax.adam(learning_rate=0.007))
      state=optimus.init(param)
      K=Jestimatek(SHam,dExp,Vary,param,M,Nucl,mode)
      TRACK.reset()
      TRACK.n+=9 
      def Errorcost1(params,exper,K):
          Hame,_=Fromsigtophy(params,SHam,Vary)
          if mode=='p':
              _,simul,_,nmax=JCalpowder(Hame,dExp,iwas,jwas,kwas,weight,hulk,Nucl,ktra=K,ndev=1,info=True)
          elif mode == 'c':
              reto=Calresonant(Hame,dExp,Nucl,ktra=K,info=True)
              simul,nmax=reto[1],reto[7]
          normi=jxn.maximum(jxn.linalg.norm(simul),1e-8)
          norme=jxn.maximum(jxn.linalg.norm(exper),1e-8)
          return jxn.sum((simul/normi-exper/norme)**2),nmax
      Degrad=jx.value_and_grad(Errorcost1,argnums=0,has_aux=True)
      step=0
      @partial(jx.jit, static_argnames=('K',))
      def updatenext(parats,current,exper,K):
          (error,nmax),grad=Degrad(parats,exper,K)
          gnorm=Progradnorm(parats,grad)
          next,state=optimus.update(grad,current,parats)
          param=Proyectar(optax.apply_updates(parats,next))
          return param,state,error,gnorm,nmax
      converged=False
      try:
          while step<(maximal):
              prev=(param,state)
              param,state,error,gnorm,nmax=updatenext(param,state,expr,K)
              if float(nmax)>K:
                  TRACK.n+=cgrad              
                  K=int(np.ceil(float(nmax)*1.5))+4
                  param,state=prev
                  continue
              TRACK.record(float(error),lambda: np.concatenate([np.atleast_1d(np.asarray(v,float))
                           for v in Fromsigtophy(prev[0],SHam,Vary)[1].values()]),w=cgrad)
              if error<eps or gnorm<(gtoler)*error:
                  converged=True
                  break
              if step%10==0:
                  print(f"Step {step+1:3d} | Error: {error:.5e} |")
                  Showparam(param,Vary)
              step+=1
      except KeyboardInterrupt:
          print("\n"+"="*50)
          print(f"Process stopped at step: {step}, with error: {error:.5e}")
          print("="*50)
          Hat,praval=Fromsigtophy(param,Ham,Vary)
          Showparam(param,Vary)
          fitres,Blis=BuildresJax(praval,Hat,Exp,expr,Vary,mode,method='Adam',iwas=iwas,jwas=jwas,kwas=kwas,weight=weight,hulk=hulk,
                                success=False,message='Interrupted by user',iterations=step,K=K)
          result['Ham']=fitres.Ham
          result['fit']=fitres
          print(f"\nchi2 = {fitres.chi2:.5e} | chi2 residual = {fitres.redchi2:.5e}")
          if fitres.paramerrorsdict is not None:
            print(f"Parameters error (1 sigma): {Jformaterrors(fitres.paramerrorsdict)}")
          Plotbriggs(Blis,expr,fitres.spc)
          return fitres.Ham,fitres

      print(f"Process ended at step {step+1:3d}, with error: {error:.5e} |")
      Hat,praval=Fromsigtophy(param,Ham,Vary)
      Showparam(param,Vary)
      fitres,Blis=BuildresJax(praval,Hat,Exp,expr,Vary,mode,method='Adam',iwas=iwas,jwas=jwas,kwas=kwas,weight=weight,hulk=hulk,
                                success=converged,message='Converged' if converged else 'Maximum iterations reached',iterations=step,K=K)
      
      result['Ham']=fitres.Ham
      result['fit']=fitres
      print(f"\nchi2 = {fitres.chi2:.5e} | chi2 residual = {fitres.redchi2:.5e}")
      if fitres.paramerrorsdict is not None:
        print(f"Parameters error (1 sigma): {Jformaterrors(fitres.paramerrorsdict)}")
      Plotbriggs(Blis,expr,fitres.spc)
      return fitres.Ham,fitres
    else:
      Ham=deepcopy(Hamer)
      class StaticHam:
        def replace(self,**kwargs):
          import copy
          nham=copy.copy(self)
          for k,v in kwargs.items():
            setattr(nham,k,v)
          return nham
      class StaticExp:
          pass
      SHam=StaticHam()
      SHam.S1,SHam.S2=float(Ham.S1),float(Ham.S2)
      SHam.I1,SHam.I2=float(Ham.I1),float(Ham.I2)
      SHam.L1,SHam.L2=float(Ham.L1),float(Ham.L2)
      SHam.lc1,SHam.lc2=float(Ham.lc1),float(Ham.lc2)
      SHam.Bk2=np.array(Ham.Bk2).tolist()
      SHam.Bk4=np.array(Ham.Bk4).tolist()
      SHam.Bk6=np.array(Ham.Bk6).tolist()
      SHam.Hpp=np.array(Ham.Hpp)
      SHam.eta=float(Ham.eta)
      SHam.g1,SHam.g2=np.array(Ham.g1),np.array(Ham.g2)
      SHam.A1,SHam.A2=np.array(Ham.A1),np.array(Ham.A2)
      SHam.Q1,SHam.Q2=np.array(Ham.Q1),np.array(Ham.Q2)
      SHam.D1,SHam.D2=np.array(Ham.D1),np.array(Ham.D2)
      SHam.A1_2,SHam.A2_1,SHam.X1_2=np.array(Ham.A1_2),np.array(Ham.A2_1),np.array(Ham.X1_2)
      Vary.Hpp=np.array(Vary.Hpp)
      dExp=StaticExp()
      dExp.Points=int(Exp.Points)
      dExp.Freq=float(Exp.Freq)
      dExp.Temperature=float(Exp.Temperature)
      dExp.Frange=np.array(Exp.Frange)
      dExp.Sampleframe1,dExp.Sampleframe2=Exp.Sampleframe1,Exp.Sampleframe2
      dExp.Molframe1,dExp.Molframe2=Exp.Molframe1,Exp.Molframe2
      dExp.gframe1,dExp.gframe2=Exp.gframe1,Exp.gframe2
      dExp.Aframe1,dExp.Aframe2=Exp.Aframe1,Exp.Aframe2
      dExp.Qframe1,dExp.Qframe2=Exp.Qframe1,Exp.Qframe2
      dExp.Dframe1,dExp.Dframe2=Exp.Dframe1,Exp.Dframe2
      dExp.Fdirection=Exp.Fdirection
      dExp.Mwdirection=Exp.Mwdirection
      def initpara2(Ham,Vara):
          param={}
          def safelog(val,under,over):
              div=jxn.where(over==under,1e-10,over-under)
              frat=(val-under)/div
              safe=jxn.clip(frat,0,1)
              return safe
          if Vara.g1!=0.0:
              param['gx1']=safelog(Ham.g1[0],Vara.g1[0],Vara.g1[1])
              param['gy1']=safelog(Ham.g1[1],Vara.g1[2],Vara.g1[3])
              param['gz1']=safelog(Ham.g1[2],Vara.g1[4],Vara.g1[5])
          if Vara.g2!=0.0:
              param['gx2']=safelog(Ham.g2[0],Vara.g2[0],Vara.g2[1])
              param['gy2']=safelog(Ham.g2[1],Vara.g2[2],Vara.g2[3])
              param['gz2']=safelog(Ham.g2[2],Vara.g2[4],Vara.g2[5])
          if Vara.A1!=0.0:
              param['Ax1']=safelog(Ham.A1[0],Vara.A1[0],Vara.A1[1])
              param['Ay1']=safelog(Ham.A1[1],Vara.A1[2],Vara.A1[3])
              param['Az1']=safelog(Ham.A1[2],Vara.A1[4],Vara.A1[5])
          if Vara.A2!=0.0:
              param['Ax2']=safelog(Ham.A2[0],Vara.A2[0],Vara.A2[1])
              param['Ay2']=safelog(Ham.A2[1],Vara.A2[2],Vara.A2[3])
              param['Az2']=safelog(Ham.A2[2],Vara.A2[4],Vara.A2[5])
          if Vara.D1!=0.0:
              param['D1']=safelog(Ham.D1[0],Vara.D1[0],Vara.D1[1])
              param['E1']=safelog(Ham.D1[1],Vara.D1[2],Vara.D1[3])
          if Vara.D2!=0.0:
              param['D2']=safelog(Ham.D2[0],Vara.D2[0],Vara.D2[1])
              param['E2']=safelog(Ham.D2[1],Vara.D2[2],Vara.D2[3])
          if Vara.Q1!=0.0:
              param['Qx1']=safelog(Ham.Q1[0],Vara.Q1[0],Vara.Q1[1])
              param['Qy1']=safelog(Ham.Q1[1],Vara.Q1[2],Vara.Q1[3])
              param['Qz1']=safelog(Ham.Q1[2],Vara.Q1[4],Vara.Q1[5])
          if Vara.Q2!=0.0:
              param['Qx2']=safelog(Ham.Q2[0],Vara.Q2[0],Vara.Q2[1])
              param['Qy2']=safelog(Ham.Q2[1],Vara.Q2[2],Vara.Q2[3])
              param['Qz2']=safelog(Ham.Q2[2],Vara.Q2[4],Vara.Q2[5])
          if jxn.any(Vara.Hpp):
              param['Hpp1']=safelog(Ham.Hpp[0],Vara.Hpp[0],Vara.Hpp[1])
              param['Hpp2']=safelog(Ham.Hpp[1],Vara.Hpp[2],Vara.Hpp[3])
          return param
      param=initpara2(Ham,Vary)
      optimus=optax.adam
      optimus=optax.chain(optax.clip_by_global_norm(1.0),optax.adam(learning_rate=0.007))#,optax.zero_nans(),optax.adam(learning_rate=0.1))
      state=optimus.init(param)
      def Errorcost2(params,exper):
          Hame,_=Fromsigtophy2(params,SHam,Vary)
          if mode=='p':
              _,simul=JMulpol(Hame,dExp,graph=False,ktra=1000)
          elif mode=='c':
              _,simul=JMusic(Hame,dExp,graph=False)
          #Change to L2 norm instead of maximum 
          normi=jxn.maximum(jxn.linalg.norm(simul),1e-8)
          norme=jxn.maximum(jxn.linalg.norm(exper),1e-8)
          enorm=exper/norme
          snorm=simul/normi
          return jxn.sum((snorm-enorm)**2)
      Degrad=jx.value_and_grad(Errorcost2,argnums=0)
      step=0
      @jx.jit
      def updatenext(parats,current,exper):
          error,grad=Degrad(parats,exper)
          gnorm=Progradnorm(parats,grad)
          next,state=optimus.update(grad,current,parats)
          param=Proyectar(optax.apply_updates(parats,next))
          return param,state,error,gnorm
      converged=False
      try:
          while step<(maximal):
              param,state,error,gnorm=updatenext(param,state,expr)
              if error<eps or gnorm<(gtoler)*error:
                  converged=True
                  break
              if step%10==0:
                  print(f"Step {step+1:3d} | Error: {error:.5e} |")
                  Showparam2(param,Vary)
              step+=1
      except KeyboardInterrupt:
          print("\n"+"="*50)
          print(f"Process stopped at step: {step}, with error: {error:.5e}")
          print("="*50)
          Hat,praval=Fromsigtophy2(param,Ham,Vary)
          Showparam2(param,Vary)
          fitres,Blis=BuildresJax2(praval,Hat,Exp,expr,Vary,mode,method='Adam',
                                success=False,message='Interrupted by user',iterations=step)
          result['Ham']=fitres.Ham
          result['fit']=fitres
          print(f"\nchi2 = {fitres.chi2:.5e} | chi2 residual = {fitres.redchi2:.5e}")
          if fitres.paramerrorsdict is not None:
            print(f"Parameters error (1 sigma): {Jformaterrors(fitres.paramerrorsdict)}")
          Plotbriggs(Blis,expr,fitres.spc)
          return fitres.Ham,fitres
      
      print(f"Process ended at step {step+1:3d}, with error: {error:.5e}")
      Hat,praval=Fromsigtophy2(param,Ham,Vary)
      Showparam2(param,Vary)
      fitres,Blis=BuildresJax2(praval,Hat,Exp,expr,Vary,mode,method='Adam',
                                success=converged,message='Converged' if converged else 'Maximum iterations reached',iterations=step)
      

      result['Ham']=fitres.Ham
      result['fit']=fitres
      print(f"\nchi2 = {fitres.chi2:.5e} | chi2 residual = {fitres.redchi2:.5e}")
      if fitres.paramerrorsdict is not None:
        print(f"Parameters error (1 sigma): {Jformaterrors(fitres.paramerrorsdict)}")
      Plotbriggs(Blis,expr,fitres.spc)
      return fitres.Ham,fitres
@partial(jx.custom_jvp,nondiff_argnums=(1,2))
def containeigh(A,hifi=False,epse=50):
    '''
    Wrap function for the eignevalues determination using JAX and making sure the value doesn't go to infinity by the energy degeneration.
    
    Parameters
    ----------
    
    A : jax.numpy.array
        Total hamiltonian matrix.
    
    epse : float
        Perturbation value in case there is energy degeneration. Default is 50.
    hifi : bool
        Bool value to calculate the jacobian matrix and with it the variance.
    
    Returns
    -------
    
    w : jax.numpy.array
        Energy values of the hamiltonian.
    v : jax.numpy.array
        Eigenvectors of the hamiltonian.
    '''
    w,v=jxn.linalg.eigh(A)
    return w,v

@containeigh.defjvp
def containeigh_jvp(hifi,epse,primals,tangents):
    (A,),(dA,)=primals,tangents
    w,v=containeigh(A,hifi,epse)
    scale=jxn.linalg.norm(A)
    #Scale of the perturbation, lower than the hamiltonian 
    eps1=epse*scale*jxn.finfo(A.dtype).eps
    eps=jxn.where(hifi,eps1*1e-2,eps1*1.0)
    #Hermitic condition
    dAh=0.5*(dA+jxn.swapaxes(dA,-1,-2).conj())
    vH=jxn.swapaxes(v,-1,-2).conj()
    M=vH@dAh@v
    dw=jxn.real(jxn.diagonal(M,axis1=-2,axis2=-1))
    wi=w[...,:,None]
    wj=w[...,None,:]
    denom=wj-wi             
    F=denom/(denom**2+eps**2)
    dV=v@(F*M)
    return (w,v),(dw,dV)
    
def Jformaterrors(ed):
    if ed is None:
        return None
    labels={'g':['gx','gy','gz'],'A':['Ax','Ay','Az'],'D':['D','E'],
            'Q':['Qx','Qy','Qz'],'Hpp':['Hppg','Hppl'],
            'g1':['gx_1','gy_1','gz_1'],'g2':['gx_2','gy_2','gz_2'],
            'A1':['Ax_1','Ay_1','Az_1'],'A2':['Ax_2','Ay_2','Az_2'],
            'D1':['D_1','E_1'],'D2':['D_2','E_2'],
            'Q1':['Qx_1','Qy_1','Qz_1'],'Q2':['Qx_2','Qy_2','Qz_2']}
    paramer=[]
    for key,arr in ed.items():
        arr=np.atleast_1d(np.asarray(arr))
        names=labels.get(key,[f'{key}{i}' for i in range(len(arr))])
        for name,val in zip(names,arr):
            if np.isnan(val):
                paramer.append(f"{name}: Undeterminated")
            else:
                paramer.append(f"{name}: {val:.4g}")
    return " | ".join(paramer)

In [ ]:
import json, time, io, os, contextlib
import numpy as np, pandas as pd, scipy
import jax, jax.numpy as jxn
from scipy.stats import qmc
from epraya.base_fit import Packtoscipy, UnpackHam

def Initialconf(Vary,samples=30,seed=42):
    low,high=[],[]
    if Vary.g!=0.0: 
        low+=[Vary.g[0],Vary.g[2],Vary.g[4]]
        high+=[Vary.g[1],Vary.g[3],Vary.g[5]]
    if Vary.A!=0.0:
        low+=[Vary.A[0],Vary.A[2],Vary.A[4]]
        high+=[Vary.A[1],Vary.A[3],Vary.A[5]]
    if Vary.D!=0.0:
        low+=[Vary.D[0],Vary.D[2]]
        high+=[Vary.D[1],Vary.D[3]]
    if np.any(Vary.Hpp):
        low+=[Vary.Hpp[0],Vary.Hpp[2]]
        high+=[Vary.Hpp[1],Vary.Hpp[3]]
    pts=qmc.LatinHypercube(d=len(low),seed=seed).random(n=samples)
    return qmc.scale(pts,l_bounds=low,u_bounds=high)
 
def Addenoise(spectrum,snr,seed=None):
    rng=np.random.default_rng(seed)
    sigma=np.ptp(spectrum)/float(snr)
    return spectrum+rng.normal(0.0,sigma,spectrum.shape),sigma

Ham1,Exp1,Vary1=Jstart()
Ham1.S,Ham1.I=1,1
Ham1.g=[2.003,3.001,2.5]
Ham1.A=[300,250,200]
Ham1.D=[200,600]
Ham1.Hpp=[10,5]
Exp1.Points,Exp1.Freq,Exp1.Frange,Exp1.Temperature=2048,9.43,[0,600],300
Vary1.g=[1,4,1,4,1,4]
Vary1.A=[0,400,0,400,0,400] 
Vary1.D=[0,500,200,800]
Vary1.Hpp=[0.5,30,0.5,30]
B1,spc1=JPowder(Ham1,Exp1,M=120,graph=False)   
spc1=np.asarray(spc1)
 
Mfit=40
Nsamples=6
pinitipoints=Initialconf(Vary1,Nsamples,seed=123)
ptrue=Packtoscipy(Ham1,Vary1)[0]
tol=scipy.stats.chi2.ppf(0.99,len(ptrue))
snr=[30,20,10,5]
TRACK.budget=1000                                       

iw=[jxn.array(a) for a in Delaunay(Exp1,Mfit)]
K=int(JCalpowder(Ham1,Exp1,*iw,ktra=None,ndev=1,info=True)[2])
yref=jxn.asarray(spc1)/jxn.linalg.norm(spc1)
def loss(g):
    s=JCalpowder(Ham1.replace(g=g),Exp1,*iw,ktra=K,ndev=1)[1]
    return jxn.sum((s/jxn.linalg.norm(s)-yref)**2)
f,vg=jax.jit(loss),jax.jit(jax.value_and_grad(loss))
g0=jxn.asarray(Ham1.g,float)
jax.block_until_ready(f(g0))
jax.block_until_ready(vg(g0))
def tmean(fun,n=3):
    t0=time.perf_counter()
    for _ in range(n): 
        jax.block_until_ready(fun(g0))
    return (time.perf_counter()-t0)/n
CGRAD=tmean(vg)/tmean(f)
print(f"K = {K} | c = {CGRAD:.2f}")
 
runs={'Briggs': lambda H,E,V,y: Briggs(H,E,V,y,maximal=10**4,eps=1e-11,mode='p',M=Mfit,gtoler=1e-5,cgrad=CGRAD)}
out='btarget2.jsonl'
if os.path.exists(out): 
    os.remove(out)
Plotbriggs=lambda *a, **k: None 
for i, snrva in enumerate(snr):
    spc4,sigma=Addenoise(spc1,snrva,seed=100+i)
    sigrel=sigma/np.linalg.norm(spc4)
    strue=JCalpowder(Ham1,Exp1,*iw,ktra=K,ndev=1)[1]         
    yn=jxn.asarray(spc4)/jxn.linalg.norm(spc4)
    chi2t=float(jxn.sum((strue/jxn.linalg.norm(strue)-yn)**2))
    TRACK.target=chi2t+tol*sigrel**2
    print(f"SNR {snrva}: chi2_ref={chi2t:.4e} | objetive={TRACK.target:.4e}")
    for pidx,p0 in enumerate(pinitipoints):
        for name,run in runs.items():
            H0=UnpackHam(p0,Ham1,Vary1)
            t0=time.perf_counter()
            
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    run(H0,Exp1,Vary1,spc4)
                status='stopped'
            except TargetReached:
                status='target'
            except BudgetReached:
                status='budget'
            last=TRACK.array()
            
            row=dict(sample='p',method=name,snr=snrva,start=pidx,status=status,neval=float(TRACK.n),t=time.perf_counter()-t0,t_first=float(last[0,2]),cgrad=CGRAD,
                       best=float(TRACK.best),d=float((TRACK.best-chi2t)/sigrel**2),params=last[-1,3:].tolist())
            print(f"SNR {snrva} | arranque {pidx} | {status} | neval={TRACK.n:.0f} | t={row['t']:.0f} s", flush=True)
            with open(out,'a') as fh:
                fh.write(json.dumps(row)+'\n')
            np.save(f'histbriggs_{name}_{snrva}_{pidx}.npy', last)
 
df=pd.read_json(out, lines=True)
def resume(x):
    ok=x.status=='target'
    return pd.Series(dict(psuccess=ok.mean(),ERT=x.neval.sum()/ok.sum() if ok.any() else np.inf,mediansuccess=x.loc[ok,'neval'].median(),dmedianfail=x.loc[~ok,'d'].median(),
                          t_median=x.t.median(),t_compile_median=x.t_first.median()))
print(df.groupby(['method', 'snr']).apply(resume))

In [5]:
import json, time, io, os, contextlib
import numpy as np, pandas as pd, scipy
import jax, jax.numpy as jxn
from scipy.stats import qmc
from epraya.base_fit import Packtoscipy, UnpackHam

def Initialconf(Vary,samples=30,seed=42):
    low,high=[],[]
    if Vary.g!=0.0: 
        low+=[Vary.g[0],Vary.g[2],Vary.g[4]]
        high+=[Vary.g[1],Vary.g[3],Vary.g[5]]
    if Vary.A!=0.0:
        low+=[Vary.A[0],Vary.A[2],Vary.A[4]]
        high+=[Vary.A[1],Vary.A[3],Vary.A[5]]
    if Vary.D!=0.0:
        low+=[Vary.D[0],Vary.D[2]]
        high+=[Vary.D[1],Vary.D[3]]
    if np.any(Vary.Hpp):
        low+=[Vary.Hpp[0],Vary.Hpp[2]]
        high+=[Vary.Hpp[1],Vary.Hpp[3]]
    pts=qmc.LatinHypercube(d=len(low),seed=seed).random(n=samples)
    return qmc.scale(pts,l_bounds=low,u_bounds=high)
 
def Addenoise(spectrum,snr,seed=None):
    rng=np.random.default_rng(seed)
    sigma=np.ptp(spectrum)/float(snr)
    return spectrum+rng.normal(0.0,sigma,spectrum.shape),sigma

Ham1,Exp1,Vary1=Jstart()
Ham1.S,Ham1.I=1,1
Ham1.g=[2.003,3.001,2.5]
Ham1.A=[300,250,200]
Ham1.D=[200,600]
Ham1.Hpp=[10,5]
Exp1.Points,Exp1.Freq,Exp1.Frange,Exp1.Temperature=2048,9.43,[0,600],300
Vary1.g=[1,4,1,4,1,4]
Vary1.A=[0,400,0,400,0,400] 
Vary1.D=[0,500,200,800]
Vary1.Hpp=[0.5,30,0.5,30]
Ccal=Calresonant(Ham1,Exp1,diagram=False,ktra=None)  
B1,spc1=Ccal[0],Ccal[1]
spc1=np.asarray(spc1)

Nsamples=6
pinitipoints=Initialconf(Vary1,Nsamples,seed=123)
ptrue=Packtoscipy(Ham1,Vary1)[0]
tol=scipy.stats.chi2.ppf(0.99,len(ptrue))
snr=[30,20,10,5]
TRACK.budget=1000                                       

K=int(Calresonant(Ham1,Exp1,ktra=None,info=True)[7])
yref=jxn.asarray(spc1)/jxn.linalg.norm(spc1)
def loss(g):
    s=Calresonant(Ham1.replace(g=g),Exp1,ktra=K)[1]
    return jxn.sum((s/jxn.linalg.norm(s)-yref)**2)
f,vg=jax.jit(loss),jax.jit(jax.value_and_grad(loss))
g0=jxn.asarray(Ham1.g,float)
jax.block_until_ready(f(g0))
jax.block_until_ready(vg(g0))
def tmean(fun,n=3):
    t0=time.perf_counter()
    for _ in range(n): 
        jax.block_until_ready(fun(g0))
    return (time.perf_counter()-t0)/n
CGRAD=tmean(vg)/tmean(f)
print(f"K = {K} | c = {CGRAD:.2f}")
 
runs={'Briggs': lambda H,E,V,y: Briggs(H,E,V,y,maximal=10**4,eps=1e-11,mode='c',M=30,gtoler=1e-5,cgrad=CGRAD)}
out='cbtarget.jsonl'
if os.path.exists(out): 
    os.remove(out)
Plotbriggs=lambda *a, **k: None 
for i, snrva in enumerate(snr):
    spc4,sigma=Addenoise(spc1,snrva,seed=100+i)
    sigrel=sigma/np.linalg.norm(spc4)
    strue=Calresonant(Ham1,Exp1,ktra=K)[1]         
    yn=jxn.asarray(spc4)/jxn.linalg.norm(spc4)
    chi2t=float(jxn.sum((strue/jxn.linalg.norm(strue)-yn)**2))
    TRACK.target=chi2t+tol*sigrel**2
    print(f"SNR {snrva}: chi2_ref={chi2t:.4e} | objetive={TRACK.target:.4e}")
    for pidx,p0 in enumerate(pinitipoints):
        for name,run in runs.items():
            H0=UnpackHam(p0,Ham1,Vary1)
            t0=time.perf_counter()
            
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    run(H0,Exp1,Vary1,spc4)
                status='stopped'
            except TargetReached:
                status='target'
            except BudgetReached:
                status='budget'
            last=TRACK.array()
            
            row=dict(sample='c',method=name,snr=snrva,start=pidx,status=status,neval=float(TRACK.n),t=time.perf_counter()-t0,t_first=float(last[0,2]),cgrad=CGRAD,
                       best=float(TRACK.best),d=float((TRACK.best-chi2t)/sigrel**2),params=last[-1,3:].tolist())
            print(f"SNR {snrva} | arranque {pidx} | {status} | neval={TRACK.n:.0f} | t={row['t']:.0f} s", flush=True)
            with open(out,'a') as fh:
                fh.write(json.dumps(row)+'\n')
            np.save(f'histb_{name}_{snrva}_{pidx}.npy', last)
 
df=pd.read_json(out, lines=True)
def resume(x):
    ok=x.status=='target'
    return pd.Series(dict(psuccess=ok.mean(),ERT=x.neval.sum()/ok.sum() if ok.any() else np.inf,mediansuccess=x.loc[ok,'neval'].median(),dmedianfail=x.loc[~ok,'d'].median(),
                          t_median=x.t.median(),t_compile_median=x.t_first.median()))
print(df.groupby(['method', 'snr']).apply(resume))

K = 27 | c = 1.63
SNR 30: chi2_ref=8.8970e-02 | objetive=8.9981e-02
SNR 30 | arranque 0 | budget | neval=1002 | t=16 s
SNR 30 | arranque 1 | budget | neval=1002 | t=15 s
SNR 30 | arranque 2 | budget | neval=1002 | t=18 s
SNR 30 | arranque 3 | stopped | neval=633 | t=14 s
SNR 30 | arranque 4 | target | neval=497 | t=8 s
SNR 30 | arranque 5 | stopped | neval=652 | t=12 s
SNR 20: chi2_ref=1.9284e-01 | objetive=1.9485e-01
SNR 20 | arranque 0 | budget | neval=1002 | t=14 s
SNR 20 | arranque 1 | budget | neval=1002 | t=15 s
SNR 20 | arranque 2 | budget | neval=1002 | t=14 s
SNR 20 | arranque 3 | budget | neval=1002 | t=14 s
SNR 20 | arranque 4 | target | neval=226 | t=6 s
SNR 20 | arranque 5 | stopped | neval=895 | t=15 s
SNR 10: chi2_ref=5.6598e-01 | objetive=5.7135e-01
SNR 10 | arranque 0 | budget | neval=1002 | t=14 s
SNR 10 | arranque 1 | budget | neval=1002 | t=13 s
SNR 10 | arranque 2 | budget | neval=1002 | t=14 s
SNR 10 | arranque 3 | budget | neval=1002 | t=14 s
SNR 10 | arranque 4 